# MMM end-to-end: ingestion generica, Meridian geo, confronto ROAS, budget allocator

Pipeline completa in un solo notebook. L'MMM e' il mezzo; il **budget allocator** e' il fine:
dal modello alle decisioni di allocazione della spesa tra canali.

**Flusso:** upload file grezzi -> ingestion generica (schema inferito) -> armonizzazione
`geo x settimana x canale` -> `InputData` Meridian -> EDA -> fit bayesiano geo ->
confronto ROAS incrementale vs benchmark attribuito -> ottimizzazione budget -> Excel multi-foglio.

**Prima di iniziare:**
- Runtime -> Cambia tipo di runtime -> **GPU (T4)** -> Salva (il fit senza GPU e' molto lento).
- Esegui le celle **in ordine**. Il notebook non gira mai tutto di fila: a ogni checkpoint
  stampa un riepilogo e chiede conferma con `input()` (rispondi `y` per proseguire, `n` per fermarti).
- Nessuna configurazione manuale richiesta: periodo, granularita', geo, valuta, KPI e canali
  vengono **auto-rilevati** dai dati e mostrati al Checkpoint 1 per conferma/override.

**Dove personalizzare (tutto nella cella CONFIG + DIZIONARI):**
- `SINONIMI_COLONNE` -> aggiungi qui i nomi di colonna dei tuoi export se non vengono riconosciuti.
- `REGOLE_CANALI` -> regole regex -> macro-canale (es. nuovi job board o piattaforme).
- `GEO_SINONIMI` / `PROVINCIA_A_REGIONE` -> tassonomia geografica.
- Override opzionali `PERIODO`, `GEO`, `VALUTA`, `KPI` (default `"auto"`).
- Parametri del fit e goal di ottimizzazione (default: **budget totale fisso**, vincoli +/-30%).

**Checkpoint di conferma:** 1) config auto-rilevata, 2) dopo harmonization, 3) dopo `InputData`,
4) prima del fit, 5) prima dell'ottimizzazione budget, 6) prima dell'export Excel.

**Privacy:** i dati restano in questa sessione Colab; il notebook non li invia altrove.

## 1. Setup: dipendenze e import

Installa Google Meridian (include TensorFlow) e le librerie di supporto. Richiede qualche minuto.

In [ ]:
# ============================================================================
# CELLA 1 - SETUP: dipendenze, import, seed, cartelle di output
# ============================================================================
# Versione FISSATA: vedi il blocco di controllo in fondo a questa cella.
!pip install -q "google-meridian==1.8.0" openpyxl statsmodels

# L'install qui sopra stampa quasi sempre alcune righe rosse di 'dependency
# conflicts' su pacchetti preinstallati dell'ambiente (SDK Gemini, YDF, ...) che
# questo notebook non importa. NON sono errori. L'unica prova che conta e' se gli
# import qui sotto passano: il verdetto viene stampato in fondo a questa cella.
try:
    import os, io, re, csv, json, math, pickle, inspect, unicodedata, difflib, warnings
    import datetime as dt
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    import tensorflow as tf
    import tensorflow_probability as tfp
    import arviz as az

    import meridian
    from meridian import constants
    from meridian.data import load
    from meridian.model import model, spec, prior_distribution
    from meridian.analysis import analyzer, optimizer, visualizer, summarizer
except ImportError as _err:
    print()
    print('!' * 76)
    print('ERRORE VERO - una libreria necessaria non si e' + chr(39) + ' importata:')
    print('   ' + str(_err))
    print()
    print('Questo NON e' + chr(39) + ' uno degli avvisi ignorabili di pip: il notebook non')
    print('puo' + chr(39) + ' proseguire. Rilancia questa cella; se il problema persiste,')
    print('riavvia la sessione e riesegui dalla prima cella.')
    print('!' * 76)
    raise

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Riproducibilita' ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Cartelle di output (Excel, figure, modello salvato) ---
OUTPUT_DIR = 'output_mmm'
FIG_DIR = os.path.join(OUTPUT_DIR, 'fig')
os.makedirs(FIG_DIR, exist_ok=True)

# --- Log versioni (finira' nel foglio README dell'Excel) ---
VERSIONI = {
    'python': '.'.join(map(str, __import__('sys').version_info[:3])),
    'google-meridian': getattr(meridian, '__version__', 'n/d'),
    'tensorflow': tf.__version__,
    'tensorflow-probability': tfp.__version__,
    'arviz': az.__version__,
    'pandas': pd.__version__,
    'numpy': np.__version__,
}
print('Versioni:', json.dumps(VERSIONI, indent=2))

# --- Blocco versione: Meridian DEVE essere la 1.8.0 -------------------------
# Il vincolo aperto ">=1.6" faceva prendere a pip l'ultima release pubblicata.
# Il 30 agosto era la 1.8.0 e il fit girava; le versioni successive lo rompono
# (roi_m in float32 contro un backend che vuole float64, poi tf.int32 dentro il
# backend numpy di TFP). Qui sopra la versione e' fissata a ==1.8.0: questo
# controllo verifica che l'ambiente sia davvero quello e si ferma se non lo e',
# invece di far scoprire il problema dieci minuti dopo, dentro al fit.
MERIDIAN_ATTESA = '1.8.0'
try:
    from importlib.metadata import version as _pkg_version
    _ver_meridian = _pkg_version('google-meridian')
except Exception:
    _ver_meridian = getattr(meridian, '__version__', 'n/d')

print()
print('Versioni effettive dopo l' + chr(39) + 'installazione:')
print('   google-meridian         ' + str(_ver_meridian))
print('   tensorflow              ' + str(tf.__version__))
print('   tensorflow-probability  ' + str(tfp.__version__))

if str(_ver_meridian) != MERIDIAN_ATTESA:
    print()
    print('!' * 76)
    print('VERSIONE DI MERIDIAN SBAGLIATA: il notebook si ferma qui.')
    print()
    print('   attesa:  ' + MERIDIAN_ATTESA)
    print('   trovata: ' + str(_ver_meridian))
    print()
    print('I quattro run della tesi sono stati fatti con la 1.8.0. Cambiare')
    print('versione cambia i numeri e fa perdere il confronto con i run gia' + chr(39))
    print('fatti, quindi NON aggiornare il notebook alla Meridian nuova.')
    print()
    print('Come rimediare:')
    print('   1. Runtime -> Riavvia sessione')
    print('   2. riesegui questa cella dall' + chr(39) + 'inizio')
    print('Se la versione sbagliata torna, l' + chr(39) + 'installazione qui sopra non ha')
    print('avuto effetto: controlla la riga di pip, deve dire ==1.8.0.')
    print('!' * 76)
    raise RuntimeError(
        'google-meridian ' + str(_ver_meridian) + ' invece di ' + MERIDIAN_ATTESA)

# --- requirements_freeze.txt: subito, non a fine run ------------------------
# L'Appendice A della tesi dice che le versioni sono congelate qui. Va scritto
# ADESSO e non alla fine, perche' se il run si interrompe a meta' il file non
# verrebbe mai prodotto. Conta soprattutto per tensorflow-probability, che con
# la 1.8.0 e' una build notturna: quelle da PyPI prima o poi spariscono, e senza
# il numero esatto l'ambiente non e' piu' ricostruibile.
import subprocess as _subprocess
PERCORSO_FREEZE = os.path.join(OUTPUT_DIR, 'requirements_freeze.txt')
try:
    _freeze = _subprocess.run(
        [__import__('sys').executable, '-m', 'pip', 'freeze'],
        capture_output=True, text=True, check=True).stdout
    with open(PERCORSO_FREEZE, 'w', encoding='utf-8') as _f:
        _f.write(_freeze)
    print()
    print('Versioni congelate in ' + PERCORSO_FREEZE +
          ' (' + str(len(_freeze.splitlines())) + ' pacchetti).')
except Exception as _e:
    print()
    print('ATTENZIONE: non sono riuscito a scrivere ' + PERCORSO_FREEZE + ': ' + str(_e))
    print('Il run puo' + chr(39) + ' proseguire, ma l' + chr(39) + 'ambiente non resta documentato.')


gpu = tf.config.list_physical_devices('GPU')
if gpu:
    print('GPU disponibile:', gpu[0].name)
else:
    print('ATTENZIONE: nessuna GPU rilevata. Il fit sara' + chr(39) + ' molto lento.')
    print('Runtime -> Cambia tipo di runtime -> GPU (T4).')

# --- Verdetto sull'ambiente -------------------------------------------------
# Se l'esecuzione e' + chr(39) + ' arrivata fin qui, gli import sono passati: qualunque riga
# rossa stampata da pip sopra riguarda pacchetti che questo notebook non usa.
print()
if gpu:
    print('Avvisi di pip ignorabili: riguardano pacchetti preinstallati di Colab')
    print('che questo notebook non usa. Ambiente OK.')
else:
    print('Avvisi di pip ignorabili: riguardano pacchetti preinstallati di Colab')
    print('che questo notebook non usa. Librerie OK; manca solo la GPU (vedi sopra).')


## 2. CONFIG + DIZIONARI (estendibili)

Tutto cio' che puoi voler modificare sta qui: sinonimi delle colonne, regole dei canali,
tassonomia geografica, valute, override, parametri del fit e goal di ottimizzazione.
Se al Checkpoint 1 qualcosa non viene riconosciuto, **aggiungi il sinonimo/la regola qui**
e riesegui da questa cella in poi.

In [ ]:
# ============================================================================
# CELLA 2 - CONFIG + DIZIONARI (tutto cio' che e' personalizzabile sta qui)
# ============================================================================

# ---------------------------------------------------------------------------
# 2.1 OVERRIDE OPZIONALI - lascia "auto" per l'auto-rilevamento dai dati.
# ---------------------------------------------------------------------------
PERIODO = 'auto'    # es. ('2024-01-01', '2025-12-31') per forzare la finestra
GEO     = 'auto'    # es. 'regione' / 'provincia' / 'nazionale' per forzare il livello
VALUTA  = 'auto'    # es. 'EUR' per forzare la valuta target
KPI     = 'auto'    # es. 'candidature' per forzare la scelta del KPI tra i candidati

AUTO_CONFERMA = False   # True = salta le richieste di conferma (usalo solo per riesecuzioni)
VALUTA_TARGET = 'EUR'   # valuta unica della base dati armonizzata

EXPORT_COMPLETO = True  # True = Excel COMPLETO: oltre ai fogli per il manager aggiunge
                        # quelli tecnici (modello, EDA, diagnostica, incertezza, dati puliti,
                        # README). E' il default perche' per l'uso di tesi gli intervalli di
                        # credibilita' servono SEMPRE: un export senza non e' rifacibile
                        # se non rilanciando tutto.
                        # Per la versione essenziale da consegnare a un business (solo
                        # Report_Manager, Legenda, Dettaglio_Campagne, Scenari_WhatIf):
                        # EXPORT_COMPLETO = False

# ---------------------------------------------------------------------------
# 2.2 SINONIMI COLONNE (IT/EN) - matching esatto, per sottostringa e fuzzy.
#     Aggiungi liberamente varianti: tutto viene normalizzato (minuscole,
#     senza accenti, senza punteggiatura) prima del confronto.
# ---------------------------------------------------------------------------
SOGLIA_FUZZY = 0.82   # soglia di somiglianza (0-1) per il fuzzy matching

SINONIMI_COLONNE = {
    'spend': ['spend', 'spesa', 'costo', 'cost', 'costs', 'amount spent',
              'importo speso', 'investimento', 'media cost', 'costo totale',
              'spesa totale', 'total spent', 'adv cost', 'budget speso'],
    'impressions': ['impressions', 'impression', 'impressioni', 'impr',
                    'visualizzazioni', 'views', 'ad impressions'],
    'clicks': ['clicks', 'click', 'clic', 'clics', 'link clicks',
               'clic sul link', 'clic totali'],
    'sessions': ['sessions', 'sessioni', 'visits', 'visite', 'users', 'utenti'],
    'date': ['date', 'data', 'day', 'giorno', 'week', 'settimana',
             'week start', 'inizio settimana', 'settimana iniziale',
             'data inizio', 'reporting date', 'giorno di riferimento',
             'month', 'mese', 'periodo', 'time', 'reporting starts'],
    'geo': ['geo', 'region', 'regione', 'provincia', 'province', 'citta',
            'city', 'area', 'territorio', 'zona', 'dma', 'location',
            'localita', 'mercato', 'market', 'area geografica'],
    'kpi': ['conversions', 'conversioni', 'conversion', 'purchase', 'acquisti',
            'applications', 'candidature', 'application', 'candidatura',
            'lead', 'leads', 'iscrizioni', 'signups', 'placements',
            'assunzioni', 'hires', 'submissions', 'domande ricevute'],
    'revenue': ['revenue', 'fatturato', 'ricavi', 'ricavo', 'entrate',
                'valore conversioni', 'conversion value', 'gmv',
                'margine lordo', 'gross margin'],
    'campaign': ['campaign', 'campagna', 'campaign name', 'nome campagna',
                 'adset', 'ad set', 'gruppo di annunci', 'ad group'],
    'channel': ['channel', 'canale', 'platform', 'piattaforma', 'publisher',
                'network', 'fonte canale', 'channel grouping',
                'raggruppamento canali'],
    'source': ['source', 'sorgente', 'utm source', 'origine'],
    'medium': ['medium', 'mezzo', 'utm medium'],
    'population': ['population', 'popolazione', 'pop', 'abitanti', 'residenti'],
    'currency': ['currency', 'valuta', 'divisa', 'currency code', 'codice valuta'],
    'quarter': ['quarter', 'trimestre', 'fiscal quarter', 'q'],
    'roas': ['roas', 'roi', 'return on ad spend', 'ritorno sulla spesa',
             'roas attribuito'],
}

# ---------------------------------------------------------------------------
# 2.3 REGOLE CANALI - lista ORDINATA di (regex, macro-canale): vince la prima
#     regola che matcha. Etichette speciali: _ORGANICO_ (va in organic_media,
#     senza spesa), _ESCLUDI_ (riga scartata, es. traffico diretto).
# ---------------------------------------------------------------------------
# I CANALI portano il nome della PIATTAFORMA (Google Ads, Meta Ads, ...):
# la tipologia di campagna (Search, PMax, Display, Retargeting...) NON sta qui,
# sta nel SOTTOCANALE (REGOLE_SOTTOCANALE piu' sotto).
REGOLE_CANALI = [
    (r'google|adwords|g[ _-]?ads|\bsem\b|paid[ _-]?search|search[ _-]?ads|youtube|\byt\b', 'Google Ads'),
    (r'bing|microsoft[ _-]?ads', 'Microsoft Ads'),
    (r'linkedin', 'LinkedIn Ads'),
    (r'meta|facebook|instagram|\bfb\b|\big\b', 'Meta Ads'),
    (r'tiktok|snapchat|twitter|\bx[ _-]?ads\b|paid[ _-]?social', 'Social - altro'),
    # Job board: canali SEPARATI, non un unico aggregato. La categoria non e'
    # omogenea (modelli d'asta, audience e saturazioni diverse) e il modello
    # stima meglio cio' che e' economicamente distinto. Per riaccorparle basta
    # sostituire queste quattro righe con una sola.
    # NB: InfoJobs ha chiuso il 31/12/2025 (annunci confluiti in Subito) e
    # Monster non e' piu' operativa: restano solo come etichette storiche.
    (r'indeed', 'Indeed'),
    (r'subito|infojobs', 'Subito Lavoro'),
    (r'jooble', 'Jooble'),
    (r'bakeca.*lavoro|trovolavoro|careerjet|talent[ _-]?com|adzuna|jobrapido|monster|job[ _-]?board', 'Altre job board'),
    (r'display|programmatic|\bgdn\b|dv360|banner|\bctv\b|\bott\b', 'Programmatic/Display'),
    (r'email|newsletter|\bdem\b|\bcrm\b', 'Email'),
    (r'organic|organico|\bseo\b', '_ORGANICO_'),
    # NB: le regex vengono applicate al testo NORMALIZZATO (minuscolo, senza punteggiatura):
    # "(none)" diventa "none", quindi niente parentesi nelle regole.
    (r'\bdirect\b|diretto|\bnone\b|\bnot set\b', '_ESCLUDI_'),
    (r'referral|affiliat', 'Referral'),
]
CANALE_NON_MAPPATO = 'Altro'   # etichetta per cio' che nessuna regola cattura

# ---------------------------------------------------------------------------
# 2.3-bis GERARCHIA A 4 LIVELLI: gruppo > canale > sottocanale > campagna
#
#   - GRUPPO (vista aggregata): paid media, job board, owned, terze parti...
#   - CANALE (livello del MODELLO): e' qui che Meridian stima ROAS e alloca.
#   - SOTTOCANALE (vista fine): tipo di campagna dedotto dal NOME campagna
#     (es. Performance Max, Display, Search Brand dentro "Google Ads").
#   - CAMPAGNA: la singola riga dell'export.
#
#   NB: il modello causale lavora a livello CANALE (le singole campagne sono
#   troppe e collineari per stimarne l'effetto separato). Sottocanali e
#   campagne alimentano le viste descrittive e il riparto del budget.
# ---------------------------------------------------------------------------
GRUPPO_CANALE = {
    'Google Ads': 'Paid media', 'Microsoft Ads': 'Paid media', 'Meta Ads': 'Paid media',
    'LinkedIn Ads': 'Paid media', 'Social - altro': 'Paid media',
    'Programmatic/Display': 'Paid media',
    'Indeed': 'Job board', 'Subito Lavoro': 'Job board',
    'Jooble': 'Job board', 'Altre job board': 'Job board',
    'Email': 'Owned media',
    'Referral': 'Terze parti',
    'Altro': 'Altro',
}
GRUPPO_DEFAULT = 'Altro'

# Regole ORDINATE (regex sul nome campagna normalizzato) -> sottocanale.
REGOLE_SOTTOCANALE = [
    (r'performance ?max|\bpmax\b|\bp max\b', 'Performance Max'),
    (r'shopping', 'Shopping'),
    (r'display|\bgdn\b|banner|programmatic', 'Display'),
    (r'video|youtube|\byt\b|reels|bumper', 'Video'),
    (r'retargeting|remarketing|\brtg\b|retention', 'Retargeting'),
    (r'prospecting|acquisition|awareness|reach|traffic', 'Prospecting'),
    (r'lead ?gen|lead ?ads|instant ?form|modulo', 'Lead Gen'),
    (r'\bbrand\b|branded', 'Search Brand'),
    (r'generic|no ?brand|non ?brand|generica', 'Search Generica'),
    (r'rete di ricerca|\bsearch\b', 'Search'),
    (r'sponsored|sponsorizzat', 'Sponsored'),
    (r'\bdem\b|newsletter', 'DEM/Newsletter'),
]
SOTTOCANALE_DEFAULT = 'Generale'   # quando nessuna regola matcha il nome campagna

# ---------------------------------------------------------------------------
# 2.4 TASSONOMIA GEO - regioni italiane, province -> regione, sinonimi EN.
#     Aggiungi qui eventuali nomi custom usati nei tuoi export.
# ---------------------------------------------------------------------------
REGIONI_IT = ['piemonte', 'valle d aosta', 'lombardia', 'trentino alto adige',
              'veneto', 'friuli venezia giulia', 'liguria', 'emilia romagna',
              'toscana', 'umbria', 'marche', 'lazio', 'abruzzo', 'molise',
              'campania', 'puglia', 'basilicata', 'calabria', 'sicilia', 'sardegna']

_PROVINCE = {
    'Piemonte': [('torino','to'),('vercelli','vc'),('novara','no'),('cuneo','cn'),
                 ('asti','at'),('alessandria','al'),('biella','bi'),('verbano cusio ossola','vb')],
    'Valle d Aosta': [('aosta','ao')],
    'Lombardia': [('varese','va'),('como','co'),('sondrio','so'),('milano','mi'),
                  ('bergamo','bg'),('brescia','bs'),('pavia','pv'),('cremona','cr'),
                  ('mantova','mn'),('lecco','lc'),('lodi','lo'),('monza e della brianza','mb')],
    'Trentino Alto Adige': [('bolzano','bz'),('trento','tn')],
    'Veneto': [('verona','vr'),('vicenza','vi'),('belluno','bl'),('treviso','tv'),
               ('venezia','ve'),('padova','pd'),('rovigo','ro')],
    'Friuli Venezia Giulia': [('udine','ud'),('gorizia','go'),('trieste','ts'),('pordenone','pn')],
    'Liguria': [('imperia','im'),('savona','sv'),('genova','ge'),('la spezia','sp')],
    'Emilia Romagna': [('piacenza','pc'),('parma','pr'),('reggio emilia','re'),('modena','mo'),
                       ('bologna','bo'),('ferrara','fe'),('ravenna','ra'),
                       ('forli cesena','fc'),('rimini','rn')],
    'Toscana': [('massa carrara','ms'),('lucca','lu'),('pistoia','pt'),('firenze','fi'),
                ('livorno','li'),('pisa','pi'),('arezzo','ar'),('siena','si'),
                ('grosseto','gr'),('prato','po')],
    'Umbria': [('perugia','pg'),('terni','tr')],
    'Marche': [('pesaro e urbino','pu'),('ancona','an'),('macerata','mc'),
               ('ascoli piceno','ap'),('fermo','fm')],
    'Lazio': [('viterbo','vt'),('rieti','ri'),('roma','rm'),('latina','lt'),('frosinone','fr')],
    'Abruzzo': [('l aquila','aq'),('teramo','te'),('pescara','pe'),('chieti','ch')],
    'Molise': [('campobasso','cb'),('isernia','is')],
    'Campania': [('caserta','ce'),('benevento','bn'),('napoli','na'),('avellino','av'),('salerno','sa')],
    'Puglia': [('foggia','fg'),('bari','ba'),('taranto','ta'),('brindisi','br'),
               ('lecce','le'),('barletta andria trani','bt')],
    'Basilicata': [('potenza','pz'),('matera','mt')],
    'Calabria': [('cosenza','cs'),('catanzaro','cz'),('reggio calabria','rc'),
                 ('crotone','kr'),('vibo valentia','vv')],
    'Sicilia': [('trapani','tp'),('palermo','pa'),('messina','me'),('agrigento','ag'),
                ('caltanissetta','cl'),('enna','en'),('catania','ct'),('ragusa','rg'),('siracusa','sr')],
    'Sardegna': [('sassari','ss'),('nuoro','nu'),('cagliari','ca'),('oristano','or'),('sud sardegna','su')],
}
PROVINCIA_A_REGIONE = {}   # nome provincia (normalizzato) -> regione
SIGLA_A_REGIONE = {}       # sigla a 2 lettere -> regione
for _reg, _lista in _PROVINCE.items():
    for _nome, _sigla in _lista:
        PROVINCIA_A_REGIONE[_nome] = _reg
        SIGLA_A_REGIONE[_sigla] = _reg

# Sinonimi geografici (EN -> IT, abbreviazioni, varianti). Estendibile.
GEO_SINONIMI = {
    'lombardy': 'lombardia', 'piedmont': 'piemonte', 'tuscany': 'toscana',
    'sicily': 'sicilia', 'sardinia': 'sardegna', 'apulia': 'puglia',
    'latium': 'lazio', 'aosta valley': 'valle d aosta',
    'trentino south tyrol': 'trentino alto adige',
    'emilia': 'emilia romagna', 'friuli': 'friuli venezia giulia',
    'milan': 'milano', 'rome': 'roma', 'turin': 'torino', 'naples': 'napoli',
    'florence': 'firenze', 'venice': 'venezia', 'genoa': 'genova',
    'padua': 'padova', 'syracuse': 'siracusa',
}

# ---------------------------------------------------------------------------
# 2.5 VALUTE - simboli e codici ISO riconosciuti nei valori/nomi colonna.
# ---------------------------------------------------------------------------
SIMBOLI_VALUTA = {'€': 'EUR', '$': 'USD', '£': 'GBP'}
CODICI_VALUTA = ['EUR', 'USD', 'GBP', 'CHF', 'PLN', 'SEK', 'DKK', 'NOK']
TASSI_CAMBIO = {}   # es. {'USD': 0.92} = 1 USD -> 0.92 EUR; se manca, chiesto a runtime

# ---------------------------------------------------------------------------
# 2.6 PARAMETRI DEL FIT MERIDIAN (priori debolmente informativi)
# ---------------------------------------------------------------------------
PRIOR_ROI_MU = 0.2       # media (log) della LogNormal sul ROI dei canali
PRIOR_ROI_SIGMA = 0.9    # dev. std (log): larga = debolmente informativa

# --- prior informativi per canale (facoltativi, default SPENTI) -------------
# Con USA_PRIOR_INFORMATIVI = False il notebook si comporta esattamente come
# prima: un solo prior LogNormal(PRIOR_ROI_MU, PRIOR_ROI_SIGMA) uguale per tutti
# i canali. I run gia' fatti restano riproducibili.
#
# Acceso, il prior sul ROI di ogni canale viene ancorato a un ROAS di
# riferimento: mu = log(ROAS di riferimento), sigma = PRIOR_ROI_SIGMA_INF.
#
# DA DOVE PRENDERE I ROAS DI RIFERIMENTO: da fonti note PRIMA del periodo che
# si sta stimando - il ROAS dell'anno precedente, un test geo chiuso, una
# stima di piattaforma di un periodo passato. MAI la media del periodo stesso:
# quello sarebbe usare i dati due volte, una per il prior e una per la
# verosimiglianza, e il posterior confermerebbe cio' che gli e' stato dato in
# ingresso invece di stimarlo.
#
# COSA VUOL DIRE sigma = 0.30: e' la dev. std in scala logaritmica. Al 90% di
# probabilita' il ROI vero sta fra exp(-1,645*0,30) = 61% e exp(+1,645*0,30) =
# 164% del riferimento. Stretto abbastanza da spostare le stime, largo
# abbastanza da lasciare che siano i dati a smentire il riferimento.
USA_PRIOR_INFORMATIVI = False      # default spento
PRIOR_ROI_SIGMA_INF   = 0.30
PRIOR_ROAS_CANALE     = {}         # {nome canale: ROAS di riferimento}
# Esempio (i nomi devono essere quelli di CANALI_MODELLO, tutti presenti):
# PRIOR_ROAS_CANALE = {'Google Ads': 1.5, 'Meta Ads': 1.6, 'LinkedIn Ads': 0.9,
#                      'Indeed': 2.5, 'Subito Lavoro': 2.9, 'Jooble': 2.6,
#                      'Altre job board': 2.8}
KNOTS_PER_QUARTER = 3    # flessibilita' della baseline temporale (nodi per trimestre)
N_CHAINS = 4
N_ADAPT = 500
N_BURNIN = 500
N_KEEP = 1000

# ---------------------------------------------------------------------------
# 2.7 GOAL DI OTTIMIZZAZIONE (cuore della tesi)
#     Default: massimizzare le conversioni incrementali a BUDGET TOTALE FISSO,
#     riallocando la spesa tra canali (nessun aumento di budget).
# ---------------------------------------------------------------------------
GOAL_OTTIMIZZAZIONE = 'budget_fisso'   # 'budget_fisso' (default tesi) | 'budget_flessibile'
BUDGET_TOTALE = 'storico'              # 'storico' = spesa osservata nel periodo; oppure un numero
VINCOLO_SPESA_PCT = 0.30               # bound simmetrico +/-30% sulla spesa di ogni canale
VINCOLI_PER_CANALE = {}                # override per canale: {'Google Ads': (0.20, 0.50), ...}
# Scenario documentato nella tesi: pavimento di spesa su LinkedIn per employer
# branding. Il canale ha ROI vero sotto il pareggio, quindi senza vincolo
# l'ottimizzatore lo azzererebbe; col pavimento si ferma al paletto e
# l'eguaglianza dei rendimenti marginali vale tra i soli canali liberi.
# VINCOLI_PER_CANALE = {'LinkedIn Ads': (0.0, 0.30)}  # pavimento employer branding, Sez. 3.7
TARGET_MROI = 1.0                      # per lo scenario secondario a budget flessibile
SCENARI_MOLTIPLICATORI = [0.8, 0.9, 1.0, 1.1, 1.2]   # what-if sul budget totale
IGP_FORMULA = None                     # es. lambda df: ... ; se None la colonna IGP non viene creata

# ---------------------------------------------------------------------------
# 2.8 HELPER: checkpoint human-in-the-loop, menu a scelta, data quality log
# ---------------------------------------------------------------------------
DQ_REPORT = []   # data quality report globale (stampato e salvato nell'Excel)

def segnala(problema, scelta=''):
    # Registra un problema di qualita' dati e la scelta fatta per gestirlo.
    riga = {'problema': str(problema), 'scelta': str(scelta)}
    DQ_REPORT.append(riga)
    print('[DQ] ' + riga['problema'] + ((' -> ' + riga['scelta']) if scelta else ''))

def intestazione_run():
    # Una riga d'identita' della run, stampata in cima a OGNI checkpoint.
    # Serve quando un output viene incollato altrove: senza questa riga non si
    # capisce da quale esecuzione venga.
    _g = globals().get
    _prior = 'informativi per canale' if _g('USA_PRIOR_INFORMATIVI') else 'debolmente informativi'
    return ('RUN | dati: ' + str(_g('ORIGINE_DATI') or '(cella 4 non ancora eseguita)') +
            ' | nodi/trim: ' + str(_g('KNOTS_PER_QUARTER', '?')) +
            ' | prior: ' + _prior +
            ' | seed: ' + str(_g('SEED', '?')))


def checkpoint(nome, righe):
    # Stampa un riepilogo leggibile e chiede conferma prima di proseguire.
    print()
    print('=' * 72)
    print('CHECKPOINT - ' + nome)
    print(intestazione_run())
    print('=' * 72)
    for r in righe:
        print('  - ' + str(r))
    print('=' * 72)
    if AUTO_CONFERMA:
        print('[AUTO_CONFERMA attiva: proseguo senza chiedere]')
        return
    risposta = input('Confermi e prosegui? [y/n] ').strip().lower()
    if risposta not in ('y', 'yes', 's', 'si'):
        raise SystemExit('Esecuzione interrotta al checkpoint: ' + nome +
                         '. Correggi la CONFIG o i dizionari e riesegui.')

def scegli_opzione(domanda, opzioni, default=0):
    # Menu numerato a runtime; invio vuoto = default.
    print()
    print(domanda)
    for i, o in enumerate(opzioni):
        print('  [' + str(i) + '] ' + str(o) + ('   (default)' if i == default else ''))
    if AUTO_CONFERMA:
        print('[AUTO_CONFERMA: scelgo il default ' + str(default) + ']')
        return default
    while True:
        r = input('Scelta [' + str(default) + ']: ').strip()
        if r == '':
            return default
        if r.isdigit() and 0 <= int(r) < len(opzioni):
            return int(r)
        print('Valore non valido, riprova.')

def chiedi_numero(domanda, default=None, vuoto_ok=True):
    # Chiede un numero a runtime; invio vuoto -> default (o None).
    while True:
        r = input(domanda + ' ').strip().replace(',', '.')
        if r == '':
            return default if not vuoto_ok else default
        try:
            return float(r)
        except ValueError:
            print('Numero non valido, riprova.')

print('CONFIG e dizionari caricati.')
print('Ruoli colonna riconosciuti:', ', '.join(SINONIMI_COLONNE.keys()))
print('Macro-canali nelle regole:', ', '.join(sorted({m for _, m in REGOLE_CANALI if not m.startswith('_')})))

## 3. Funzioni di ingestion generica

Lettura robusta (encoding, delimitatori, header su righe diverse, righe di totale),
inferenza dello schema tramite sinonimi + fuzzy matching, parsing date multi-formato,
rilevamento valuta/geo/canali. Nessuna colonna hardcoded: il notebook capisce i dati da solo.

In [ ]:
# ============================================================================
# CELLA 3 - FUNZIONI DI INGESTION GENERICA
# ============================================================================

# ---------------------------------------------------------------------------
# 3.1 Normalizzazione testo e matching colonne (sinonimi + fuzzy)
# ---------------------------------------------------------------------------
def norm_txt(s):
    # minuscole, senza accenti, solo [a-z0-9] e spazi singoli
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r'[^a-z0-9]+', ' ', s).strip()
    return s

def abbina_colonna(nome):
    # Restituisce (ruolo, punteggio) per il nome colonna, o (None, 0).
    # Priorita': match esatto > parola contenuta > fuzzy (difflib).
    n = norm_txt(nome)
    if not n:
        return None, 0.0
    migliore, punteggio = None, 0.0
    for ruolo, alias in SINONIMI_COLONNE.items():
        for a in alias:
            an = norm_txt(a)
            if n == an:
                return ruolo, 1.0
            p = 0.0
            # parola intera contenuta (es. "costo (eur)" contiene "costo")
            if re.search(r'(^| )' + re.escape(an) + r'( |$)', n):
                p = 0.93
            else:
                p = difflib.SequenceMatcher(None, n, an).ratio()
            if p > punteggio:
                migliore, punteggio = ruolo, p
    if punteggio >= SOGLIA_FUZZY:
        return migliore, punteggio
    return None, punteggio

def mappa_ruoli_colonne(df):
    # Assegna a ogni ruolo la colonna migliore (una colonna -> un solo ruolo).
    candidati = []
    for col in df.columns:
        ruolo, p = abbina_colonna(col)
        if ruolo:
            candidati.append((p, ruolo, col))
    candidati.sort(reverse=True)
    ruoli, usate = {}, set()
    for p, ruolo, col in candidati:
        if ruolo not in ruoli and col not in usate:
            ruoli[ruolo] = col
            usate.add(col)
    return ruoli

# ---------------------------------------------------------------------------
# 3.2 Lettura robusta dei file (encoding, delimitatore, header, totali)
# ---------------------------------------------------------------------------
def _decodifica(dati_bytes):
    for enc in ('utf-8-sig', 'utf-8', 'utf-16', 'cp1252', 'latin-1'):
        try:
            return dati_bytes.decode(enc), enc
        except (UnicodeDecodeError, UnicodeError):
            continue
    return dati_bytes.decode('latin-1', errors='replace'), 'latin-1(force)'

def _rileva_delimitatore(testo):
    campione = '\n'.join(testo.splitlines()[:30])
    try:
        return csv.Sniffer().sniff(campione, delimiters=',;\t|').delimiter
    except csv.Error:
        conte = {d: campione.count(d) for d in [',', ';', '\t', '|']}
        return max(conte, key=conte.get) if max(conte.values()) > 0 else ','

def leggi_file_grezzo(nome, dati_bytes):
    # Restituisce dict {nome_tabella: DataFrame senza header} per csv/tsv/txt/xlsx/json.
    est = os.path.splitext(nome)[1].lower()
    tabelle = {}
    if est in ('.xlsx', '.xls'):
        fogli = pd.read_excel(io.BytesIO(dati_bytes), sheet_name=None, header=None)
        for nf, df in fogli.items():
            if df.dropna(how='all').shape[0] > 1:
                tabelle[nome + '::' + str(nf)] = df
    elif est == '.json':
        df = pd.json_normalize(json.loads(dati_bytes.decode('utf-8', errors='replace')))
        tabelle[nome] = pd.DataFrame(np.vstack([df.columns.values, df.values]))
    else:  # csv / tsv / txt
        testo, enc = _decodifica(dati_bytes)
        delim = _rileva_delimitatore(testo)
        # numero massimo di campi: cosi' le righe di preambolo corte non rompono il parser
        n_max = max(len(r) for r in csv.reader(io.StringIO(testo), delimiter=delim))
        df = pd.read_csv(io.StringIO(testo), sep=delim, header=None, dtype=str,
                         skip_blank_lines=False, engine='python',
                         names=list(range(n_max)))
        tabelle[nome] = df
        print('  lettura ' + nome + ': encoding=' + enc + ', sep=' + repr(delim))
    return tabelle

def trova_header(df, max_righe=15):
    # L'header e' la riga (tra le prime max_righe) con piu' celle che matchano un sinonimo.
    migliore, punteggio = 0, -1
    for i in range(min(max_righe, len(df))):
        riga = df.iloc[i]
        n_match = sum(1 for v in riga if pd.notna(v) and abbina_colonna(v)[0])
        n_piene = int(riga.notna().sum())
        p = n_match * 10 + n_piene
        if n_match > 0 and p > punteggio:
            migliore, punteggio = i, p
    return migliore

_RE_TOTALI = re.compile(r'^\s*(tot(ale|al)?s?|subtot|somma|grand total|complessivo|riepilogo)\b', re.I)

def pulisci_tabella(df_raw):
    # Applica header rilevato, elimina righe vuote e righe di totale/subtotale.
    h = trova_header(df_raw)
    intestazioni = [str(v).strip() if pd.notna(v) else 'col_' + str(j)
                    for j, v in enumerate(df_raw.iloc[h])]
    df = df_raw.iloc[h + 1:].copy()
    df.columns = intestazioni
    df = df.dropna(how='all')
    df = df.loc[:, ~df.columns.duplicated()]
    maschera_tot = df.apply(
        lambda r: any(isinstance(v, str) and _RE_TOTALI.match(v) for v in r.values[:3]), axis=1)
    n_tot = int(maschera_tot.sum())
    if n_tot:
        segnala(str(n_tot) + ' righe di totale/subtotale trovate', 'scartate')
    return df[~maschera_tot], h

# ---------------------------------------------------------------------------
# 3.3 Pulizia numerica e rilevamento valuta
# ---------------------------------------------------------------------------
def pulisci_numerico(serie):
    # Converte stringhe tipo "1.234,56 EUR" / "$1,234.56" in float.
    # Restituisce (serie_float, set_simboli_valuta_trovati).
    valute = set()
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors='coerce'), valute
    def _uno(v):
        if pd.isna(v):
            return np.nan
        s = str(v).strip()
        for sim, iso in SIMBOLI_VALUTA.items():
            if sim in s:
                valute.add(iso)
                s = s.replace(sim, '')
        for iso in CODICI_VALUTA:
            if re.search(r'\b' + iso + r'\b', s, re.I):
                valute.add(iso)
                s = re.sub(r'\b' + iso + r'\b', '', s, flags=re.I)
        s = re.sub(r'[  ]', '', s)
        s = s.replace('%', '')   # percentuali: il chiamante decide se dividere per 100
        if s in ('', '-', 'na', 'n/a', 'nd', 'n.d.'):
            return np.nan
        if ',' in s and '.' in s:
            # l'ultimo separatore e' il decimale
            if s.rfind(',') > s.rfind('.'):
                s = s.replace('.', '').replace(',', '.')
            else:
                s = s.replace(',', '')
        elif ',' in s:
            if re.match(r'^-?\d+,\d{1,2}$', s):
                s = s.replace(',', '.')       # virgola decimale (IT)
            else:
                s = s.replace(',', '')        # virgola migliaia
        elif '.' in s and re.match(r'^-?\d{1,3}(\.\d{3})+$', s):
            s = s.replace('.', '')            # punto migliaia (IT: "2.000" = duemila)
        try:
            return float(s)
        except ValueError:
            return np.nan
    return serie.map(_uno), valute

def rileva_valuta(df, ruoli, valute_valori):
    # Valuta da: colonna 'currency', simboli nei valori, nome colonna spesa.
    trovate = set(valute_valori)
    if 'currency' in ruoli:
        vals = df[ruoli['currency']].dropna().astype(str).str.strip().str.upper().unique()
        trovate |= {v for v in vals if v in CODICI_VALUTA}
        trovate |= {SIMBOLI_VALUTA[v] for v in vals if v in SIMBOLI_VALUTA}
    if 'spend' in ruoli:
        grezzo = str(ruoli['spend'])          # nome colonna NON normalizzato (per i simboli)
        trovate |= {iso for sim, iso in SIMBOLI_VALUTA.items() if sim in grezzo}
        n = norm_txt(grezzo)
        trovate |= {iso for iso in CODICI_VALUTA if iso.lower() in n.split()}
        if 'eur' in n.split() or 'euro' in n:
            trovate.add('EUR')
    return trovate

# ---------------------------------------------------------------------------
# 3.4 Parsing date robusto (multi-formato, dayfirst IT, ISO week, seriali Excel)
# ---------------------------------------------------------------------------
_FORMATI_DATA = ['%d/%m/%Y', '%d-%m-%Y', '%d.%m.%Y', '%Y-%m-%d', '%Y/%m/%d',
                 '%d/%m/%y', '%Y%m%d', '%d %b %Y', '%d %B %Y', '%b %d, %Y', '%Y-%m']

def parse_date_serie(serie):
    # Prova formati espliciti, poi dayfirst, poi ISO week, poi seriali Excel.
    # Sceglie il parsing che produce piu' date valide e plausibili (1990-2100).
    s = serie.astype(str).str.strip()
    # "Settimana del 03/02/2025" -> estrai la data
    s = s.str.replace(r'^\D*?(\d)', r'\1', regex=True, n=1)
    tentativi = []
    for fmt in _FORMATI_DATA:
        try:
            tentativi.append(pd.to_datetime(s, format=fmt, errors='coerce'))
        except (ValueError, TypeError):
            pass
    tentativi.append(pd.to_datetime(s, dayfirst=True, errors='coerce', format='mixed'))
    tentativi.append(pd.to_datetime(s, dayfirst=False, errors='coerce', format='mixed'))
    # ISO week "2025-W14" / "2025W14"
    m = s.str.extract(r'^(\d{4})[-_ ]?[wW](\d{1,2})$')
    if m[0].notna().sum() > 0:
        iso = pd.to_datetime(m[0] + '-W' + m[1].str.zfill(2) + '-1',
                             format='%G-W%V-%u', errors='coerce')
        tentativi.append(iso)
    # seriali Excel (numeri ~ 20000-60000)
    num = pd.to_numeric(s, errors='coerce')
    if num.between(20000, 60000).any():
        tentativi.append(pd.to_datetime(num.where(num.between(20000, 60000)),
                                        unit='D', origin='1899-12-30', errors='coerce'))
    def _plausibili(t):
        return t.where((t > pd.Timestamp('1990-01-01')) & (t < pd.Timestamp('2100-01-01')))
    # parte dal tentativo con piu' date valide e riempie i buchi con gli altri
    # (gestisce colonne con formati misti, es. date + settimane ISO)
    tentativi = sorted((_plausibili(t) for t in tentativi),
                       key=lambda t: int(t.notna().sum()), reverse=True)
    migliore = tentativi[0]
    for t in tentativi[1:]:
        migliore = migliore.fillna(t)
    return migliore

def rileva_cadenza(date_valide):
    # 'giornaliera' | 'settimanale' | 'mensile' | 'irregolare' dai delta mediani.
    u = pd.Series(sorted(date_valide.dropna().unique()))
    if len(u) < 3:
        return 'irregolare'
    delta = u.diff().dropna().dt.days.median()
    if delta <= 1.5:
        return 'giornaliera'
    if 6 <= delta <= 8:
        return 'settimanale'
    if 27 <= delta <= 32:
        return 'mensile'
    return 'irregolare'

# ---------------------------------------------------------------------------
# 3.5 Geo: rilevamento colonna, normalizzazione, livello, mappatura a regione
# ---------------------------------------------------------------------------
def normalizza_geo(v):
    n = norm_txt(v)
    return GEO_SINONIMI.get(n, n)

def classifica_geo_valori(valori):
    # Quota di valori riconosciuti come regioni / province / sigle.
    vals = [normalizza_geo(v) for v in valori if pd.notna(v)]
    vals = [v for v in vals if v]
    if not vals:
        return 'sconosciuto', 0.0
    q_reg = sum(1 for v in vals if v in REGIONI_IT) / len(vals)
    q_pro = sum(1 for v in vals if v in PROVINCIA_A_REGIONE) / len(vals)
    q_sig = sum(1 for v in vals if len(v) == 2 and v in SIGLA_A_REGIONE) / len(vals)
    livelli = {'regione': q_reg, 'provincia': q_pro, 'sigla_provincia': q_sig}
    liv = max(livelli, key=livelli.get)
    return (liv, livelli[liv]) if livelli[liv] >= 0.5 else ('custom', livelli[liv])

def rileva_geo_colonna(df, ruoli):
    # Prima la colonna col ruolo 'geo'; altrimenti cerca tra le colonne testuali
    # quella i cui valori matchano la tassonomia geografica.
    if 'geo' in ruoli:
        liv, quota = classifica_geo_valori(df[ruoli['geo']].dropna().unique()[:200])
        return ruoli['geo'], liv
    for col in df.columns:
        if df[col].dtype == object:
            liv, quota = classifica_geo_valori(df[col].dropna().unique()[:200])
            if liv != 'custom' and quota >= 0.6:
                return col, liv
    return None, None

def geo_a_livello(valore, livello_origine, livello_target):
    # Converte un valore geo al livello target (es. provincia -> regione).
    n = normalizza_geo(valore)
    if livello_target == 'nazionale':
        return 'Italia'
    if livello_origine == livello_target:
        if livello_origine == 'regione':
            return n.title()
        if livello_origine == 'sigla_provincia':
            return n.upper()
        return str(valore).strip().title() if livello_origine == 'custom' else n.title()
    if livello_target == 'regione':
        if livello_origine == 'provincia' and n in PROVINCIA_A_REGIONE:
            return PROVINCIA_A_REGIONE[n]
        if livello_origine == 'sigla_provincia' and n in SIGLA_A_REGIONE:
            return SIGLA_A_REGIONE[n]
        if n in REGIONI_IT:
            return n.title()
    return None   # non mappabile -> segnalato a monte

# ---------------------------------------------------------------------------
# 3.6 Canali: etichetta grezza -> macro-canale tramite REGOLE_CANALI
# ---------------------------------------------------------------------------
def mappa_canale(etichetta):
    n = norm_txt(etichetta)
    for regex, macro in REGOLE_CANALI:
        if re.search(regex, n, re.I):
            return macro
    return CANALE_NON_MAPPATO

def mappa_sottocanale(nome_campagna):
    # Tipo di campagna (Performance Max, Display, Retargeting...) dal nome.
    n = norm_txt(nome_campagna)
    for regex, sotto in REGOLE_SOTTOCANALE:
        if re.search(regex, n, re.I):
            return sotto
    return SOTTOCANALE_DEFAULT

def etichetta_canale_riga(df, ruoli, nome_file):
    # Costruisce l'etichetta grezza del canale: channel > source/medium > campaign > nome file.
    if 'channel' in ruoli:
        return df[ruoli['channel']].astype(str)
    if 'source' in ruoli or 'medium' in ruoli:
        s = df[ruoli['source']].astype(str) if 'source' in ruoli else ''
        m = df[ruoli['medium']].astype(str) if 'medium' in ruoli else ''
        return (s + ' / ' + m) if isinstance(s, pd.Series) else m
    if 'campaign' in ruoli:
        return df[ruoli['campaign']].astype(str)
    base = os.path.basename(nome_file).split('::')[0]
    return pd.Series([os.path.splitext(base)[0]] * len(df), index=df.index)

# ---------------------------------------------------------------------------
# 3.7 Ingestion di un singolo file -> DataFrame standardizzato + log
# ---------------------------------------------------------------------------
COLONNE_METRICHE = ['spend', 'impressions', 'clicks', 'sessions', 'kpi', 'revenue', 'population']

def ingest_file(nome, df_raw):
    log = {'file': nome, 'problemi': []}
    df, riga_header = pulisci_tabella(df_raw)
    log['header_riga'] = riga_header
    log['righe_lette'] = len(df)
    if df.empty:
        log['ruolo_file'] = 'vuoto'
        return None, log

    ruoli = mappa_ruoli_colonne(df)

    # se il file ha PIU' colonne data plausibili (es. estrazioni con date di
    # conversione, valutazione, inserimento...), la scelta di quale data
    # definisce l'osservazione/evento spetta all'utente, a runtime
    col_date = [c for c in df.columns if abbina_colonna(c)[0] == 'date']
    if len(col_date) > 1:
        default_data = col_date.index(ruoli['date']) if ruoli.get('date') in col_date else 0
        idx_data = scegli_opzione(
            nome + ": trovate piu' colonne data. Quale definisce QUANDO avviene l'evento/osservazione?",
            [str(c) for c in col_date], default=default_data)
        ruoli['date'] = col_date[idx_data]

    log['colonne_mappate'] = {r: str(c) for r, c in ruoli.items()}

    out = pd.DataFrame(index=df.index)

    # --- date ---
    if 'date' in ruoli:
        out['date'] = parse_date_serie(df[ruoli['date']])
        n_nulle = int(out['date'].isna().sum())
        if n_nulle:
            segnala(nome + ': ' + str(n_nulle) + ' date non interpretabili', 'righe scartate')
        log['cadenza'] = rileva_cadenza(out['date'])
        log['periodo'] = (str(out['date'].min().date()) if out['date'].notna().any() else None,
                          str(out['date'].max().date()) if out['date'].notna().any() else None)
    else:
        log['cadenza'] = None

    # --- metriche numeriche + valuta ---
    valute_valori = set()
    for ruolo in COLONNE_METRICHE:
        if ruolo in ruoli:
            out[ruolo], v = pulisci_numerico(df[ruoli[ruolo]])
            valute_valori |= v
    log['valute'] = sorted(rileva_valuta(df, ruoli, valute_valori))

    # --- geo ---
    geo_col, geo_liv = rileva_geo_colonna(df, ruoli)
    if geo_col is not None:
        out['geo_raw'] = df[geo_col].astype(str)
        log['geo'] = {'colonna': str(geo_col), 'livello': geo_liv}
    else:
        log['geo'] = None

    # --- canale + gerarchia fine (campagna -> sottocanale) ---
    out['canale_raw'] = etichetta_canale_riga(df, ruoli, nome)
    out['channel'] = out['canale_raw'].map(mappa_canale)
    if 'campaign' in ruoli:
        out['campagna'] = df[ruoli['campaign']].astype(str).str.strip()
    else:
        out['campagna'] = out['canale_raw'].astype(str)
    out['sottocanale'] = out['campagna'].map(mappa_sottocanale)

    # --- colonne numeriche non riconosciute -> candidate variabili di controllo ---
    riconosciute = set(ruoli.values())
    controlli = []
    for col in df.columns:
        if col in riconosciute:
            continue
        vals, _ = pulisci_numerico(df[col])
        # gli ID numerici non sono controlli: serie di INTERI quasi tutti diversi
        # su molte righe (le serie continue con decimali sono controlli legittimi)
        n_val = int(vals.notna().sum())
        interi = bool((vals.dropna() % 1 == 0).all()) if n_val else False
        quasi_unica = interi and n_val >= 30 and vals.nunique() / max(n_val, 1) > 0.8
        if vals.notna().mean() > 0.6 and vals.nunique() > 2 and not quasi_unica:
            nome_ctrl = 'ctrl_' + re.sub(r'[^a-z0-9]+', '_', norm_txt(col)).strip('_')
            out[nome_ctrl] = vals
            controlli.append(str(col) + ' -> ' + nome_ctrl)
    log['controlli_candidati'] = controlli

    # --- classificazione del ruolo del file ---
    ha_media = ('spend' in ruoli) or ('impressions' in ruoli)
    ha_kpi = ('kpi' in ruoli) or ('revenue' in ruoli)
    if 'population' in ruoli and 'date' not in ruoli:
        log['ruolo_file'] = 'popolazione'
    elif ha_media and ha_kpi:
        log['ruolo_file'] = 'media+kpi'
    elif ha_media:
        log['ruolo_file'] = 'media'
    elif ha_kpi:
        log['ruolo_file'] = 'outcome'
    elif controlli and 'date' in ruoli:
        log['ruolo_file'] = 'controllo'
    elif 'date' in ruoli:
        # file con date ma senza metriche numeriche: probabile ELENCO EVENTI
        # (una riga = una conversione/candidatura, es. estrazione DB o GA4)
        log['ruolo_file'] = 'eventi'
        log['problemi'].append('nessuna metrica numerica: se ogni riga e' + chr(39) +
                               ' un evento (es. una candidatura), al Checkpoint 1 potrai '
                               'usare il CONTEGGIO RIGHE come KPI')
    else:
        log['ruolo_file'] = 'non classificato'
        log['problemi'].append('nessuna colonna di spesa/KPI riconosciuta: '
                               'estendi SINONIMI_COLONNE se il file e' + chr(39) + ' rilevante')

    if 'date' in out.columns:
        out = out[out['date'].notna()]
    log['canali'] = sorted(out['channel'].dropna().unique().tolist()) if 'channel' in out.columns else []
    log['righe_valide'] = len(out)
    return out, log

def ingest_tutti(files_grezzi):
    # files_grezzi: dict {nome_file: bytes}. Restituisce lista di (df, log).
    risultati = []
    for nome, dati in files_grezzi.items():
        print()
        print('--- Ingestion: ' + nome + ' ---')
        try:
            tabelle = leggi_file_grezzo(nome, dati)
        except Exception as e:
            segnala(nome + ': lettura fallita (' + str(e) + ')', 'file saltato')
            continue
        for nome_tab, df_raw in tabelle.items():
            try:
                df, log = ingest_file(nome_tab, df_raw)
            except Exception as e:
                segnala(nome_tab + ': ingestion fallita (' + str(e) + ')', 'tabella saltata')
                continue
            print('  ruolo=' + log.get('ruolo_file', '?') +
                  ' | righe=' + str(log.get('righe_valide', 0)) +
                  ' | cadenza=' + str(log.get('cadenza')) +
                  ' | valute=' + str(log.get('valute')) +
                  ' | geo=' + str(log.get('geo')))
            if log.get('colonne_mappate'):
                print('  colonne: ' + json.dumps(log['colonne_mappate'], ensure_ascii=False))
            for p in log.get('problemi', []):
                print('  [!] ' + p)
            if df is not None and len(df):
                risultati.append((df, log))
    return risultati

print('Funzioni di ingestion pronte.')

## 4. Upload dei dati

**Cosa caricare in questa cella** (formati: csv, tsv, txt, xlsx, json; lingue IT/EN;
granularita' giornaliera o settimanale; geo se disponibile):

1. **Export media per canale/piattaforma** (Google Ads, Meta, LinkedIn, Indeed, ...):
   colonne attese (nomi liberi): data, spesa, impression e/o click, geo (regione/provincia/citta'),
   eventualmente campagna o source/medium. Un file per piattaforma o un unico file: indifferente.
2. **Export KPI** (da GA o estrazione DB): data, geo, conversioni/candidature
   (ed eventualmente revenue/fatturato). Preferisci una fonte NON di piattaforma
   (le conversioni tracciate dalle piattaforme sono attribuite, non incrementali).
3. *(Opzionale)* **Variabili di controllo**: serie con data (+ geo) es. domanda organica,
   vacancies, indici stagionali.
4. *(Opzionale)* **Population per geo**: colonne geo, popolazione. Se manca, verra' chiesta
   una proxy piu' avanti.

**NON caricare qui il file ROAS benchmark**: ha una cella dedicata (sezione 11).

In [ ]:
# ============================================================================
# CELLA 4 - INPUT DATI: upload, cartella Drive, o cartella gia' sulla macchina
# ============================================================================
FILES_GREZZI = {}
ORIGINE_DATI = None

# Estensioni che contengono dati. Tutto il resto (.zip, .ipynb, .py, .pkl,
# .png, .md, .txt di log...) viene IGNORATO: provare a leggerlo riempirebbe il
# data quality report di errori che non riguardano i dati.
ESTENSIONI_DATI = ('.csv', '.tsv', '.txt', '.xlsx', '.xls', '.json')


def _e_file_dati(nome):
    return os.path.splitext(nome)[1].lower() in ESTENSIONI_DATI


def _leggi_cartella(cartella):
    # Legge i soli file dati di una cartella (non ricorsivo).
    presi, ignorati = {}, []
    for nome in sorted(os.listdir(cartella)):
        percorso = os.path.join(cartella, nome)
        if not os.path.isfile(percorso):
            continue
        if not _e_file_dati(nome):
            ignorati.append(nome)
            continue
        with open(percorso, 'rb') as f:
            presi[nome] = f.read()
    return presi, ignorati


modo = scegli_opzione(
    'Come vuoi fornire i file?',
    ['Upload manuale (files.upload)',
     'Cartella su Google Drive',
     'Percorso di una cartella o di un file presente sulla macchina'],
    default=0)

_ignorati = []
if modo == 0:
    from google.colab import files as colab_files
    print('Seleziona TUTTI i file dati (media, KPI, controlli, population)...')
    caricati = colab_files.upload()
    for nome, dati in caricati.items():
        if _e_file_dati(nome):
            FILES_GREZZI[nome] = dati
        else:
            _ignorati.append(nome)
    ORIGINE_DATI = 'upload manuale'
elif modo == 1:
    from google.colab import drive
    drive.mount('/content/drive')
    cartella = input('Percorso cartella Drive (es. /content/drive/MyDrive/mmm_dati): ').strip()
    FILES_GREZZI, _ignorati = _leggi_cartella(cartella)
    ORIGINE_DATI = cartella
else:
    # Nessun mount e nessun upload: serve quando i dati stanno GIA' sulla
    # macchina - zip scompattato in /content, repo clonato, VM aziendale.
    # E' anche il rimedio al motivo per cui questa cella veniva saltata.
    percorso = input(
        'Percorso della cartella o del file (es. /content/mmm/dati_simulati/modello): '
    ).strip().strip('"').strip(chr(39))
    assert os.path.exists(percorso), 'Percorso inesistente: ' + percorso
    if os.path.isdir(percorso):
        FILES_GREZZI, _ignorati = _leggi_cartella(percorso)
    else:
        assert _e_file_dati(percorso), ('Estensione non riconosciuta come dati: ' +
                                        percorso)
        with open(percorso, 'rb') as f:
            FILES_GREZZI[os.path.basename(percorso)] = f.read()
    ORIGINE_DATI = percorso

assert FILES_GREZZI, ('Nessun file dati trovato (origine: ' + str(ORIGINE_DATI) +
                      '). Estensioni accettate: ' + ', '.join(ESTENSIONI_DATI))
print()
print(str(len(FILES_GREZZI)) + ' file dati: ' + ', '.join(FILES_GREZZI.keys()))
if _ignorati:
    print(str(len(_ignorati)) + ' file ignorati perche' + chr(39) + ' non dati: ' +
          ', '.join(_ignorati))
print('Origine dati: ' + str(ORIGINE_DATI))


## 5. Ingestion + CHECKPOINT 1 (config auto-rilevata)

Esegue l'ingestion su tutti i file, deduce la configurazione (periodo, granularita', geo,
valuta, KPI, mappatura canali, population) e si ferma per la tua conferma.
Se qualcosa non torna: estendi i dizionari nella cella CONFIG e riesegui da li'.

In [ ]:
# ============================================================================
# CELLA 5 - INGESTION DI TUTTI I FILE + AUTO-CONFIG + CHECKPOINT 1
# ============================================================================
INGEST = ingest_tutti(FILES_GREZZI)
assert INGEST, 'Nessuna tabella valida dopo la ingestion: controlla i log sopra.'

DF_MEDIA_LIST = [(df, log) for df, log in INGEST if log['ruolo_file'] in ('media', 'media+kpi')]
DF_OUTCOME_LIST = [(df, log) for df, log in INGEST if log['ruolo_file'] == 'outcome']
DF_CTRL_LIST = [(df, log) for df, log in INGEST if log['ruolo_file'] == 'controllo']
DF_POP_LIST = [(df, log) for df, log in INGEST if log['ruolo_file'] == 'popolazione']

assert DF_MEDIA_LIST, 'Nessun file media riconosciuto (serve almeno spesa o impression).'

# ---------------------------------------------------------------------------
# 5.1 Scelta del KPI tra i candidati rilevati
# ---------------------------------------------------------------------------
candidati_kpi = []
for df, log in DF_OUTCOME_LIST:
    if 'kpi' in df.columns:
        candidati_kpi.append((df, log, 'kpi',
            log['file'] + ' :: ' + log['colonne_mappate'].get('kpi', 'kpi') + ' (fonte outcome)'))
for df, log in DF_MEDIA_LIST:
    if 'kpi' in df.columns:
        candidati_kpi.append((df, log, 'kpi',
            log['file'] + ' :: ' + log['colonne_mappate'].get('kpi', 'kpi') +
            ' (ATTENZIONE: conversioni di piattaforma = attribuite)'))
# file "elenco eventi" (una riga = una conversione, es. estrazione DB/GA):
# il KPI e' il conteggio delle righe per data (e geo). Vale per qualunque file
# con date ma senza una colonna KPI numerica (anche se contiene ricavi o altro)
for df, log in INGEST:
    if ('date' in df.columns and 'kpi' not in df.columns and
            log['ruolo_file'] in ('eventi', 'controllo', 'outcome', 'non classificato')):
        candidati_kpi.append((df.assign(kpi=1.0), log, 'conteggio',
            log['file'] + ' :: conteggio righe (1 riga = 1 evento/conversione)'))
assert candidati_kpi, ('Nessuna colonna KPI (conversioni/candidature) trovata: '
                       'estendi SINONIMI_COLONNE["kpi"] e riesegui.')

if isinstance(KPI, str) and KPI != 'auto':
    idx_kpi = next((i for i, c in enumerate(candidati_kpi) if KPI.lower() in c[3].lower()), 0)
elif len(candidati_kpi) == 1:
    idx_kpi = 0
else:
    # Il menu puo' arrivare a una decina di voci e la scelta giusta si perde
    # nella lista: queste righe la indicano prima di mostrarlo.
    print()
    print('QUALE SCEGLIERE: la voce marcata "(fonte outcome)", cioe' + chr(39) + ' il KPI')
    print('OSSERVATO e deduplicato (estrazione database o GA). Il default [0] e' + chr(39) + ' quella.')
    print('PERCHE' + chr(39) + ' LE ALTRE ESISTONO: l' + chr(39) + 'ingestion e' + chr(39) + ' generica e non sa in anticipo cosa')
    print('riceve; per un inserzionista senza sistema interno di tracciamento l' + chr(39) + 'unico')
    print('dato disponibile sarebbe proprio quello di piattaforma, che pero' + chr(39) + ' e' + chr(39) + '')
    print('ATTRIBUITO dalla piattaforma stessa e non deduplicato fra piattaforme.')
    idx_kpi = scegli_opzione('Piu' + chr(39) + ' candidati KPI trovati: quale usare come outcome del modello?',
                             [c[3] for c in candidati_kpi], default=0)
KPI_DF, KPI_LOG, KPI_TIPO, KPI_DESCR = candidati_kpi[idx_kpi]
if 'piattaforma' in KPI_DESCR:
    segnala('KPI da fonte di piattaforma (attribuito)', 'usato con riserva: preferire GA/DB')

# ---------------------------------------------------------------------------
# 5.2 Periodo: intersezione tra fonti chiave (media e outcome) vs unione
# ---------------------------------------------------------------------------
def _range(lista):
    mins, maxs = [], []
    for df, log in lista:
        if 'date' not in df.columns or not df['date'].notna().any():
            continue
        d_min, d_max = df['date'].min(), df['date'].max()
        # una riga mensile/settimanale copre fino a fine mese/settimana:
        # senza questa correzione la fonte piu' aggregata troncherebbe la finestra
        if log.get('cadenza') == 'mensile':
            d_max = d_max + pd.offsets.MonthEnd(0)
        elif log.get('cadenza') == 'settimanale':
            d_max = d_max + pd.Timedelta(days=6)
        mins.append(d_min)
        maxs.append(d_max)
    return (min(mins), max(maxs)), (max(mins), min(maxs))

(unione_min, unione_max), (inter_min, inter_max) = _range(DF_MEDIA_LIST + [(KPI_DF, KPI_LOG)])
if isinstance(PERIODO, (tuple, list)):
    inter_min, inter_max = pd.Timestamp(PERIODO[0]), pd.Timestamp(PERIODO[1])
assert inter_min < inter_max, 'Le fonti media e KPI non hanno un periodo in comune.'

# ---------------------------------------------------------------------------
# 5.3 Livello geo comune (il piu' fine condiviso da tutte le fonti chiave)
# ---------------------------------------------------------------------------
livelli = []
for df, log in DF_MEDIA_LIST + [(KPI_DF, KPI_LOG)]:
    livelli.append(log['geo']['livello'] if log.get('geo') else 'nazionale')

if isinstance(GEO, str) and GEO != 'auto':
    GEO_LIVELLO = GEO
elif 'nazionale' in livelli:
    GEO_LIVELLO = 'nazionale'
    if len(set(livelli)) > 1:
        segnala("almeno una fonte chiave (media o KPI) non ha una colonna geo",
                "modello degradato a nazionale")
else:
    unici = set(livelli)
    if len(unici) == 1:
        GEO_LIVELLO = livelli[0]     # tutte le fonti allo stesso livello (anche custom)
    elif unici <= {'provincia', 'sigla_provincia', 'regione'}:
        GEO_LIVELLO = 'regione'      # livello comune piu' fine allineabile
        segnala("fonti con livelli geo diversi: " + str(sorted(unici)),
                "aggregazione al livello comune: regione")
    else:
        segnala("mix di tassonomie geo non allineabili (custom + standard)",
                "degradato a nazionale: estendi GEO_SINONIMI per allineare i nomi")
        GEO_LIVELLO = 'nazionale'

# ---------------------------------------------------------------------------
# 5.4 Valute rilevate
# ---------------------------------------------------------------------------
VALUTE_TROVATE = set()
for df, log in INGEST:
    VALUTE_TROVATE |= set(log.get('valute', []))
if isinstance(VALUTA, str) and VALUTA != 'auto':
    VALUTA_TARGET = VALUTA
if not VALUTE_TROVATE:
    segnala('nessuna valuta rilevata nei file', 'assumo ' + VALUTA_TARGET)
    VALUTE_TROVATE = {VALUTA_TARGET}

# ---------------------------------------------------------------------------
# 5.5 Mappatura canali e cadenze
# ---------------------------------------------------------------------------
mappa_canali_vista = {}
for df, log in DF_MEDIA_LIST:
    for grezzo, macro in df.groupby('canale_raw')['channel'].first().items():
        mappa_canali_vista[str(grezzo)[:60]] = macro
non_mappati = sorted({g for g, m in mappa_canali_vista.items() if m == CANALE_NON_MAPPATO})
if non_mappati:
    segnala('etichette canale non mappate: ' + '; '.join(non_mappati[:10]),
            'finiscono in "' + CANALE_NON_MAPPATO + '" (estendi REGOLE_CANALI per separarle)')

# ---------------------------------------------------------------------------
# 5.5-bis ACCORPAMENTO DEI CANALI - avviso in evidenza.
# Piu' etichette grezze che finiscono sullo stesso canale di modello cambiano
# il risultato SENZA generare alcun errore: il modello stima un effetto solo
# dove ce ne sarebbero due o piu'. E' il tipo di problema che si scopre solo
# rileggendo questo checkpoint riga per riga, quindi va reso impossibile da
# non vedere.
# ---------------------------------------------------------------------------
CANALI_CP1 = sorted({m for m in mappa_canali_vista.values() if not str(m).startswith('_')})
_grezzi_per_canale = {}
for _g, _m in mappa_canali_vista.items():
    if not str(_m).startswith('_'):
        _grezzi_per_canale.setdefault(_m, []).append(_g)
ACCORPATI = {m: sorted(gs) for m, gs in _grezzi_per_canale.items() if len(gs) > 1}
MENO_CANALI_CHE_FILE = len(CANALI_CP1) < len(DF_MEDIA_LIST)

if ACCORPATI or MENO_CANALI_CHE_FILE:
    print()
    print('!' * 76)
    print('!!!' + 'AVVISO: ACCORPAMENTO DEI CANALI'.center(70) + '!!!')
    print('!' * 76)
    print('Canali di modello: ' + str(len(CANALI_CP1)) +
          '    |    file media in ingresso: ' + str(len(DF_MEDIA_LIST)))
    print('Canali: ' + ', '.join(CANALI_CP1))
    print()
    for _m, _gs in sorted(ACCORPATI.items()):
        print('  ' + str(len(_gs)) + ' etichette grezze diverse -> UN SOLO canale "' + _m + '":')
        for _g in _gs:
            print('        ' + _g)
    if MENO_CANALI_CHE_FILE and not ACCORPATI:
        print('  I canali di modello sono MENO dei file media in ingresso.')
    print()
    print('Conseguenza: il modello stima UN SOLO effetto per ogni canale accorpato.')
    print('I canali fusi non avranno un ROAS proprio e non saranno allocabili')
    print('separatamente dall' + chr(39) + 'ottimizzatore.')
    print('Se non e' + chr(39) + ' voluto, aggiungi una regola in REGOLE_CANALI.')
    print('!' * 76)
    print()
    segnala('accorpamento canali: ' + str(len(CANALI_CP1)) + ' canali di modello da ' +
            str(len(DF_MEDIA_LIST)) + ' file media' +
            (' (' + '; '.join(sorted(ACCORPATI)) + ')' if ACCORPATI else ''),
            'da confermare al Checkpoint 1: se non voluto, estendere REGOLE_CANALI')

cadenze = sorted({log['cadenza'] for _, log in DF_MEDIA_LIST + [(KPI_DF, KPI_LOG)] if log.get('cadenza')})
if 'mensile' in cadenze:
    segnala('almeno una fonte e' + chr(39) + ' MENSILE: Meridian lavora a settimane',
            'le fonti mensili verranno ripartite uniformemente sulle settimane (approssimazione)')

ha_population = bool(DF_POP_LIST) or any('population' in df.columns for df, _ in INGEST)

CONFIG = {
    'periodo': (str(inter_min.date()), str(inter_max.date())),
    'periodo_unione': (str(unione_min.date()), str(unione_max.date())),
    'granularita': cadenze,
    'geo_livello': GEO_LIVELLO,
    'valute_trovate': sorted(VALUTE_TROVATE),
    'valuta_target': VALUTA_TARGET,
    'kpi': KPI_DESCR,
    'population': 'presente' if ha_population else 'ASSENTE (verra' + chr(39) + ' chiesta una proxy)',
}

righe_cp1 = [
    'Finestra di modellazione (INTERSEZIONE fonti chiave): ' + CONFIG['periodo'][0] + ' -> ' + CONFIG['periodo'][1],
    'Unione dei periodi (per confronto): ' + CONFIG['periodo_unione'][0] + ' -> ' + CONFIG['periodo_unione'][1],
    'Granularita' + chr(39) + ' rilevate: ' + str(cadenze) + ' -> aggregazione a settimana ISO (lunedi' + chr(39) + ')',
    'Livello geo del modello: ' + GEO_LIVELLO,
    'Valute trovate: ' + str(sorted(VALUTE_TROVATE)) + ' -> target ' + VALUTA_TARGET,
    'KPI scelto: ' + KPI_DESCR,
    'Population: ' + CONFIG['population'],
    'File media: ' + str(len(DF_MEDIA_LIST)) + ' | outcome: ' + str(len(DF_OUTCOME_LIST)) +
    ' | eventi: ' + str(len([1 for _, l in INGEST if l['ruolo_file'] == 'eventi'])) +
    ' | controlli: ' + str(len(DF_CTRL_LIST)) + ' | population: ' + str(len(DF_POP_LIST)),
    'Canali di modello: ' + str(len(CANALI_CP1)) + ' (' + ', '.join(CANALI_CP1) + ')' +
    ('   <<< ACCORPAMENTO: vedi avviso sopra' if (ACCORPATI or MENO_CANALI_CHE_FILE) else ''),
    'Mappatura canali (grezzo -> macro):',
] + ['    ' + g + '  ->  ' + m for g, m in sorted(mappa_canali_vista.items(), key=lambda x: x[1])]

checkpoint('1. Configurazione auto-rilevata', righe_cp1)

## 6. Harmonization + CHECKPOINT 2

Costruisce la base tidy `geo x settimana ISO x canale`: valuta unica, tassonomia geo unica,
griglia settimanale continua, dedup, distinzione 0 (nessuna attivita') vs NaN (dato mancante).

In [ ]:
# ============================================================================
# CELLA 6 - HARMONIZATION -> base tidy geo x settimana x canale
# ============================================================================

def a_settimana(serie_date):
    # Lunedi' della settimana ISO.
    return serie_date - pd.to_timedelta(serie_date.dt.weekday, unit='D')

# ---------------------------------------------------------------------------
# 6.1 Tassi di cambio: chiesti a runtime solo se servono
# ---------------------------------------------------------------------------
for v in sorted(VALUTE_TROVATE - {VALUTA_TARGET}):
    if v not in TASSI_CAMBIO:
        t = chiedi_numero("Tasso di cambio 1 " + v + " -> " + VALUTA_TARGET +
                          " (es. 0.92); vuoto = 1.0:", default=1.0)
        TASSI_CAMBIO[v] = float(t if t else 1.0)
        segnala("conversione valuta " + v + " -> " + VALUTA_TARGET,
                "tasso " + str(TASSI_CAMBIO[v]))

def _converti_valuta(df, log, colonne):
    valute = [v for v in log.get('valute', []) if v != VALUTA_TARGET]
    if not valute:
        return df
    if len(valute) > 1:
        segnala(log['file'] + ": piu' valute nello stesso file", "uso il tasso della prima: " + valute[0])
    tasso = TASSI_CAMBIO.get(valute[0], 1.0)
    for c in colonne:
        if c in df.columns:
            df[c] = df[c] * tasso
    return df

# ---------------------------------------------------------------------------
# 6.2 Geo al livello comune + espansione delle fonti mensili
# ---------------------------------------------------------------------------
def _applica_geo(df, log):
    liv = log['geo']['livello'] if log.get('geo') else 'nazionale'
    if GEO_LIVELLO == 'nazionale' or liv == 'nazionale':
        df = df.copy(); df['geo'] = 'Italia'
        return df
    df = df.copy()
    df['geo'] = df['geo_raw'].map(lambda v: geo_a_livello(v, liv, GEO_LIVELLO))
    n_persi = int(df['geo'].isna().sum())
    if n_persi:
        esempi = sorted(df.loc[df['geo'].isna(), 'geo_raw'].astype(str).unique()[:5])
        segnala(log['file'] + ": " + str(n_persi) + " righe con geo non mappabile a livello " +
                GEO_LIVELLO + " (es. " + ", ".join(esempi) + ")",
                "scartate: estendi GEO_SINONIMI/PROVINCIA_A_REGIONE per recuperarle")
        df = df[df['geo'].notna()]
    return df

_FLUSSI = ['spend', 'impressions', 'clicks', 'sessions', 'kpi', 'revenue']

def _espandi_mensile(df, log):
    # Ripartisce i valori mensili sulle settimane del mese (flussi divisi,
    # livelli come i controlli ripetuti). Approssimazione dichiarata al CP1.
    if log.get('cadenza') != 'mensile':
        return df
    righe = []
    for _, r in df.iterrows():
        inizio = pd.Timestamp(r['date']).replace(day=1)
        fine = inizio + pd.offsets.MonthEnd(0)
        settimane = pd.date_range(a_settimana(pd.Series([inizio]))[0], fine, freq='7D')
        settimane = [s for s in settimane if s.month == inizio.month or s + pd.Timedelta(days=6) >= inizio]
        for s in settimane:
            r2 = r.copy(); r2['date'] = s
            for c in _FLUSSI:
                if c in df.columns and pd.notna(r2.get(c)):
                    r2[c] = r2[c] / len(settimane)
            righe.append(r2)
    return pd.DataFrame(righe)

def harmonize():
    # --- media (paid + organico) ---
    media_parti, organico_parti = [], []
    for df, log in DF_MEDIA_LIST:
        d = _applica_geo(df, log)
        d = _converti_valuta(d, log, ['spend', 'revenue'])
        d = _espandi_mensile(d, log)
        d = d[(d['date'] >= inter_min) & (d['date'] <= inter_max)]
        d['week'] = a_settimana(d['date'])
        paid = d[~d['channel'].isin(['_ORGANICO_', '_ESCLUDI_'])]
        media_parti.append(paid)
        org = d[d['channel'] == '_ORGANICO_']
        if len(org):
            organico_parti.append(org)
        n_escl = int((d['channel'] == '_ESCLUDI_').sum())
        if n_escl:
            segnala(log['file'] + ": " + str(n_escl) + " righe direct/none", "escluse dal media")
    media = pd.concat(media_parti, ignore_index=True)

    metriche = [c for c in ['spend', 'impressions', 'clicks', 'sessions'] if c in media.columns]
    tidy = media.groupby(['geo', 'week', 'channel'], as_index=False)[metriche].sum(min_count=1)
    # vista fine (descrittiva): settimana x canale x sottocanale x campagna
    campagne = media.groupby(['week', 'channel', 'sottocanale', 'campagna'],
                             as_index=False)[metriche].sum(min_count=1)
    # duplicati esatti: conteggio informativo (le somme li gestiscono comunque)
    dup_esatti = int(media.duplicated().sum())
    if dup_esatti:
        segnala(str(dup_esatti) + " righe media duplicate esatte", "aggregate per somma")

    # --- outcome (KPI dalla fonte scelta al CP1) ---
    d = _applica_geo(KPI_DF, KPI_LOG)
    d = _converti_valuta(d, KPI_LOG, ['revenue'])
    d = _espandi_mensile(d, KPI_LOG)
    d = d[(d['date'] >= inter_min) & (d['date'] <= inter_max)]
    d['week'] = a_settimana(d['date'])
    agg_kpi = {'kpi': 'sum'}
    if 'revenue' in d.columns:
        agg_kpi['revenue'] = 'sum'
    base = d.groupby(['geo', 'week'], as_index=False).agg(agg_kpi)

    # --- controlli (file dedicati + colonne ctrl_ ovunque) ---
    # I controlli senza geo (nazionali) vengono propagati a tutti i geo del modello.
    # NB: se il KPI e' un CONTEGGIO di eventi, le colonne numeriche di quel file
    # sono attributi dei singoli record (codici, importi), NON serie di controllo.
    fonti_ctrl = DF_CTRL_LIST + DF_MEDIA_LIST
    if KPI_TIPO != 'conteggio':
        fonti_ctrl = fonti_ctrl + [(KPI_DF, KPI_LOG)]
    ctrl_geo_parti, ctrl_naz_parti = [], []
    for df, log in fonti_ctrl:
        col_ctrl = [c for c in df.columns if c.startswith('ctrl_')]
        if not col_ctrl or 'date' not in df.columns:
            continue
        d = _applica_geo(df, log)
        d = _espandi_mensile(d, log)
        d = d[(d['date'] >= inter_min) & (d['date'] <= inter_max)]
        d['week'] = a_settimana(d['date'])
        # i controlli sono livelli/indici -> media settimanale
        agg = d.groupby(['geo', 'week'], as_index=False)[col_ctrl].mean()
        if GEO_LIVELLO != 'nazionale' and set(agg['geo'].unique()) == {'Italia'}:
            segnala("controllo nazionale " + ", ".join(col_ctrl),
                    "propagato identico a tutti i geo (varia solo nel tempo)")
            ctrl_naz_parti.append(agg.drop(columns='geo').groupby('week', as_index=False).mean())
        else:
            ctrl_geo_parti.append(agg)
    controlli, controlli_naz = None, None
    if ctrl_geo_parti:
        controlli = ctrl_geo_parti[0]
        for parte in ctrl_geo_parti[1:]:
            controlli = controlli.merge(parte, on=['geo', 'week'], how='outer')
    if ctrl_naz_parti:
        controlli_naz = ctrl_naz_parti[0]
        for parte in ctrl_naz_parti[1:]:
            controlli_naz = controlli_naz.merge(parte, on='week', how='outer')

    # --- organico (impression senza spesa) ---
    organico = None
    if organico_parti:
        org = pd.concat(organico_parti, ignore_index=True)
        col_org = 'impressions' if 'impressions' in org.columns else (
                  'clicks' if 'clicks' in org.columns else None)
        if col_org:
            organico = org.groupby(['geo', 'week'], as_index=False)[[col_org]].sum()
            organico = organico.rename(columns={col_org: 'organico'})

    return tidy, base, controlli, controlli_naz, organico, campagne

DF_TIDY, DF_BASE, DF_CTRL, DF_CTRL_NAZ, DF_ORGANICO, DF_CAMPAGNE = harmonize()

# ---------------------------------------------------------------------------
# 6.3 Griglia settimanale continua geo x week (0 = nessuna attivita' sui media;
#     NaN = dato mancante sul KPI, gestito sotto)
# ---------------------------------------------------------------------------
GEOS = sorted(set(DF_TIDY['geo']) | set(DF_BASE['geo']))
SETTIMANE = pd.date_range(a_settimana(pd.Series([inter_min]))[0],
                          a_settimana(pd.Series([inter_max]))[0], freq='7D')
CANALI = sorted(DF_TIDY['channel'].unique())

griglia = pd.MultiIndex.from_product([GEOS, SETTIMANE, CANALI],
                                     names=['geo', 'week', 'channel']).to_frame(index=False)
DF_TIDY = griglia.merge(DF_TIDY, on=['geo', 'week', 'channel'], how='left')
for c in ['spend', 'impressions', 'clicks', 'sessions']:
    if c in DF_TIDY.columns:
        DF_TIDY[c] = DF_TIDY[c].fillna(0.0)          # 0 = nessuna attivita'
        neg = int((DF_TIDY[c] < 0).sum())
        if neg:
            segnala(str(neg) + " valori negativi in " + c, "portati a 0")
            DF_TIDY[c] = DF_TIDY[c].clip(lower=0)

griglia_base = pd.MultiIndex.from_product([GEOS, SETTIMANE],
                                          names=['geo', 'week']).to_frame(index=False)
DF_BASE = griglia_base.merge(DF_BASE, on=['geo', 'week'], how='left')

# --- gestione KPI mancante (0 e NaN NON sono la stessa cosa) ---
n_kpi_nan = int(DF_BASE['kpi'].isna().sum())
if n_kpi_nan:
    quota = round(100 * n_kpi_nan / len(DF_BASE), 1)
    scelta = scegli_opzione(
        "KPI mancante in " + str(n_kpi_nan) + " celle geo x settimana (" + str(quota) + "%). Come gestirlo?",
        ["Interpolazione lineare per geo (consigliata se buchi sparsi)",
         "Imputa 0 (solo se mancante = davvero zero conversioni)",
         "Escludi i geo con piu' del 20% di settimane mancanti, interpola il resto"],
        default=0)
    if scelta == 2:
        quote_geo = DF_BASE.groupby('geo')['kpi'].apply(lambda s: s.isna().mean())
        geo_out = sorted(quote_geo[quote_geo > 0.20].index)
        if geo_out:
            segnala("geo esclusi per KPI troppo lacunoso: " + ", ".join(geo_out), "rimossi dal modello")
            GEOS = [g for g in GEOS if g not in geo_out]
            DF_BASE = DF_BASE[DF_BASE['geo'].isin(GEOS)]
            DF_TIDY = DF_TIDY[DF_TIDY['geo'].isin(GEOS)]
    if scelta in (0, 2):
        DF_BASE['kpi'] = DF_BASE.groupby('geo')['kpi'].transform(
            lambda s: s.interpolate(limit_direction='both'))
        segnala("KPI mancante", "interpolazione lineare per geo")
    else:
        DF_BASE['kpi'] = DF_BASE['kpi'].fillna(0.0)
        segnala("KPI mancante", "imputato 0 su scelta utente")

# --- controlli: merge + riempimento conservativo ---
CONTROLLI = []
if DF_CTRL is not None:
    DF_BASE = DF_BASE.merge(DF_CTRL, on=['geo', 'week'], how='left')
if DF_CTRL_NAZ is not None:
    DF_BASE = DF_BASE.merge(DF_CTRL_NAZ, on='week', how='left')
if DF_ORGANICO is not None:
    DF_BASE = DF_BASE.merge(DF_ORGANICO, on=['geo', 'week'], how='left')
    DF_BASE['organico'] = DF_BASE['organico'].fillna(0.0)
for c in [c for c in DF_BASE.columns if c.startswith('ctrl_')]:
    n_nan = int(DF_BASE[c].isna().sum())
    if n_nan > len(DF_BASE) * 0.5:
        segnala("controllo " + c + " mancante in >50% delle celle", "scartato")
        DF_BASE = DF_BASE.drop(columns=[c])
        continue
    if n_nan:
        DF_BASE[c] = DF_BASE.groupby('geo')[c].transform(
            lambda s: s.interpolate(limit_direction='both'))
        DF_BASE[c] = DF_BASE[c].fillna(DF_BASE[c].median())
        segnala("controllo " + c + ": " + str(n_nan) + " NaN", "interpolati")
    CONTROLLI.append(c)

# --- population per geo (se disponibile nei dati) ---
POP_SERIE = None
pop_fonti = DF_POP_LIST + [(df, log) for df, log in INGEST if 'population' in df.columns]
if pop_fonti and GEO_LIVELLO != 'nazionale':
    df_p, log_p = pop_fonti[0]
    liv_p = log_p['geo']['livello'] if log_p.get('geo') else 'nazionale'
    if log_p.get('geo'):
        d = df_p.copy()
        d['geo'] = d['geo_raw'].map(lambda v: geo_a_livello(v, liv_p, GEO_LIVELLO))
        POP_SERIE = d.groupby('geo')['population'].sum()
        POP_SERIE = POP_SERIE.reindex(GEOS)

# ---------------------------------------------------------------------------
# 6.4 Data quality report + CHECKPOINT 2
# ---------------------------------------------------------------------------
print()
print("---- DATA QUALITY REPORT (scelte e problemi) ----")
for r in DQ_REPORT:
    print("  * " + r['problema'] + (("  ->  " + r['scelta']) if r['scelta'] else ""))

spesa_canale = DF_TIDY.groupby('channel')['spend'].sum().round(0)
righe_cp2 = [
    "Base tidy: " + str(len(DF_TIDY)) + " righe = " + str(len(GEOS)) + " geo x " +
    str(len(SETTIMANE)) + " settimane x " + str(len(CANALI)) + " canali",
    "Geo (" + GEO_LIVELLO + "): " + ", ".join(GEOS[:12]) + (" ..." if len(GEOS) > 12 else ""),
    "Settimane: " + str(SETTIMANE[0].date()) + " -> " + str(SETTIMANE[-1].date()),
    "Spesa totale per canale (" + VALUTA_TARGET + "): " +
    "; ".join(c + "=" + format(int(v), ",") for c, v in spesa_canale.items()),
    "KPI totale nel periodo: " + format(int(DF_BASE['kpi'].sum()), ","),
    "Variabili di controllo tenute: " + (", ".join(CONTROLLI) if CONTROLLI else "nessuna"),
    "Vista fine: " + str(DF_CAMPAGNE['campagna'].nunique()) + " campagne in " +
    str(DF_CAMPAGNE['sottocanale'].nunique()) + " sottocanali (" +
    ", ".join(sorted(DF_CAMPAGNE['sottocanale'].unique())) + ")",
    "Organico: " + ("presente" if DF_ORGANICO is not None else "assente"),
    "Convenzione: media=0 significa nessuna attivita'; i NaN del KPI sono stati gestiti come sopra",
]
checkpoint("2. Base armonizzata", righe_cp2)

## 7. `InputData` Meridian + CHECKPOINT 3

Pivot al formato wide richiesto da `DataFrameDataLoader` (una riga per geo x settimana),
scelta della metrica di esecuzione media per canale (impression > click > spesa come proxy),
population e `revenue_per_kpi` opzionale.

In [ ]:
# ============================================================================
# CELLA 7 - COSTRUZIONE InputData MERIDIAN (geo) + CHECKPOINT 3
# ============================================================================

def _slug(canale):
    return re.sub(r'[^A-Za-z0-9]+', '', str(canale))

def build_input_data():
    global POP_SERIE, REVENUE_PER_KPI_PRESENTE, RPK_MEDIO

    # --- metrica di esecuzione per canale: impressions > clicks > spend(proxy) ---
    metrica_canale = {}
    for c in CANALI:
        sotto = DF_TIDY[DF_TIDY['channel'] == c]
        if 'impressions' in sotto.columns and sotto['impressions'].sum() > 0:
            metrica_canale[c] = 'impressions'
        elif 'clicks' in sotto.columns and sotto['clicks'].sum() > 0:
            metrica_canale[c] = 'clicks'
        else:
            metrica_canale[c] = 'spend'
            segnala("canale " + c + " senza impression/click",
                    "uso la spesa come metrica di esecuzione (proxy)")

    # --- pivot wide: una riga per geo x settimana ---
    wide = DF_BASE[['geo', 'week', 'kpi'] + CONTROLLI +
                   (['organico'] if 'organico' in DF_BASE.columns else []) +
                   (['revenue'] if 'revenue' in DF_BASE.columns else [])].copy()
    for c in CANALI:
        s = _slug(c)
        sotto = DF_TIDY[DF_TIDY['channel'] == c]
        piv_m = sotto.pivot(index=['geo', 'week'], columns='channel',
                            values=metrica_canale[c]).rename(columns={c: 'media_' + s})
        piv_s = sotto.pivot(index=['geo', 'week'], columns='channel',
                            values='spend').rename(columns={c: 'spend_' + s})
        wide = wide.merge(piv_m.reset_index(), on=['geo', 'week'], how='left')
        wide = wide.merge(piv_s.reset_index(), on=['geo', 'week'], how='left')
    for col in wide.columns:
        if col.startswith(('media_', 'spend_')):
            wide[col] = wide[col].fillna(0.0)

    # --- population ---
    if GEO_LIVELLO == 'nazionale':
        POP_SERIE = pd.Series(1.0, index=GEOS)
    elif POP_SERIE is None or POP_SERIE.isna().any():
        scelta = scegli_opzione(
            "Population per geo non trovata nei dati (serve al modello geo). Come procedere?",
            ["Carico ora un CSV con colonne geo,population",
             "Usa una proxy uniforme (tutti i geo uguali) - AVVISO: pesa i geo allo stesso modo"],
            default=0)
        if scelta == 0:
            from google.colab import files as colab_files
            up = colab_files.upload()
            df_p = pd.read_csv(io.BytesIO(list(up.values())[0]))
            df_p.columns = [norm_txt(c) for c in df_p.columns]
            col_g = [c for c in df_p.columns if abbina_colonna(c)[0] == 'geo'][0]
            col_p = [c for c in df_p.columns if abbina_colonna(c)[0] == 'population'][0]
            df_p['geo'] = df_p[col_g].map(lambda v: geo_a_livello(v, classifica_geo_valori(
                df_p[col_g].unique())[0], GEO_LIVELLO))
            POP_SERIE = df_p.groupby('geo')[col_p].sum().reindex(GEOS)
        else:
            POP_SERIE = pd.Series(1.0, index=GEOS)
            segnala("population mancante", "proxy uniforme accettata dall'utente")
        if POP_SERIE.isna().any():
            mancanti = sorted(POP_SERIE[POP_SERIE.isna()].index)
            segnala("population mancante per: " + ", ".join(mancanti), "imputata la mediana")
            POP_SERIE = POP_SERIE.fillna(POP_SERIE.median())
    wide['population'] = wide['geo'].map(POP_SERIE)

    # --- revenue_per_kpi (facoltativo: abilita il ROAS monetario) ---
    REVENUE_PER_KPI_PRESENTE = False
    RPK_MEDIO = None
    usa_revenue = 'revenue' in wide.columns and wide['revenue'].sum() > 0
    if usa_revenue:
        rpk_med = float((wide['revenue'] / wide['kpi'].replace(0, np.nan)).median())
        usa_revenue = scegli_opzione(
            "Trovata una colonna ricavi nella fonte KPI (valore mediano per conversione: " +
            str(round(rpk_med, 2)) + " " + VALUTA_TARGET + "). Usarla come valore monetario?",
            ["Si', abilita il ROAS monetario", "No, ottimizza le conversioni"], default=0) == 0
    if usa_revenue:
        rpk = wide['revenue'] / wide['kpi'].replace(0, np.nan)
        wide['revenue_per_kpi'] = rpk.fillna(rpk.median())
        REVENUE_PER_KPI_PRESENTE = True
        RPK_MEDIO = float(rpk.median())
        print("revenue_per_kpi derivato dalla colonna revenue (mediana: " +
              str(round(RPK_MEDIO, 2)) + " " + VALUTA_TARGET + ")")
    else:
        # Il prompt diceva solo che si puo' lasciare vuoto, non cosa si perde.
        print()
        print('VALORE MONETARIO DELLA CONVERSIONE - cosa cambia se lo lasci vuoto:')
        print('  - niente ROAS monetario: il ROAS resta in conversioni per euro speso,')
        print('    quindi non e' + chr(39) + ' un numero confrontabile con quelli aziendali;')
        print('  - niente confronto IN LIVELLO col benchmark di piattaforma, che e' + chr(39) + '')
        print('    monetario: alla cella 11 resta confrontabile solo l' + chr(39) + 'ORDINAMENTO')
        print('    dei canali, non la loro distanza;')
        print('  - il punto di pareggio non e' + chr(39) + ' riportabile in percentuale (100% =')
        print('    break-even), perche' + chr(39) + ' senza un valore il pareggio non e' + chr(39) + ' definito.')
        print('Se un valore anche solo approssimato esiste, conviene metterlo.')
        val = chiedi_numero(
            "Valore medio di una conversione in " + VALUTA_TARGET +
            " (abilita ROAS monetario e confronto diretto col benchmark)." +
            " Vuoto = ottimizza le conversioni senza valore monetario:", default=None)
        if val:
            wide['revenue_per_kpi'] = float(val)
            REVENUE_PER_KPI_PRESENTE = True
            RPK_MEDIO = float(val)
            segnala("revenue_per_kpi", "valore costante fornito a runtime: " + str(val))
        else:
            segnala("revenue_per_kpi non fornito",
                    "il modello ottimizza le CONVERSIONI; il ROAS del benchmark (monetario) " +
                    "sara' confrontabile solo come ranking, non in livello")

    wide['time'] = wide['week'].dt.strftime('%Y-%m-%d')
    wide = wide.sort_values(['geo', 'week']).reset_index(drop=True)

    # --- CoordToColumns + loader ---
    # PRINCIPIO DI AGGREGAZIONE: il modello riceve SOLO serie per CANALE
    # (una colonna di esecuzione e una di spesa per canale). Campagne e
    # tipologie NON entrano in InputData: nessun parametro del modello
    # (beta, adstock, saturazione) viene stimato a livello campagna.
    # Il dettaglio campagne vive solo nelle viste descrittive (DF_CAMPAGNE)
    # e nel riparto operativo a valle dell'ottimizzazione.
    media_cols = ['media_' + _slug(c) for c in CANALI]
    spend_cols = ['spend_' + _slug(c) for c in CANALI]
    assert len(media_cols) == len(CANALI) and len(spend_cols) == len(CANALI), \
        'violato il principio: una e una sola serie media+spesa per canale'
    kwargs_coord = dict(time='time', geo='geo', kpi='kpi', population='population',
                        media=media_cols, media_spend=spend_cols)
    if CONTROLLI:
        kwargs_coord['controls'] = CONTROLLI
    if REVENUE_PER_KPI_PRESENTE:
        kwargs_coord['revenue_per_kpi'] = 'revenue_per_kpi'
    if 'organico' in wide.columns and wide['organico'].sum() > 0:
        kwargs_coord['organic_media'] = ['organico']
    coord = load.CoordToColumns(**kwargs_coord)

    kwargs_loader = dict(
        df=wide, kpi_type='non_revenue', coord_to_columns=coord,
        media_to_channel={'media_' + _slug(c): c for c in CANALI},
        media_spend_to_channel={'spend_' + _slug(c): c for c in CANALI})
    if 'organic_media' in kwargs_coord:
        kwargs_loader['organic_media_to_channel'] = {'organico': 'Organico'}
    loader = load.DataFrameDataLoader(**kwargs_loader)
    return loader.load(), wide, metrica_canale

INPUT_DATA, DF_WIDE, METRICA_CANALE = build_input_data()

N_GEOS = len(INPUT_DATA.geo)
N_SETTIMANE = len(INPUT_DATA.time)
CANALI_MODELLO = [str(c) for c in np.array(INPUT_DATA.media_channel).tolist()]

righe_cp3 = [
    "InputData Meridian costruito (kpi_type='non_revenue')",
    "Dimensioni: " + str(N_GEOS) + " geo x " + str(N_SETTIMANE) + " settimane x " +
    str(len(CANALI_MODELLO)) + " canali media",
    "Canali (ordine del modello): " + ", ".join(CANALI_MODELLO),
    "Metrica di esecuzione per canale: " +
    "; ".join(c + "=" + m for c, m in METRICA_CANALE.items()),
    "Controlli: " + (", ".join(CONTROLLI) if CONTROLLI else "nessuno"),
    "Organic media: " + ("si'" if 'organico' in DF_WIDE.columns and DF_WIDE['organico'].sum() > 0 else "no"),
    "revenue_per_kpi: " + ("presente (ROAS monetario abilitato)" if REVENUE_PER_KPI_PRESENTE
                           else "assente (outcome = conversioni)"),
    "Population: min=" + format(int(POP_SERIE.min()), ",") + " max=" + format(int(POP_SERIE.max()), ","),
]
checkpoint("3. InputData Meridian", righe_cp3)

## 8. EDA / validazione

Copertura e missing, outlier, trend e stagionalita', spesa vs KPI, correlazioni tra canali
e VIF (collinearita': se alto, i ROAS dei singoli canali saranno identificati male),
quote di spesa. Tabelle e figure finiscono nel foglio `EDA` dell'Excel.

In [ ]:
# ============================================================================
# CELLA 8 - EDA / VALIDAZIONE
# ============================================================================
EDA_TABELLE = {}

def run_eda():
    metriche = [c for c in ['spend', 'impressions', 'clicks'] if c in DF_TIDY.columns]
    naz = DF_TIDY.groupby(['week', 'channel'], as_index=False)[metriche].sum()
    kpi_sett = DF_BASE.groupby('week', as_index=False)['kpi'].sum()

    # --- 8.1 copertura per canale ---
    cop = []
    for c in CANALI:
        s = naz[naz['channel'] == c]
        attive = (s['spend'] > 0).mean()
        cop.append({'canale': c, 'spesa_totale': round(s['spend'].sum(), 0),
                    'quota_spesa_%': round(100 * s['spend'].sum() / naz['spend'].sum(), 1),
                    'settimane_attive_%': round(100 * attive, 1),
                    'metrica_esecuzione': METRICA_CANALE[c]})
    EDA_TABELLE['copertura'] = pd.DataFrame(cop)
    print(EDA_TABELLE['copertura'].to_string(index=False))

    # --- 8.2 outlier (oltre 3 IQR dal quartile, per canale, spesa settimanale) ---
    out_righe = []
    for c in CANALI:
        s = naz.loc[naz['channel'] == c, 'spend']
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        n_out = int(((s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)).sum())
        if n_out:
            out_righe.append({'serie': 'spesa ' + c, 'n_outlier': n_out})
    s = kpi_sett['kpi']
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    n_out = int(((s < q1 - 3 * (q3 - q1)) | (s > q3 + 3 * (q3 - q1))).sum())
    if n_out:
        out_righe.append({'serie': 'KPI', 'n_outlier': n_out})
    EDA_TABELLE['outlier'] = pd.DataFrame(out_righe) if out_righe else pd.DataFrame(
        [{'serie': '-', 'n_outlier': 0}])
    for r in out_righe:
        segnala("outlier in " + r['serie'] + " (" + str(r['n_outlier']) + " settimane)",
                "tenuti nel modello, verificare nel foglio EDA")

    # --- 8.3 trend: spesa per canale (aree) + KPI (linea) ---
    fig, ax1 = plt.subplots(figsize=(12, 5))
    piv = naz.pivot(index='week', columns='channel', values='spend').fillna(0)
    ax1.stackplot(piv.index, [piv[c] for c in piv.columns], labels=list(piv.columns), alpha=0.7)
    ax1.set_ylabel('Spesa settimanale (' + VALUTA_TARGET + ')')
    ax1.legend(loc='upper left', fontsize=8)
    ax2 = ax1.twinx()
    ax2.plot(kpi_sett['week'], kpi_sett['kpi'], color='black', lw=1.5, label='KPI')
    ax2.set_ylabel('KPI (conversioni)')
    plt.title('Spesa per canale e KPI per settimana')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'eda_trend.png'), dpi=130)
    plt.show()
    EDA_TABELLE['trend'] = piv.join(kpi_sett.set_index('week')).reset_index()

    # --- 8.4 stagionalita': KPI medio per settimana ISO dell'anno ---
    st = DF_BASE.copy()
    st['iso_week'] = st['week'].dt.isocalendar().week.astype(int)
    stag = st.groupby('iso_week', as_index=False)['kpi'].mean()
    plt.figure(figsize=(10, 3.5))
    plt.plot(stag['iso_week'], stag['kpi'])
    plt.title('Stagionalita del KPI (media per settimana ISO)')
    plt.xlabel('settimana ISO'); plt.ylabel('KPI medio')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'eda_stagionalita.png'), dpi=130)
    plt.show()
    EDA_TABELLE['stagionalita'] = stag

    # --- 8.5 spesa vs KPI per canale (scatter, livello nazionale-settimana) ---
    n = len(CANALI)
    ncol = min(3, n)
    nrig = math.ceil(n / ncol)
    fig, assi = plt.subplots(nrig, ncol, figsize=(4 * ncol, 3.2 * nrig), squeeze=False)
    for i, c in enumerate(CANALI):
        ax = assi[i // ncol][i % ncol]
        s = naz[naz['channel'] == c].merge(kpi_sett, on='week')
        ax.scatter(s['spend'], s['kpi'], s=12, alpha=0.6)
        ax.set_title(c, fontsize=9)
        ax.set_xlabel('spesa'); ax.set_ylabel('KPI')
    for j in range(n, nrig * ncol):
        assi[j // ncol][j % ncol].axis('off')
    plt.suptitle('Spesa settimanale vs KPI (correlazione, non causalita)', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'eda_spesa_vs_kpi.png'), dpi=130, bbox_inches='tight')
    plt.show()

    # --- 8.6 correlazioni tra canali + VIF (collinearita') ---
    piv_c = naz.pivot(index='week', columns='channel', values='spend').fillna(0)
    corr = piv_c.join(kpi_sett.set_index('week')['kpi']).corr().round(2)
    EDA_TABELLE['correlazioni'] = corr.reset_index().rename(columns={'index': 'serie'})
    plt.figure(figsize=(1.1 * len(corr), 0.9 * len(corr)))
    plt.imshow(corr.values, vmin=-1, vmax=1, cmap='RdBu_r')
    plt.xticks(range(len(corr)), corr.columns, rotation=45, ha='right', fontsize=8)
    plt.yticks(range(len(corr)), corr.columns, fontsize=8)
    for i in range(len(corr)):
        for j in range(len(corr)):
            plt.text(j, i, str(corr.values[i, j]), ha='center', va='center', fontsize=7)
    plt.title('Correlazioni spesa canali / KPI')
    plt.colorbar(shrink=0.8)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'eda_correlazioni.png'), dpi=130)
    plt.show()

    try:
        from statsmodels.stats.outliers_influence import variance_inflation_factor
        X = piv_c.loc[:, piv_c.std() > 0].copy()
        X = (X - X.mean()) / X.std()
        X.insert(0, 'const', 1.0)
        vif = pd.DataFrame({
            'canale': X.columns[1:],
            'VIF': [round(variance_inflation_factor(X.values, i), 2)
                    for i in range(1, X.shape[1])]})
        EDA_TABELLE['vif'] = vif
        print()
        print(vif.to_string(index=False))
        alti = vif[vif['VIF'] > 5]['canale'].tolist()
        if alti:
            segnala("VIF > 5 per: " + ", ".join(map(str, alti)),
                    "collinearita' elevata: i ROAS di questi canali avranno intervalli larghi "
                    "e vanno letti con cautela (limite noto dei MMM)")
    except Exception as e:
        print("[avviso] VIF non calcolabile: " + str(e))
        EDA_TABELLE['vif'] = pd.DataFrame([{'canale': 'n/d', 'VIF': np.nan}])

    # --- 8.7 quota spesa vs quota "risposta apparente" (corr grezza, descrittiva) ---
    quote = EDA_TABELLE['copertura'][['canale', 'quota_spesa_%']].copy()
    quote['corr_con_kpi'] = [round(float(piv_c[c].corr(kpi_sett.set_index('week')['kpi'])), 2)
                             if c in piv_c.columns else np.nan for c in quote['canale']]
    EDA_TABELLE['quote'] = quote
    print()
    print(quote.to_string(index=False))

    # --- 8.8 vista AGGREGATA per gruppo e vista FINE per sottocanale ---
    grp = EDA_TABELLE['copertura'][['canale', 'spesa_totale', 'quota_spesa_%']].copy()
    grp.insert(0, 'gruppo', grp['canale'].map(lambda c: GRUPPO_CANALE.get(c, GRUPPO_DEFAULT)))
    EDA_TABELLE['gruppi'] = grp.groupby('gruppo', as_index=False)[
        ['spesa_totale', 'quota_spesa_%']].sum().round(1)
    print()
    print('Vista aggregata per gruppo:')
    print(EDA_TABELLE['gruppi'].to_string(index=False))

    sc = DF_CAMPAGNE.groupby(['channel', 'sottocanale'], as_index=False).agg(
        spesa=('spend', 'sum'), n_campagne=('campagna', 'nunique'))
    sc['quota_nel_canale_%'] = (100 * sc['spesa'] /
                                sc.groupby('channel')['spesa'].transform('sum')).round(1)
    sc['spesa'] = sc['spesa'].round(0)
    sc = sc.rename(columns={'channel': 'canale'}).sort_values(
        ['canale', 'spesa'], ascending=[True, False])
    EDA_TABELLE['sottocanali'] = sc
    print()
    print('Vista fine per sottocanale (descrittiva):')
    print(sc.to_string(index=False))

run_eda()
print()
print("EDA completata: tabelle in EDA_TABELLE, figure in " + FIG_DIR)

## 8-bis. Diagnostica split tipologie + checkpoint (il modello NON cambia)

Il modello stima a livello CANALE. Questa cella risponde alla domanda: "le tipologie
dentro un canale (es. Search, PMax, Display dentro Google Ads) avrebbero i numeri per
diventare canali-modello separati?" Criteri per dire SI': quota rilevante della spesa
totale, presenza quasi continua nella finestra, spesa NON collineare con le altre
tipologie.

Gira PRIMA del fit: se trova tipologie separabili te lo segnala con un checkpoint,
ti propone la **regola gia' pronta** da incollare in `REGOLE_CANALI` e ti fa scegliere
se proseguire cosi' o fermarti per applicare lo split. Lo split resta comunque una
scelta deliberata (regola + riesecuzione dalla cella 5), mai una modifica automatica.

In [ ]:
# ============================================================================
# CELLA 8-bis - DIAGNOSTICA SPLIT TIPOLOGIE + CHECKPOINT SPLIT
# La cella NON modifica il modello: se trova tipologie separabili te lo dice
# PRIMA del fit, ti suggerisce la regola pronta e ti lascia scegliere se
# fermarti per applicarla (split = sempre scelta deliberata, mai automatica).
# ============================================================================
# Soglie del verdetto (modificabili qui):
QUOTA_MIN_SPLIT = 0.10      # quota minima sulla spesa TOTALE del modello (10-15%)
PRESENZA_MIN_SPLIT = 0.90   # quota minima di settimane con spesa > 0 nella finestra
CORR_MAX_SPLIT = 0.80       # correlazione massima con le altre tipologie del canale

def diagnostica_split_tipologie():
    spesa_tot = float(DF_TIDY['spend'].sum())
    righe = []
    for canale in sorted(DF_CAMPAGNE['channel'].unique()):
        sc = DF_CAMPAGNE[DF_CAMPAGNE['channel'] == canale]
        tipologie = sorted(sc['sottocanale'].dropna().unique())
        if len(tipologie) < 2:
            continue    # una sola tipologia: niente da separare
        # serie settimanali di spesa per tipologia (griglia completa della finestra)
        piv = sc.groupby(['week', 'sottocanale'])['spend'].sum().unstack(fill_value=0.0)
        piv = piv.reindex(SETTIMANE, fill_value=0.0)
        corr = piv.corr()
        for t in tipologie:
            quota = float(piv[t].sum()) / max(spesa_tot, 1e-9)
            presenza = float((piv[t] > 0).mean())
            altre = [x for x in tipologie if x != t and piv[x].std() > 0]
            if piv[t].std() > 0 and altre:
                corr_max = float(corr.loc[t, altre].max())
            else:
                corr_max = np.nan
            motivi = []
            if quota < QUOTA_MIN_SPLIT:
                motivi.append('quota spesa ' + str(round(100 * quota, 1)) + '% < ' +
                              str(int(100 * QUOTA_MIN_SPLIT)) + '%')
            if presenza < PRESENZA_MIN_SPLIT:
                motivi.append('attiva solo nel ' + str(round(100 * presenza, 1)) +
                              '% delle settimane (< ' + str(int(100 * PRESENZA_MIN_SPLIT)) + '%)')
            if np.isnan(corr_max):
                motivi.append('serie costante o non valutabile')
            elif corr_max >= CORR_MAX_SPLIT:
                motivi.append('correlazione ' + str(round(corr_max, 2)) + ' >= ' +
                              str(CORR_MAX_SPLIT) + ' con le altre tipologie (collineare)')
            separabile = not motivi
            righe.append({
                'canale': canale, 'tipologia': t,
                'quota_spesa_totale_%': round(100 * quota, 1),
                'settimane_attive_%': round(100 * presenza, 1),
                'corr_max_con_altre': round(corr_max, 2) if not np.isnan(corr_max) else np.nan,
                'verdetto': ('SEPARABILE come canale-modello' if separabile
                             else 'resta aggregata in "' + canale + '"'),
                'motivi': '; '.join(motivi) if motivi else 'tutti i criteri soddisfatti',
            })
    return pd.DataFrame(righe)

DIAG_SPLIT = diagnostica_split_tipologie()

# Regole gia' pronte per lo split, popolate se ci sono tipologie separabili
# (utili anche all'export: restano disponibili come variabile globale).
REGOLE_SUGGERITE = []

if len(DIAG_SPLIT):
    print(DIAG_SPLIT.to_string(index=False))
    SEPARABILI = DIAG_SPLIT[DIAG_SPLIT['verdetto'].str.startswith('SEPARABILE')].copy()

    if len(SEPARABILI):
        # Per ogni tipologia separabile, propone una regola pronta da incollare in
        # REGOLE_CANALI. Il pattern e' quello con cui la tipologia e' gia' riconosciuta
        # in REGOLE_SOTTOCANALE, cosi' cattura le stesse campagne.
        _regex_per_sotto = {sotto: regex for regex, sotto in REGOLE_SOTTOCANALE}
        print()
        print('=' * 72)
        print('SPLIT POSSIBILE: ' + str(len(SEPARABILI)) + ' tipologia/e supera/no tutti i criteri.')
        print('=' * 72)
        print('Regole suggerite (incollale in REGOLE_CANALI nella cella CONFIG, SOPRA la')
        print("regola generica del canale d'origine: mappa_canale usa la prima regex che matcha).")
        print()
        for _, rr in SEPARABILI.iterrows():
            canale = rr['canale']; tip = rr['tipologia']
            nuovo = canale + ' - ' + tip
            rgx = _regex_per_sotto.get(tip)
            if rgx:
                print("    (r'" + rgx + "', '" + nuovo + "'),")
                REGOLE_SUGGERITE.append((rgx, nuovo))
            else:
                print("    # '" + tip + "': nessun pattern di naming dedicato in REGOLE_SOTTOCANALE.")
                print("    # Definisci a mano la regex che identifica queste campagne e mappala")
                print("    # a '" + nuovo + "'.")
        print()
        print('Poi riesegui dalla cella 5 (ingestion): il fit stimera i canali separati.')

        scelta = scegli_opzione(
            'La 8-bis ha trovato tipologie separabili. Come procedo?',
            ['Proseguo senza split (resta il livello canale attuale)',
             'Mi fermo: applico le regole sopra in REGOLE_CANALI e rieseguo da cella 5'],
            default=0)
        if scelta == 1:
            raise SystemExit(
                'Fermato alla 8-bis per applicare lo split: incolla le regole suggerite in '
                'REGOLE_CANALI (cella CONFIG), poi riesegui dalla cella 5.')
        print('Proseguo senza split: il modello stima al livello canale attuale.')
    else:
        print()
        print('Nessuna tipologia supera tutti i criteri: il livello canale attuale e corretto.')

    print()
    print('NB: la 8-bis e solo diagnostica; lo split e sempre una scelta deliberata')
    print('    (regola in REGOLE_CANALI + riesecuzione), mai una modifica automatica del modello.')
else:
    print('Nessun canale con piu tipologie: diagnostica split non applicabile.')


## 9. CHECKPOINT 4 + fit del modello Meridian (GPU, ~10 min misurati)

Priori debolmente informativi (LogNormal sul ROI), baseline flessibile
(`KNOTS_PER_QUARTER` nodi per trimestre). Il checkpoint mostra la specifica PRIMA
di lanciare il campionamento, perche' il fit e' costoso.

In [ ]:
# ============================================================================
# CELLA 9 - CHECKPOINT 4 (ModelSpec) + FIT + DIAGNOSTICA DI CONVERGENZA
# ============================================================================
N_KNOTS = min(N_SETTIMANE, max(1, int(round(N_SETTIMANE / 13.0 * KNOTS_PER_QUARTER))))

# ---------------------------------------------------------------------------
# 9.0 PRIOR SUL ROI
#     Spento (default): un solo LogNormal debolmente informativo per tutti i
#     canali - identico a come si e' sempre comportato questo notebook.
#     Acceso: mu e sigma diventano VETTORI nell'ordine di CANALI_MODELLO, con
#     mu = log(ROAS di riferimento) canale per canale.
# ---------------------------------------------------------------------------
PRIOR_INFO_ATTIVI = bool(USA_PRIOR_INFORMATIVI)
PRIOR_DETTAGLIO = []      # righe leggibili per il checkpoint e per l'Excel

if PRIOR_INFO_ATTIVI:
    _mancanti = [c for c in CANALI_MODELLO if c not in PRIOR_ROAS_CANALE]
    if _mancanti:
        raise SystemExit(
            'PRIOR INFORMATIVI: PRIOR_ROAS_CANALE non copre tutti i canali del '
            'modello. Mancano ' + str(len(_mancanti)) + ' canali su ' +
            str(len(CANALI_MODELLO)) + ': ' + ', '.join(_mancanti) + '. '
            'Aggiungili nella CELLA 2 e riesegui. Nessun default silenzioso: un '
            'prior inventato su un canale non si vede nell' + chr(39) + 'output e '
            'si propaga a tutte le stime.')
    _non_pos = [c for c in CANALI_MODELLO if not (float(PRIOR_ROAS_CANALE[c]) > 0)]
    if _non_pos:
        raise SystemExit(
            'PRIOR INFORMATIVI: il prior e' + chr(39) + ' LogNormal e mu = log(ROAS), '
            'quindi i ROAS di riferimento devono essere > 0. Non validi: ' +
            ', '.join(_non_pos))
    _MU_ROI = np.array([np.log(float(PRIOR_ROAS_CANALE[c])) for c in CANALI_MODELLO],
                       dtype=np.float32)
    _SIGMA_ROI = np.full(len(CANALI_MODELLO), float(PRIOR_ROI_SIGMA_INF),
                         dtype=np.float32)
    for _i, _c in enumerate(CANALI_MODELLO):
        PRIOR_DETTAGLIO.append(
            _c + ': ROAS rif. ' + str(round(float(PRIOR_ROAS_CANALE[_c]), 3)) +
            ' -> LogNormal(mu=' + str(round(float(_MU_ROI[_i]), 4)) +
            ', sigma=' + str(round(float(_SIGMA_ROI[_i]), 3)) + ')')
else:
    # scalari, esattamente come prima che questo interruttore esistesse
    _MU_ROI = PRIOR_ROI_MU
    _SIGMA_ROI = PRIOR_ROI_SIGMA
    PRIOR_DETTAGLIO.append(
        'tutti i canali: LogNormal(mu=' + str(PRIOR_ROI_MU) + ', sigma=' +
        str(PRIOR_ROI_SIGMA) + ') - debolmente informativo')

righe_cp4 = [
    "Dati: " + str(N_GEOS) + " geo x " + str(N_SETTIMANE) + " settimane x " +
    str(len(CANALI_MODELLO)) + " canali (+" + str(len(CONTROLLI)) + " controlli)",
    ("Prior ROI: INFORMATIVO per canale, sigma=" + str(PRIOR_ROI_SIGMA_INF) +
     " (ancorato ai ROAS di riferimento)") if PRIOR_INFO_ATTIVI else
    ("Prior ROI: LogNormal(mu=" + str(PRIOR_ROI_MU) + ", sigma=" +
     str(PRIOR_ROI_SIGMA) + ") - debolmente informativo, uguale per tutti i canali"),
]
righe_cp4 += ["   " + _r for _r in PRIOR_DETTAGLIO]
righe_cp4 += [
    "Baseline temporale: " + str(N_KNOTS) + " nodi (" + str(KNOTS_PER_QUARTER) + " per trimestre)",
    "Campionamento NUTS: " + str(N_CHAINS) + " catene, adapt=" + str(N_ADAPT) +
    ", burnin=" + str(N_BURNIN) + ", keep=" + str(N_KEEP) + ", seed=" + str(SEED),
]
if PRIOR_INFO_ATTIVI:
    righe_cp4.append(
        "ATTENZIONE: il prior sul ROI e' ANCORATO ai ROAS di PRIOR_ROAS_CANALE. "
        "Le stime NON sono piu' indipendenti da quella fonte: dichiararlo nei risultati.")
else:
    righe_cp4.append(
        "NESSUNA calibrazione sul ROAS di attribution (fonte non sperimentale: solo benchmark)")
righe_cp4.append(
    "Tempo atteso su T4: ~10 minuti (misurato); il tempo reale viene stampato a fine fit")
checkpoint("4. Specifica del modello - il fit e' costoso", righe_cp4)

PRIOR = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(_MU_ROI, _SIGMA_ROI, name=constants.ROI_M))
MODEL_SPEC = spec.ModelSpec(prior=PRIOR, knots=N_KNOTS)
MMM = model.Meridian(input_data=INPUT_DATA, model_spec=MODEL_SPEC)

_t0_fit = dt.datetime.now()
print("Campionamento dal prior...")
MMM.sample_prior(500)
print("Campionamento dal posterior (NUTS)... questo e' il passaggio lungo.")
MMM.sample_posterior(n_chains=N_CHAINS, n_adapt=N_ADAPT, n_burnin=N_BURNIN,
                     n_keep=N_KEEP, seed=SEED)
DURATA_FIT_MIN = (dt.datetime.now() - _t0_fit).total_seconds() / 60.0
print("Fit completato in " + str(round(DURATA_FIT_MIN, 1)) + " minuti.")

# ---------------------------------------------------------------------------
# 9.1 Diagnostica: R-hat, ESS, divergenze
# ---------------------------------------------------------------------------
idata = MMM.inference_data

DIVERGENZE = 0
try:
    for chiave in ('diverging', 'divergent'):
        if hasattr(idata, 'sample_stats') and chiave in idata.sample_stats:
            DIVERGENZE = int(np.asarray(idata.sample_stats[chiave]).sum())
            break
except Exception as e:
    print("[avviso] divergenze non leggibili: " + str(e))

righe_diag = []
try:
    rhat_ds = az.rhat(idata.posterior)
    ess_ds = az.ess(idata.posterior)
    for var in rhat_ds.data_vars:
        r = np.asarray(rhat_ds[var]).ravel()
        r = r[np.isfinite(r)]
        e = np.asarray(ess_ds[var]).ravel() if var in ess_ds.data_vars else np.array([np.nan])
        e = e[np.isfinite(e)]
        if len(r):
            righe_diag.append({'parametro': str(var),
                               'rhat_max': round(float(np.max(r)), 3),
                               'ess_min': int(np.min(e)) if len(e) else np.nan})
except Exception as e:
    print("[avviso] R-hat/ESS non calcolabili: " + str(e))
DIAG_TABELLA = pd.DataFrame(righe_diag).sort_values('rhat_max', ascending=False) \
    if righe_diag else pd.DataFrame([{'parametro': 'n/d', 'rhat_max': np.nan, 'ess_min': np.nan}])

RHAT_MAX = float(DIAG_TABELLA['rhat_max'].max())
print()
print("Divergenze: " + str(DIVERGENZE) + " | R-hat max: " + str(RHAT_MAX) +
      " | ESS min: " + str(DIAG_TABELLA['ess_min'].min()))
print(DIAG_TABELLA.head(12).to_string(index=False))
if RHAT_MAX > 1.1 or DIVERGENZE > 0:
    segnala("convergenza imperfetta (R-hat max " + str(RHAT_MAX) + ", divergenze " +
            str(DIVERGENZE) + ")",
            "valutare piu' iterazioni (N_KEEP/N_ADAPT) o priori: NON fidarsi ciecamente delle stime")
else:
    print("Convergenza OK (R-hat <= 1.1, nessuna divergenza).")

# ---------------------------------------------------------------------------
# 9.2 Salvataggio del modello
# ---------------------------------------------------------------------------
percorso_modello = os.path.join(OUTPUT_DIR, 'modello_meridian.pkl')

def _salva_modello(mmm, percorso):
    # `model.save_mmm` e' DEPRECATO: durante il fit Meridian stampa l'avviso e
    # rimanda a `meridian.schema.serde`. Le versioni intermedie pero' espongono
    # ancora solo la vecchia, e quale delle due risponda dipende dalla versione
    # installata sull'ambiente, non da questo notebook. Quindi: si prova la
    # nuova, si ricade sulla vecchia, e in ultimo sul pickle - che funziona
    # sempre. La via effettivamente usata viene stampata, cosi' se un domani
    # l'ambiente cambia te ne accorgi da qui.
    try:
        from meridian.schema import serde as _serde
        _obj = getattr(_serde, 'meridian_serde', _serde)
        for _nome in ('save_mmm', 'save', 'dump', 'write'):
            _fn = getattr(_obj, _nome, None)
            if callable(_fn):
                _fn(mmm, percorso)
                return 'meridian.schema.serde.' + _nome + ' (API corrente)'
    except Exception:
        pass
    try:
        model.save_mmm(mmm, percorso)
        return 'model.save_mmm (API deprecata: aggiorna google-meridian)'
    except Exception:
        with open(percorso, 'wb') as f:
            pickle.dump(mmm, f)
        return 'pickle (nessuna API di serializzazione disponibile)'

_via_salvataggio = _salva_modello(MMM, percorso_modello)
print("Modello salvato in " + percorso_modello)
print("   via: " + _via_salvataggio)

## 10. Analisi: ROAS, contributi, curve di risposta

Metriche incrementali dal posterior: ROAS (o conversioni/euro se manca `revenue_per_kpi`),
mROAS, contributi per canale vs baseline, curve di risposta/saturazione, fit vs osservato.

In [ ]:
# ============================================================================
# CELLA 10 - ANALISI DEL MODELLO (analyzer / visualizer / summarizer)
# ============================================================================

def chiama(fn, **kw):
    # Chiamata difensiva: filtra i kwargs non supportati dalla versione installata
    # di Meridian e non fa crashare il notebook se il metodo non esiste.
    try:
        sig = inspect.signature(fn)
        kw2 = {k: v for k, v in kw.items() if k in sig.parameters}
        return fn(**kw2)
    except Exception as e:
        print("[avviso] " + getattr(fn, '__name__', str(fn)) + " non disponibile: " + str(e))
        return None

def campioni(x):
    # Tensor/array (catene, draw, canali) -> matrice (campioni, canali).
    a = np.asarray(x)
    return a.reshape(-1, a.shape[-1])

AN = analyzer.Analyzer(MMM)
UNITA_OUTCOME = VALUTA_TARGET if REVENUE_PER_KPI_PRESENTE else 'conversioni'

# --- campioni posterior delle metriche chiave ---
INC_SAMP = campioni(AN.incremental_outcome())        # outcome incrementale per canale
if REVENUE_PER_KPI_PRESENTE:
    _tmp = chiama(AN.incremental_outcome, use_kpi=True)
    INC_KPI_SAMP = campioni(_tmp) if _tmp is not None else INC_SAMP
else:
    INC_KPI_SAMP = INC_SAMP                          # outcome gia' in conversioni
ROI_SAMP = campioni(AN.roi())
_tmp = chiama(AN.marginal_roi)
MROI_SAMP = campioni(_tmp) if _tmp is not None else np.full_like(ROI_SAMP, np.nan)

SPESA_CANALE = DF_TIDY.groupby('channel')['spend'].sum().reindex(CANALI_MODELLO)
KPI_TOTALE = float(DF_BASE['kpi'].sum())

def _stat(mat, j):
    col = mat[:, j]
    return (float(np.mean(col)), float(np.quantile(col, 0.05)), float(np.quantile(col, 0.95)))

righe = []
for j, c in enumerate(CANALI_MODELLO):
    inc_m, inc_lo, inc_hi = _stat(INC_SAMP, j)
    inck_m, _, _ = _stat(INC_KPI_SAMP, j)
    roi_m, roi_lo, roi_hi = _stat(ROI_SAMP, j)
    mroi_m, mroi_lo, mroi_hi = _stat(MROI_SAMP, j)
    spesa = float(SPESA_CANALE[c])
    righe.append({
        'canale': c,
        'spesa': round(spesa, 0),
        'quota_spesa_%': round(100 * spesa / float(SPESA_CANALE.sum()), 1),
        'outcome_incrementale': round(inc_m, 1),
        'outcome_incr_ci05': round(inc_lo, 1), 'outcome_incr_ci95': round(inc_hi, 1),
        'conversioni_incrementali': round(inck_m, 1),
        'roas': round(roi_m, 3), 'roas_ci05': round(roi_lo, 3), 'roas_ci95': round(roi_hi, 3),
        'mroas': round(mroi_m, 3), 'mroas_ci05': round(mroi_lo, 3), 'mroas_ci95': round(mroi_hi, 3),
        'cpa': round(spesa / inck_m, 2) if inck_m > 0 else np.nan,
        'contributo_su_kpi_%': round(100 * inck_m / KPI_TOTALE, 1) if KPI_TOTALE else np.nan,
    })
TAB_ROAS = pd.DataFrame(righe)
print("Unita' dell'outcome incrementale e del ROAS: " + UNITA_OUTCOME +
      (" per " + VALUTA_TARGET + " speso" if True else ""))
print(TAB_ROAS.to_string(index=False))

# --- contributi: baseline vs canali ---
inc_tot_kpi = float(np.mean(INC_KPI_SAMP.sum(axis=1)))
TAB_CONTRIB = pd.DataFrame(
    [{'voce': 'Baseline (organico, stagionalita, controlli)',
      'conversioni': round(KPI_TOTALE - inc_tot_kpi, 0),
      'quota_%': round(100 * (KPI_TOTALE - inc_tot_kpi) / KPI_TOTALE, 1)}] +
    [{'voce': r['canale'], 'conversioni': round(r['conversioni_incrementali'], 0),
      'quota_%': round(100 * r['conversioni_incrementali'] / KPI_TOTALE, 1)}
     for _, r in TAB_ROAS.iterrows()])
print()
print(TAB_CONTRIB.to_string(index=False))
if inc_tot_kpi / max(KPI_TOTALE, 1e-9) < 0.05 or inc_tot_kpi / max(KPI_TOTALE, 1e-9) > 0.9:
    segnala("quota media sul KPI = " + str(round(100 * inc_tot_kpi / KPI_TOTALE, 1)) + "%",
            "valore estremo: verificare baseline/controlli prima di usare l'allocatore")

# --- grafici: ROAS con intervalli, contributi ---
plt.figure(figsize=(8, 4))
x = np.arange(len(CANALI_MODELLO))
plt.bar(x, TAB_ROAS['roas'],
        yerr=[TAB_ROAS['roas'] - TAB_ROAS['roas_ci05'], TAB_ROAS['roas_ci95'] - TAB_ROAS['roas']],
        capsize=4, alpha=0.8)
if REVENUE_PER_KPI_PRESENTE:
    plt.axhline(1.0, color='red', ls='--', lw=1, label='break-even (ROAS=1)')
    plt.legend()
plt.xticks(x, CANALI_MODELLO, rotation=30, ha='right')
plt.ylabel('ROAS (' + UNITA_OUTCOME + ' / ' + VALUTA_TARGET + ')')
plt.title('ROAS incrementale per canale (media e CI 90%)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'modello_roas.png'), dpi=130)
plt.show()

plt.figure(figsize=(8, 4))
plt.barh(TAB_CONTRIB['voce'], TAB_CONTRIB['quota_%'])
plt.xlabel('quota del KPI totale (%)')
plt.title('Scomposizione del KPI: baseline vs canali')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'contributi.png'), dpi=130)
plt.show()

# --- curve di risposta / saturazione ---
RC_DF = None
_rc = chiama(AN.response_curves,
             spend_multipliers=[round(v, 2) for v in np.linspace(0, 2, 21)])
if _rc is not None:
    try:
        RC_DF = _rc.to_dataframe().reset_index()
    except Exception as e:
        print("[avviso] curve di risposta non tabellabili: " + str(e))
if RC_DF is not None:
    col_sp = next((c for c in RC_DF.columns if 'spend' in c and 'mult' not in c), None)
    col_out = next((c for c in RC_DF.columns if 'incremental' in c or 'outcome' in c), None)
    col_ch = next((c for c in RC_DF.columns if 'channel' in c), None)
    if col_sp and col_out and col_ch:
        if 'metric' in RC_DF.columns:
            RC_DF = RC_DF[RC_DF['metric'] == 'mean']
        n = len(CANALI_MODELLO)
        ncol = min(3, n); nrig = math.ceil(n / ncol)
        fig, assi = plt.subplots(nrig, ncol, figsize=(4 * ncol, 3.2 * nrig), squeeze=False)
        for i, c in enumerate(CANALI_MODELLO):
            ax = assi[i // ncol][i % ncol]
            s = RC_DF[RC_DF[col_ch] == c].sort_values(col_sp)
            ax.plot(s[col_sp], s[col_out])
            ax.axvline(float(SPESA_CANALE[c]), color='gray', ls=':', lw=1)
            ax.set_title(c + ' (linea: spesa attuale)', fontsize=9)
            ax.set_xlabel('spesa'); ax.set_ylabel(UNITA_OUTCOME + ' incr.')
        for j in range(n, nrig * ncol):
            assi[j // ncol][j % ncol].axis('off')
        plt.suptitle('Curve di risposta e saturazione', y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, 'response_curves.png'), dpi=130, bbox_inches='tight')
        plt.show()
else:
    print("[avviso] curve di risposta non disponibili in questa versione: foglio Excel ridotto")

# --- fit vs osservato (per la Diagnostica) ---
FIT_DF = None
_eva = chiama(AN.expected_vs_actual_data)
if _eva is not None:
    try:
        d = _eva.to_dataframe().reset_index()
        col_t = next((c for c in d.columns if 'time' in c), None)
        num = [c for c in d.columns if pd.api.types.is_numeric_dtype(d[c])]
        if 'metric' in d.columns:
            d = d[d['metric'] == 'mean'] if (d['metric'] == 'mean').any() else d
        FIT_DF = d.groupby(col_t, as_index=False)[num].sum()
        col_exp = next((c for c in FIT_DF.columns if 'expected' in c), None)
        col_act = next((c for c in FIT_DF.columns if 'actual' in c), None)
        if col_exp and col_act:
            plt.figure(figsize=(11, 3.5))
            plt.plot(pd.to_datetime(FIT_DF[col_t]), FIT_DF[col_act], label='osservato', lw=1.2)
            plt.plot(pd.to_datetime(FIT_DF[col_t]), FIT_DF[col_exp], label='atteso (modello)', lw=1.2)
            plt.legend(); plt.title('Fit del modello vs KPI osservato (nazionale)')
            plt.tight_layout()
            plt.savefig(os.path.join(FIG_DIR, 'diagnostica_fit.png'), dpi=130)
            plt.show()
    except Exception as e:
        print("[avviso] fit vs osservato non disponibile: " + str(e))

# --- report HTML nativo di Meridian (facoltativo) ---
try:
    summ = summarizer.Summarizer(MMM)
    chiama(summ.output_model_results_summary,
           filename='sintesi_modello.html', filepath=OUTPUT_DIR,
           start_date=str(SETTIMANE[0].date()), end_date=str(SETTIMANE[-1].date()))
    print("Report HTML Meridian salvato in " + OUTPUT_DIR + "/sintesi_modello.html")
except Exception as e:
    print("[avviso] summarizer non disponibile: " + str(e))


# ---------------------------------------------------------------------------
# 10-bis  DIAGNOSTICA SUL MONDO SIMULATO: quota stimata vs quota VERA
#
# Si attiva SOLO se accanto al notebook esiste la verita' del dataset simulato
# (dati_simulati/verita/contributo_vero.csv). Sui dati reali quel file non
# esiste, e allora questo blocco resta muto: non stampa nulla, non tocca nulla.
#
# A cosa serve: rendere riproducibile a ogni esecuzione il confronto che
# altrimenti si fa a mano, cioe' quanto il modello ATTRIBUISCE ai media contro
# quanto i media hanno DAVVERO generato nel mondo che ha prodotto questi dati.
# E' il controllo che sui dati reali non si puo' fare - li' la verita' non e'
# osservabile - ed e' esattamente la ragione per cui il mondo simulato esiste.
#
# Solo lettura: nessuna variabile del modello viene riassegnata.
# ---------------------------------------------------------------------------
QUOTE_VERO_VS_STIMATO = None

# Il file di verita' si chiama contributo_vero.csv, ma le varianti del dataset
# lo suffissano (contributo_vero_pause.csv, _geo, _casuale...). Prima si
# cercava UN percorso fisso e, non trovandolo, la diagnostica taceva: sembrava
# che non ci fosse nulla da dire, mentre il file era solo altrove.
_NOMI_VERITA = ('contributo_vero',)
_RADICI_VERITA = []
for _r in (globals().get('ORIGINE_DATI'), '.', '..', '/content', OUTPUT_DIR):
    if not _r:
        continue
    _r = str(_r)
    if os.path.isfile(_r):
        _r = os.path.dirname(_r) or '.'
    if os.path.isdir(_r) and _r not in _RADICI_VERITA:
        _RADICI_VERITA.append(_r)
        _padre = os.path.dirname(os.path.abspath(_r))
        if os.path.isdir(_padre) and _padre not in _RADICI_VERITA:
            _RADICI_VERITA.append(_padre)

_cand_verita, _visti = [], set()
for _radice in _RADICI_VERITA:
    for _r, _sub, _nomi in os.walk(_radice):
        # niente discese infinite: solo la radice, le sue sottocartelle dati e
        # qualunque cartella che si chiami verita*
        _prof = os.path.relpath(_r, _radice).count(os.sep)
        if _prof > 3:
            _sub[:] = []
            continue
        for _n in _nomi:
            if not _n.lower().endswith('.csv'):
                continue
            if any(_n.lower().startswith(_b) for _b in _NOMI_VERITA):
                _p = os.path.abspath(os.path.join(_r, _n))
                if _p not in _visti:
                    _visti.add(_p)
                    _cand_verita.append(_p)

_file_verita = _cand_verita[0] if _cand_verita else None
if _file_verita is None:
    print()
    print('-' * 76)
    print('DIAGNOSTICA MONDO SIMULATO: NON ESEGUITA - file di verita' + chr(39) + ' non trovato.')
    print('Cercato un file contributo_vero*.csv sotto:')
    for _r in _RADICI_VERITA:
        print('   ' + os.path.abspath(_r))
    print('Se stai lavorando su dati reali e' + chr(39) + ' corretto cosi' + chr(39) +
          ': la verita' + chr(39) + ' non esiste.')
    print('Se stai usando il dataset simulato, carica anche la cartella verita/.')
    print('-' * 76)
elif len(_cand_verita) > 1:
    print('[verita' + chr(39) + '] trovati ' + str(len(_cand_verita)) +
          ' file candidati, uso: ' + _file_verita)
else:
    print('[verita' + chr(39) + '] uso ' + _file_verita)

if _file_verita is not None:
    try:
        _vt = pd.read_csv(_file_verita)
        _vt['settimana'] = pd.to_datetime(_vt['settimana'])
        # Stessa finestra temporale del modello: quote su denominatori diversi
        # non sarebbero confrontabili.
        _sett = globals().get('SETTIMANE')
        if _sett is not None and len(_sett):
            _lo, _hi = pd.Timestamp(min(_sett)), pd.Timestamp(max(_sett))
            _vt = _vt[(_vt['settimana'] >= _lo) & (_vt['settimana'] <= _hi)]
        _vero = _vt.groupby('canale')['candidature_incrementali_vere'].sum()
        _stim = TAB_ROAS.set_index('canale')['conversioni_incrementali']

        _righe_d, _s_tot, _v_tot = [], 0.0, 0.0
        for _c in CANALI_MODELLO:
            _s = float(_stim.get(_c, float('nan')))
            _v = float(_vero.get(_c, float('nan')))
            if _s == _s:
                _s_tot += _s
            if _v == _v:
                _v_tot += _v
            _righe_d.append({
                'canale': _c,
                'quota_stimata_%': round(100 * _s / KPI_TOTALE, 1),
                'quota_vera_%': round(100 * _v / KPI_TOTALE, 1),
                'scarto_pp': round(100 * (_s - _v) / KPI_TOTALE, 1),
                'stimato_su_vero': round(_s / _v, 2) if _v else float('nan'),
            })
        _righe_d.append({
            'canale': 'TOTALE MEDIA',
            'quota_stimata_%': round(100 * _s_tot / KPI_TOTALE, 1),
            'quota_vera_%': round(100 * _v_tot / KPI_TOTALE, 1),
            'scarto_pp': round(100 * (_s_tot - _v_tot) / KPI_TOTALE, 1),
            'stimato_su_vero': round(_s_tot / _v_tot, 2) if _v_tot else float('nan'),
        })
        QUOTE_VERO_VS_STIMATO = pd.DataFrame(_righe_d)

        print()
        print('=' * 76)
        print('DIAGNOSTICA MONDO SIMULATO - quota sul KPI: STIMATA vs VERA')
        print('=' * 76)
        print('verita' + chr(39) + ' letta da: ' + _file_verita)
        print()
        print(QUOTE_VERO_VS_STIMATO.to_string(index=False))
        print()
        _gap = 100 * (_s_tot - _v_tot) / KPI_TOTALE
        if abs(_gap) >= 5:
            print('LETTURA: il modello attribuisce ai media ' +
                  str(round(100 * _s_tot / KPI_TOTALE, 1)) + '% del KPI, la verita' + chr(39) + ' e' + chr(39) + ' ' +
                  str(round(100 * _v_tot / KPI_TOTALE, 1)) + '%.')
            print('Uno scarto di questa ampiezza NON e' + chr(39) + ' necessariamente un errore di')
            print('esecuzione: e' + chr(39) + ' il confondimento media/baseline della Sez. 4.8. Va letto')
            print('insieme ai parametri peggio identificati - se i peggiori sono mu_t e')
            print('knot_values, cioe' + chr(39) + ' proprio la baseline, la diagnosi e' + chr(39) + ' coerente.')
            print()
            print('ATTENZIONE: catene convergenti non smentiscono questo risultato. La')
            print('convergenza dice che il campionatore ha esplorato bene il posterior,')
            print('non che il posterior sia centrato sulla verita' + chr(39) + '.')
        else:
            print('LETTURA: quota stimata e quota vera sono vicine (scarto ' +
                  str(round(_gap, 1)) + ' p.p.): su questo dataset il modello')
            print('non mostra sovra-attribuzione sistematica ai media.')
        print('=' * 76)
    except Exception as _e:
        print('[avviso] diagnostica mondo simulato non calcolabile: ' + str(_e))


# ---------------------------------------------------------------------------
# 10-ter  PRIOR vs POSTERIOR sul ROI
#
# La domanda: il modello ha detto qualcosa sui LIVELLI, o ha solo restituito
# il prior? Con prior informativi il posterior puo' restare incollato al
# riferimento che gli e' stato dato: in quel caso la stima non e' una stima,
# e' l'input riscritto in uscita. Il rapporto posterior/prior lo rende
# visibile: vicino a 1 = i dati non hanno spostato nulla.
#
# Con i prior spenti non stampa nulla: con un prior largo e uguale per tutti
# la domanda non si pone.
# ---------------------------------------------------------------------------
PRIOR_VS_POSTERIOR = None
if globals().get('PRIOR_INFO_ATTIVI'):
    try:
        _idp = MMM.inference_data
        _pri = np.asarray(_idp.prior['roi_m'])
        _pos = np.asarray(_idp.posterior['roi_m'])
        _n_ch = len(CANALI_MODELLO)
        if _pri.shape[-1] != _n_ch or _pos.shape[-1] != _n_ch:
            raise ValueError('roi_m non ha i canali sull' + chr(39) + 'ultimo asse: ' +
                             str(_pri.shape) + ' / ' + str(_pos.shape))
        _pri = _pri.reshape(-1, _n_ch)
        _pos = _pos.reshape(-1, _n_ch)
        _righe_pp = []
        for _i, _c in enumerate(CANALI_MODELLO):
            _mp = float(np.median(_pri[:, _i]))
            _ms = float(np.median(_pos[:, _i]))
            _righe_pp.append({
                'canale': _c,
                'prior_mediana': round(_mp, 3),
                'posterior_mediana': round(_ms, 3),
                'posterior_su_prior': round(_ms / _mp, 3) if _mp else float('nan'),
            })
        PRIOR_VS_POSTERIOR = pd.DataFrame(_righe_pp)
        print()
        print('=' * 76)
        print('PRIOR vs POSTERIOR sul ROI (prior informativi attivi)')
        print('=' * 76)
        print(PRIOR_VS_POSTERIOR.to_string(index=False))
        _rap = PRIOR_VS_POSTERIOR['posterior_su_prior'].astype(float)
        _fermi = PRIOR_VS_POSTERIOR.loc[(_rap - 1.0).abs() < 0.10, 'canale'].tolist()
        print()
        if _fermi:
            print('Posterior entro il 10% del prior su: ' + ', '.join(_fermi) + '.')
            print('Su questi canali i dati non hanno aggiunto informazione sul LIVELLO:')
            print('la stima sta restituendo il riferimento che le e' + chr(39) + ' stato dato.')
            print('Da dichiarare: non e' + chr(39) + ' una conferma indipendente del ROAS di partenza.')
        else:
            print('Nessun canale con posterior incollato al prior: su tutti i canali')
            print('i dati hanno spostato il livello rispetto al riferimento.')
        print('=' * 76)
    except Exception as _e:
        print('[avviso] confronto prior/posterior non calcolabile: ' + str(_e))


## 11. Confronto ROAS: MMM incrementale vs benchmark attribuito

Carica qui il file ROAS aziendale (attribution multi-touch, trimestrale, per macro-canale
e/o campagna). Viene usato **solo come benchmark**: mai come priore di calibrazione,
per non importare il bias di attribution nel modello causale.
Il gap incrementale-vs-attribuito e' un **risultato di tesi da interpretare, non da azzerare**.

In [ ]:
# ============================================================================
# CELLA 11 - CONFRONTO ROAS MMM (incrementale) vs BENCHMARK (attribuito)
# ============================================================================
CONFRONTO = None
CONFRONTO_SALTATO = None   # None = non tentato; stringa = motivo del salto

def parse_trimestre(v):
    # "Q1 2025", "2025-Q1", "T1 2025", "1 trimestre 2025", date -> "2025Q1"
    s = norm_txt(v)
    m = re.search(r'(?:^|\D)(20\d{2})\D{0,3}[qt]\s*([1-4])', s) or \
        re.search(r'[qt]\s*([1-4])\D{0,3}(20\d{2})', s) or \
        re.search(r'([1-4])\s*(?:o)?\s*trimestre\D{0,3}(20\d{2})', s)
    if m:
        g = m.groups()
        anno, q = (g[0], g[1]) if len(g[0]) == 4 else (g[1], g[0])
        return anno + 'Q' + q
    d = parse_date_serie(pd.Series([v]))[0]
    if pd.notna(d):
        return str(d.year) + 'Q' + str((d.month - 1) // 3 + 1)
    return None

def trimestre_settimana(w):
    return str(w.year) + 'Q' + str((w.month - 1) // 3 + 1)

def scegli_colonna_canale(df, ruoli):
    # Scelta GENERICA della colonna canale nel benchmark: tra le colonne testuali
    # vince quella i cui valori si mappano meglio sulla tassonomia REGOLE_CANALI
    # (cosi' un file con piu' colonne descrittive/di raggruppamento usa quella giusta).
    candidate = [ruoli.get('channel'), ruoli.get('source'), ruoli.get('medium'),
                 ruoli.get('campaign')]
    candidate = [c for c in candidate if c is not None]
    candidate += [c for c in df.columns if c not in candidate and df[c].dtype == object]
    migliore, quota_migliore = None, -1.0
    for c in candidate:
        vals = df[c].dropna().astype(str)
        if not len(vals):
            continue
        mappati = vals.map(mappa_canale)
        quota = float((~mappati.isin([CANALE_NON_MAPPATO, '_ESCLUDI_', '_ORGANICO_'])).mean())
        if quota > quota_migliore:
            migliore, quota_migliore = c, quota
    return migliore, quota_migliore

def normalizza_roas_bench(valori_grezzi, valori_num, etichetta=''):
    # ROAS espresso in PERCENTUALE (100% = break-even) -> rapporto (1.0 = break-even),
    # cosi' e' confrontabile col ROAS monetario del modello. Rilevazione: simbolo %
    # nei valori, oppure scala implausibile per un rapporto (mediana > 20).
    try:
        quota_pct = float(valori_grezzi.astype(str).str.contains('%').mean())
    except Exception:
        quota_pct = 0.0
    mediana = float(pd.Series(valori_num).median())
    if quota_pct > 0.2 or mediana > 20:
        segnala(etichetta + ": ROAS benchmark in percentuale (100 = break-even, mediana " +
                str(round(mediana, 1)) + ")", "convertito in rapporto: diviso per 100")
        return valori_num / 100.0
    return valori_num

def compare_roas():
    global CONFRONTO
    global CONFRONTO_SALTATO
    if scegli_opzione("Hai il file ROAS benchmark (attribution trimestrale)?",
                      ["Si', lo carico ora", "No, salta il confronto"], default=0) == 1:
        CONFRONTO_SALTATO = 'scelta esplicita: nessun file benchmark fornito'
        print()
        print('!' * 76)
        print('!!!' + 'CONFRONTO COL BENCHMARK SALTATO'.center(70) + '!!!')
        print('!' * 76)
        print('Cosa NON verra' + chr(39) + ' prodotto:')
        print('  - la tabella per trimestre e canale con ROAS incrementale del modello')
        print('    accanto al ROAS dichiarato, con gap e rapporto fra i due;')
        print('  - il grafico confronto_roas.png dell' + chr(39) + 'ultimo trimestre;')
        print('  - il foglio "Confronto_ROAS" dell' + chr(39) + 'Excel, che restera' + chr(39) + ' vuoto.')
        print()
        print('E' + chr(39) + ' il confronto su cui poggia l' + chr(39) + 'argomento centrale della tesi: che')
        print('l' + chr(39) + 'attribuzione per ultimi touchpoint e l' + chr(39) + 'incrementalita' + chr(39) + ' causale danno')
        print('numeri diversi. Senza, resta il solo risultato del modello.')
        print('Per averlo: rilancia questa cella con il file benchmark trimestrale.')
        print('!' * 76)
        segnala('confronto col benchmark saltato dall' + chr(39) + 'utente',
                'foglio Confronto_ROAS vuoto; nessun gap incrementale vs attribuito')
        return

    # L'upload obbligatorio era il motivo per cui questa cella veniva saltata:
    # se il file benchmark sta gia' sulla macchina (zip scompattato, repo
    # clonato, cartella dati) non c'e' ragione di ripassare dal disco locale.
    _modo_b = scegli_opzione(
        'Come fornisci il file benchmark?',
        ['Upload dal PC (files.upload)',
         'Percorso di una cartella o di un file presente sulla macchina'],
        default=0)
    up = {}
    if _modo_b == 0:
        from google.colab import files as colab_files
        up = colab_files.upload()
    else:
        _pb = input(
            'Percorso della cartella o del file benchmark '
            '(es. /content/mmm/dati_simulati/benchmark): '
        ).strip().strip('"').strip(chr(39))
        _ext_b = ('.csv', '.tsv', '.txt', '.xlsx', '.xls', '.json')
        if os.path.isdir(_pb):
            for _nb_ in sorted(os.listdir(_pb)):
                _fb = os.path.join(_pb, _nb_)
                if os.path.isfile(_fb) and os.path.splitext(_nb_)[1].lower() in _ext_b:
                    with open(_fb, 'rb') as _fh:
                        up[_nb_] = _fh.read()
        elif os.path.isfile(_pb):
            with open(_pb, 'rb') as _fh:
                up[os.path.basename(_pb)] = _fh.read()
        else:
            print('Percorso inesistente: ' + _pb)
        if not up:
            CONFRONTO_SALTATO = 'nessun file benchmark trovato in ' + str(_pb)
            segnala('confronto col benchmark saltato: percorso senza file leggibili',
                    'foglio Confronto_ROAS vuoto; nessun gap incrementale vs attribuito')
            print('Nessun file leggibile: confronto saltato.')
            return
        print(str(len(up)) + ' file letti da ' + _pb)
    candidati_bench = []
    for nome, dati in up.items():
        for nome_tab, df_raw in leggi_file_grezzo(nome, dati).items():
            # ogni foglio viene provato separatamente: quelli non conformi
            # (riepiloghi, note, altri segmenti) vengono saltati, non bloccano
            try:
                df, _h = pulisci_tabella(df_raw)
                ruoli = mappa_ruoli_colonne(df)
                print(nome_tab + " - colonne riconosciute: " + json.dumps(
                    {r: str(c) for r, c in ruoli.items()}, ensure_ascii=False))

                b = pd.DataFrame(index=df.index)
                # periodo -> trimestre
                col_q = ruoli.get('quarter') or ruoli.get('date')
                assert col_q, "manca una colonna trimestre/periodo/data"
                b['trimestre'] = df[col_q].map(parse_trimestre)
                # canale grezzo -> macro-canale (stesse REGOLE_CANALI del modello);
                # la colonna viene scelta in base a quanto i valori sono riconoscibili
                col_c, quota_c = scegli_colonna_canale(df, ruoli)
                assert col_c is not None, "manca una colonna canale/sorgente/campagna"
                print(nome_tab + " - colonna canale scelta: '" + str(col_c) + "' (" +
                      str(int(round(100 * max(quota_c, 0)))) + "% valori riconosciuti)")
                if quota_c <= 0:
                    segnala(nome_tab + ": nessuna etichetta canale riconosciuta dalle regole",
                            "confronto con le etichette originali del file")
                etichette = df[col_c].astype(str)
                b['canale'] = etichette.map(mappa_canale)
                # le etichette non riconosciute restano col loro nome originale: nel
                # benchmark possono esserci raggruppamenti aggregati che non hanno un
                # equivalente tra i canali del modello; restano visibili nel confronto,
                # semplicemente senza riga MMM abbinata
                maschera_altro = b['canale'] == CANALE_NON_MAPPATO
                b.loc[maschera_altro, 'canale'] = etichette[maschera_altro].str.strip().str.title()
                # roas diretto (anche in percentuale) oppure revenue/spend
                if 'roas' in ruoli:
                    roas_num, _ = pulisci_numerico(df[ruoli['roas']])
                    b['roas_bench'] = normalizza_roas_bench(df[ruoli['roas']], roas_num, nome_tab)
                elif 'revenue' in ruoli and 'spend' in ruoli:
                    rev, _v1 = pulisci_numerico(df[ruoli['revenue']])
                    sp, _v2 = pulisci_numerico(df[ruoli['spend']])
                    b['roas_bench'] = rev / sp.replace(0, np.nan)
                else:
                    raise AssertionError("serve una colonna ROAS oppure revenue+spesa")
                if 'spend' in ruoli:
                    b['spesa_bench'], _ = pulisci_numerico(df[ruoli['spend']])
                b = b.dropna(subset=['trimestre', 'roas_bench'])
                if len(b):
                    candidati_bench.append((nome_tab, b))
            except Exception as e:
                print("[foglio saltato] " + nome_tab + ": " + str(e))
    assert candidati_bench, ("Nessuna tabella benchmark utilizzabile: servono almeno "
                             "trimestre/data, canale e ROAS (o revenue+spesa).")
    # se il file ha piu' fogli utilizzabili (es. segmenti diversi), la scelta e' tua
    if len(candidati_bench) > 1:
        default_b = max(range(len(candidati_bench)), key=lambda i: len(candidati_bench[i][1]))
        idx_b = scegli_opzione("Il benchmark contiene piu' tabelle/fogli utilizzabili: quale usare?",
                               [n + " (" + str(len(b)) + " righe valide)"
                                for n, b in candidati_bench], default=default_b)
        bench = candidati_bench[idx_b][1]
    else:
        bench = candidati_bench[0][1]
    bench = bench[~bench['canale'].isin(['_ORGANICO_', '_ESCLUDI_'])]

    # aggregazione campagna -> macro-canale: media pesata sulla spesa se disponibile
    if 'spesa_bench' in bench.columns and bench['spesa_bench'].notna().any():
        bench['_p'] = bench['roas_bench'] * bench['spesa_bench']
        agg = bench.groupby(['trimestre', 'canale']).agg(
            _p=('_p', 'sum'), spesa_bench=('spesa_bench', 'sum')).reset_index()
        agg['roas_bench'] = agg['_p'] / agg['spesa_bench'].replace(0, np.nan)
        bench = agg[['trimestre', 'canale', 'roas_bench', 'spesa_bench']]
    else:
        segnala("benchmark senza colonna spesa", "aggregazione a media SEMPLICE per macro-canale")
        bench = bench.groupby(['trimestre', 'canale'], as_index=False)['roas_bench'].mean()

    # ROAS MMM incrementale per trimestre (dal posterior, selected_times per trimestre)
    settimane_str = [str(w.date()) for w in SETTIMANE]
    tri_map = {}
    for w, ws in zip(SETTIMANE, settimane_str):
        tri_map.setdefault(trimestre_settimana(w), []).append(ws)

    righe = []
    for tri, tempi in sorted(tri_map.items()):
        inc = chiama(AN.incremental_outcome, selected_times=tempi)
        if inc is None:
            segnala("incremental_outcome per trimestre non supportato", "confronto sul periodo intero")
            break
        inc = campioni(inc)
        spesa_q = DF_TIDY[DF_TIDY['week'].isin(pd.to_datetime(tempi))] \
            .groupby('channel')['spend'].sum().reindex(CANALI_MODELLO)
        for j, c in enumerate(CANALI_MODELLO):
            if spesa_q[c] and spesa_q[c] > 0:
                r = inc[:, j] / float(spesa_q[c])
                righe.append({'trimestre': tri, 'canale': c,
                              'spesa_mmm': round(float(spesa_q[c]), 0),
                              'roas_mmm': round(float(np.mean(r)), 3),
                              'roas_mmm_ci05': round(float(np.quantile(r, 0.05)), 3),
                              'roas_mmm_ci95': round(float(np.quantile(r, 0.95)), 3),
                              'settimane_nel_trimestre': len(tempi)})
    mmm_q = pd.DataFrame(righe, columns=['trimestre', 'canale', 'spesa_mmm', 'roas_mmm',
                                         'roas_mmm_ci05', 'roas_mmm_ci95',
                                         'settimane_nel_trimestre'])

    CONFRONTO = mmm_q.merge(bench, on=['trimestre', 'canale'], how='outer')
    CONFRONTO['gap_bench_minus_mmm'] = CONFRONTO['roas_bench'] - CONFRONTO['roas_mmm']
    CONFRONTO['rapporto_bench_su_mmm'] = (CONFRONTO['roas_bench'] /
                                          CONFRONTO['roas_mmm'].replace(0, np.nan)).round(2)
    if not REVENUE_PER_KPI_PRESENTE:
        CONFRONTO['nota'] = ("ATTENZIONE: ROAS MMM in conversioni/" + VALUTA_TARGET +
                             ", benchmark monetario: confrontabili solo come RANKING")
        segnala("confronto ROAS senza revenue_per_kpi",
                "valido solo il confronto di ranking/quote, non di livello")
    print()
    print(CONFRONTO.to_string(index=False))

    # grafico: ultimo trimestre con entrambe le fonti
    comuni = CONFRONTO.dropna(subset=['roas_mmm', 'roas_bench'])
    if len(comuni):
        tri = sorted(comuni['trimestre'].unique())[-1]
        s = comuni[comuni['trimestre'] == tri]
        x = np.arange(len(s))
        plt.figure(figsize=(8, 4))
        plt.bar(x - 0.2, s['roas_mmm'], width=0.4, label='MMM incrementale',
                yerr=[s['roas_mmm'] - s['roas_mmm_ci05'], s['roas_mmm_ci95'] - s['roas_mmm']],
                capsize=4)
        plt.bar(x + 0.2, s['roas_bench'], width=0.4, label='Benchmark attribuito')
        plt.xticks(x, s['canale'], rotation=20, ha='right')
        plt.title('ROAS ' + tri + ': incrementale (MMM) vs attribuito (ultimi 3 touchpoint)')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, 'confronto_roas.png'), dpi=130)
        plt.show()
        print()
        print("LETTURA: l'attribution (ultimi 3 touchpoint) da' credito ai canali di chiusura;")
        print("l'MMM misura l'incrementalita' causale. Un gap positivo sui canali di search/job board")
        print("e negativo sull'upper funnel e' atteso: e' un risultato da discutere in tesi, non un errore.")

compare_roas()

## 12. CHECKPOINT 5 + Budget allocator (cuore della tesi)

Ottimizzatore nativo di Meridian (`BudgetOptimizer`). **Goal principale**: massimizzare
l'outcome incrementale a **budget totale fisso**, riallocando la spesa tra canali entro i
vincoli (`VINCOLO_SPESA_PCT`, `VINCOLI_PER_CANALE` nella CONFIG). Scenari secondari:
budget flessibile con target mROI e what-if al variare del budget totale.

In [ ]:
# ============================================================================
# CELLA 12 - BUDGET ALLOCATOR: allocazione attuale vs ottimale + scenari
# ============================================================================
SPESA_STORICA = float(SPESA_CANALE.sum())
BUDGET = SPESA_STORICA if BUDGET_TOTALE == 'storico' else float(BUDGET_TOTALE)

# vincoli per canale (quota di variazione ammessa rispetto alla spesa attuale)
if VINCOLI_PER_CANALE:
    VINCOLO_LO = [float(VINCOLI_PER_CANALE.get(c, (VINCOLO_SPESA_PCT, VINCOLO_SPESA_PCT))[0])
                  for c in CANALI_MODELLO]
    VINCOLO_HI = [float(VINCOLI_PER_CANALE.get(c, (VINCOLO_SPESA_PCT, VINCOLO_SPESA_PCT))[1])
                  for c in CANALI_MODELLO]
else:
    VINCOLO_LO = VINCOLO_HI = VINCOLO_SPESA_PCT

righe_cp5 = [
    "GOAL: " + ("massimizzare l'outcome incrementale a BUDGET FISSO (riallocazione)"
                if GOAL_OTTIMIZZAZIONE == 'budget_fisso' else "budget flessibile (target mROI)"),
    "Budget totale: " + format(int(BUDGET), ",") + " " + VALUTA_TARGET +
    (" (= spesa storica del periodo)" if BUDGET_TOTALE == 'storico' else " (override)"),
    "Vincoli di spesa per canale: +/-" + str(int(VINCOLO_SPESA_PCT * 100)) + "%" +
    (" con override: " + str(VINCOLI_PER_CANALE) if VINCOLI_PER_CANALE else ""),
    "Outcome ottimizzato: " + UNITA_OUTCOME,
    "Scenari secondari: budget flessibile (target mROI=" + str(TARGET_MROI) + "), what-if x" +
    str(SCENARI_MOLTIPLICATORI),
]
checkpoint("5. Ottimizzazione budget", righe_cp5)

BO = optimizer.BudgetOptimizer(MMM)

def ds_a_df(ds):
    # xr.Dataset dell'ottimizzatore -> DataFrame per canale (metriche 'mean').
    df = ds.to_dataframe().reset_index()
    if 'metric' in df.columns and (df['metric'] == 'mean').any():
        df = df[df['metric'] == 'mean'].drop(columns='metric')
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return df.groupby('channel', as_index=False)[num].mean()

def optimize_budget(budget, lo, hi, fisso=True, target_mroi=None):
    kw = dict(fixed_budget=fisso, budget=float(budget),
              spend_constraint_lower=lo, spend_constraint_upper=hi)
    if not fisso:
        kw = dict(fixed_budget=False, target_mroi=float(target_mroi))
    return chiama(BO.optimize, **kw)

RES = optimize_budget(BUDGET, VINCOLO_LO, VINCOLO_HI, fisso=True)
assert RES is not None, "BudgetOptimizer.optimize non riuscito: controlla i messaggi sopra."

ATT = ds_a_df(RES.nonoptimized_data).set_index('channel').reindex(CANALI_MODELLO)
OTT = ds_a_df(RES.optimized_data).set_index('channel').reindex(CANALI_MODELLO)

def _col(df, *nomi):
    for n in nomi:
        if n in df.columns:
            return df[n].astype(float)
    return pd.Series(np.nan, index=df.index)

def _conv(x):
    # outcome del modello -> conversioni (se il ROAS e' monetario, divide per il valore medio)
    return x / RPK_MEDIO if (REVENUE_PER_KPI_PRESENTE and RPK_MEDIO) else x

# --- sessioni per canale (se rilevate; altrimenti click come proxy) ---
if 'sessions' in DF_TIDY.columns and DF_TIDY['sessions'].sum() > 0:
    SESSIONI = DF_TIDY.groupby('channel')['sessions'].sum().reindex(CANALI_MODELLO)
    _nota_sessioni = 'sessioni'
elif 'clicks' in DF_TIDY.columns and DF_TIDY['clicks'].sum() > 0:
    SESSIONI = DF_TIDY.groupby('channel')['clicks'].sum().reindex(CANALI_MODELLO)
    _nota_sessioni = 'click (proxy delle sessioni)'
else:
    SESSIONI = pd.Series(np.nan, index=CANALI_MODELLO)
    _nota_sessioni = 'non disponibili'

sp_att = _col(ATT, 'spend'); sp_ott = _col(OTT, 'spend')
inc_att = _conv(_col(ATT, 'incremental_outcome', 'incremental_impact'))
inc_ott = _conv(_col(OTT, 'incremental_outcome', 'incremental_impact'))

SINTESI = pd.DataFrame({
    'canale': CANALI_MODELLO,
    'gruppo': [GRUPPO_CANALE.get(c, GRUPPO_DEFAULT) for c in CANALI_MODELLO],
    'spesa_attuale': sp_att.round(0).values,
    'spesa_consigliata': sp_ott.round(0).values,
    'delta': (sp_ott - sp_att).round(0).values,
    'delta_%': (100 * (sp_ott - sp_att) / sp_att.replace(0, np.nan)).round(1).values,
    'quota_attuale_%': (100 * sp_att / sp_att.sum()).round(1).values,
    'quota_ottimale_%': (100 * sp_ott / sp_ott.sum()).round(1).values,
    'conversioni_attuali': inc_att.round(1).values,
    'conversioni_attese': inc_ott.round(1).values,
    'delta_conversioni': (inc_ott - inc_att).round(1).values,
    'cpa_attuale': (sp_att / inc_att.replace(0, np.nan)).round(2).values,
    'cpa_atteso': (sp_ott / inc_ott.replace(0, np.nan)).round(2).values,
    'sessioni': SESSIONI.round(0).values,
    'roas_attuale': _col(ATT, 'roi').round(3).values,
    'roas_ottimale': _col(OTT, 'roi').round(3).values,
    'mroas_attuale': _col(ATT, 'mroi').round(3).values,
    'mroas_ottimale': _col(OTT, 'mroi').round(3).values,
})
if IGP_FORMULA is not None:
    SINTESI['igp'] = IGP_FORMULA(SINTESI)   # colonna opzionale, solo se definita in CONFIG

totale = {'canale': 'TOTALE', 'gruppo': ''}
for c in SINTESI.columns:
    if c not in ('canale', 'gruppo', 'delta_%', 'cpa_attuale', 'cpa_atteso',
                 'roas_attuale', 'roas_ottimale', 'mroas_attuale', 'mroas_ottimale'):
        totale[c] = round(float(pd.to_numeric(SINTESI[c], errors='coerce').sum()), 1)
totale['cpa_attuale'] = round(float(sp_att.sum() / max(inc_att.sum(), 1e-9)), 2)
totale['cpa_atteso'] = round(float(sp_ott.sum() / max(inc_ott.sum(), 1e-9)), 2)
SINTESI_TOT = pd.concat([SINTESI, pd.DataFrame([totale])], ignore_index=True)

print("Nota sessioni: " + _nota_sessioni)
print(SINTESI_TOT.to_string(index=False))

# ---------------------------------------------------------------------------
# 12.0-bis VISTA AGGREGATA per gruppo + VISTA FINE per campagna
# ---------------------------------------------------------------------------
_col_grp = ['spesa_attuale', 'spesa_consigliata', 'delta', 'conversioni_attuali',
            'conversioni_attese', 'delta_conversioni']
SINTESI_GRUPPI = SINTESI.groupby('gruppo', as_index=False)[_col_grp].sum().round(1)
SINTESI_GRUPPI['quota_attuale_%'] = (100 * SINTESI_GRUPPI['spesa_attuale'] /
                                     SINTESI_GRUPPI['spesa_attuale'].sum()).round(1)
SINTESI_GRUPPI['quota_ottimale_%'] = (100 * SINTESI_GRUPPI['spesa_consigliata'] /
                                      SINTESI_GRUPPI['spesa_consigliata'].sum()).round(1)
print()
print("Vista aggregata per gruppo:")
print(SINTESI_GRUPPI.to_string(index=False))

# Riparto del budget consigliato sulle campagne: PROPORZIONALE alle quote storiche
# di spesa (dati di piattaforma), calcolato A VALLE del modello.
# NOTA ESPLICITA: questo riparto e' DESCRITTIVO e NON causale, a differenza dei
# ROAS incrementali del modello (che sono stime causali, ma solo a livello CANALE).
# Il modello non distingue l'efficacia delle singole campagne.
_camp = DF_CAMPAGNE.groupby(['channel', 'sottocanale', 'campagna'], as_index=False).agg(
    spesa_attuale=('spend', 'sum'),
    clicks=('clicks', 'sum') if 'clicks' in DF_CAMPAGNE.columns else ('spend', 'size'))
_camp = _camp.rename(columns={'channel': 'canale'})
_camp['quota_nel_canale_%'] = (100 * _camp['spesa_attuale'] /
                               _camp.groupby('canale')['spesa_attuale'].transform('sum')).round(1)
_camp = _camp.merge(SINTESI[['canale', 'spesa_consigliata', 'delta_%']]
                    .rename(columns={'spesa_consigliata': '_spesa_canale_ott',
                                     'delta_%': 'delta_%_canale'}), on='canale', how='left')
_camp['spesa_consigliata_riparto'] = (_camp['_spesa_canale_ott'] *
                                      _camp['quota_nel_canale_%'] / 100).round(0)
_camp['delta_riparto'] = (_camp['spesa_consigliata_riparto'] - _camp['spesa_attuale']).round(0)
RIPARTO_CAMPAGNE = _camp.drop(columns=['_spesa_canale_ott']).sort_values(
    ['canale', 'spesa_attuale'], ascending=[True, False]).reset_index(drop=True)
RIPARTO_CAMPAGNE['spesa_attuale'] = RIPARTO_CAMPAGNE['spesa_attuale'].round(0)
print()
print("Vista fine: riparto proporzionale per campagna (prime 15 righe):")
print(RIPARTO_CAMPAGNE.head(15).to_string(index=False))
print("NB: riparto proporzionale alle quote storiche, il modello non stima le singole campagne.")

# ---------------------------------------------------------------------------
# 12.1 Dal modello alla decisione: quanto spostare, da dove a dove
# ---------------------------------------------------------------------------
DECISIONI = []
guadagno = float((inc_ott - inc_att).sum())
DECISIONI.append("Effetto atteso della riallocazione (stesso budget di " +
                 format(int(BUDGET), ",") + " " + VALUTA_TARGET + "): " +
                 ("+" if guadagno >= 0 else "") + str(round(guadagno, 1)) +
                 " conversioni incrementali (" +
                 str(round(100 * guadagno / max(float(inc_att.sum()), 1e-9), 1)) + "%).")
for c in SINTESI.sort_values('delta')['canale']:
    r = SINTESI[SINTESI['canale'] == c].iloc[0]
    if pd.notna(r['delta']) and abs(r['delta']) > 0.005 * BUDGET:
        verso = "RIDUCI" if r['delta'] < 0 else "AUMENTA"
        DECISIONI.append(verso + " " + c + " di " + format(int(abs(r['delta'])), ",") + " " +
                         VALUTA_TARGET + " (" + str(r['delta_%']) + "%): mROAS attuale " +
                         str(r['mroas_attuale']) + " -> quota budget " +
                         str(r['quota_attuale_%']) + "% -> " + str(r['quota_ottimale_%']) + "%")
DECISIONI.append("Regola operativa: sposta budget dai canali con mROAS basso (saturi) a quelli " +
                 "con mROAS alto, entro i vincoli; ripeti l'ottimizzazione al prossimo trimestre.")
print()
for d in DECISIONI:
    print("  * " + d)

# --- grafico allocazione attuale vs ottimale ---
x = np.arange(len(CANALI_MODELLO))
plt.figure(figsize=(9, 4))
plt.bar(x - 0.2, sp_att.values, width=0.4, label='attuale')
plt.bar(x + 0.2, sp_ott.values, width=0.4, label='ottimale')
plt.xticks(x, CANALI_MODELLO, rotation=20, ha='right')
plt.ylabel('spesa (' + VALUTA_TARGET + ')')
plt.title('Allocazione del budget: attuale vs ottimale (budget fisso, vincoli +/-' +
          str(int(VINCOLO_SPESA_PCT * 100)) + '%)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'allocazione.png'), dpi=130)
plt.show()

# ---------------------------------------------------------------------------
# 12.2 Scenario secondario: budget flessibile (livello di spesa ottimale)
# ---------------------------------------------------------------------------
FLEX_DF = None
if scegli_opzione("Eseguire anche lo scenario a budget FLESSIBILE (target mROI=" +
                  str(TARGET_MROI) + ")?", ["Si'", "No"], default=0) == 0:
    RES_FLEX = optimize_budget(BUDGET, VINCOLO_LO, VINCOLO_HI, fisso=False,
                               target_mroi=TARGET_MROI)
    if RES_FLEX is not None:
        f = ds_a_df(RES_FLEX.optimized_data).set_index('channel').reindex(CANALI_MODELLO)
        FLEX_DF = pd.DataFrame({
            'canale': CANALI_MODELLO,
            'spesa_flessibile': _col(f, 'spend').round(0).values,
            'conversioni_attese': _conv(_col(f, 'incremental_outcome', 'incremental_impact')).round(1).values,
            'mroas': _col(f, 'mroi').round(3).values})
        print()
        print("Scenario budget flessibile (spesa al livello in cui mROI ~ " + str(TARGET_MROI) + "):")
        print(FLEX_DF.to_string(index=False))
        print("Budget totale suggerito: " + format(int(FLEX_DF['spesa_flessibile'].sum()), ",") +
              " " + VALUTA_TARGET + " vs storico " + format(int(SPESA_STORICA), ","))

# ---------------------------------------------------------------------------
# 12.3 What-if: outcome ottimale al variare del budget totale
# ---------------------------------------------------------------------------
righe_wi = []
for molt in SCENARI_MOLTIPLICATORI:
    r = optimize_budget(BUDGET * molt, VINCOLO_LO, VINCOLO_HI, fisso=True)
    if r is None:
        continue
    o = ds_a_df(r.optimized_data)
    conv_tot = float(_conv(_col(o.set_index('channel'), 'incremental_outcome',
                                'incremental_impact')).sum())
    righe_wi.append({'moltiplicatore': molt,
                     'budget': round(BUDGET * molt, 0),
                     'conversioni_incrementali_attese': round(conv_tot, 1),
                     'cpa_incrementale': round(BUDGET * molt / max(conv_tot, 1e-9), 2)})
WHATIF_DF = pd.DataFrame(righe_wi)
if len(WHATIF_DF):
    print()
    print(WHATIF_DF.to_string(index=False))
    plt.figure(figsize=(7, 4))
    plt.plot(WHATIF_DF['budget'], WHATIF_DF['conversioni_incrementali_attese'], marker='o')
    plt.axvline(BUDGET, color='gray', ls=':', label='budget attuale')
    plt.xlabel('budget totale (' + VALUTA_TARGET + ')')
    plt.ylabel('conversioni incrementali attese')
    plt.title('What-if: outcome ottimale al variare del budget (rendimenti decrescenti)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'whatif.png'), dpi=130)
    plt.show()

# --- report HTML nativo dell'ottimizzatore (facoltativo) ---
try:
    chiama(RES.output_optimization_summary,
           filename='ottimizzazione.html', filepath=OUTPUT_DIR)
    print("Report HTML ottimizzazione salvato in " + OUTPUT_DIR + "/ottimizzazione.html")
except Exception as e:
    print("[avviso] report HTML ottimizzazione non disponibile: " + str(e))

## 13. CHECKPOINT 6 + export Excel multi-foglio

Workbook `mmm_risultati_[YYYYMMDD].xlsx`. Di default esporta SOLO l'essenziale
(Report_Manager, Legenda, Dettaglio_Campagne, Scenari_WhatIf); con
`EXPORT_COMPLETO = True` nella CONFIG aggiunge tutti i fogli tecnici
(Sintesi per canale/gruppo, modello, EDA, diagnostica, dati puliti, README)
per l'appendice di tesi. Grafici nativi Excel dove possibile, PNG altrove.

In [ ]:
# ============================================================================
# CELLA 13 - EXPORT EXCEL MULTI-FOGLIO + download
# ============================================================================
from openpyxl import Workbook
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage

# Ordine di lettura: prima la vista manageriale, in fondo i fogli tecnici.
ORDINE_FOGLI = ['Report_Manager', 'Legenda', 'Sintesi_Gruppi', 'Sintesi_Budget',
                'Dettaglio_Campagne', 'Scenari_WhatIf', 'Budget_Dettaglio',
                'Modello_ROAS', 'Contributi', 'Confronto_ROAS', 'Response_curves',
                'EDA', 'Diagnostica', 'Dati_puliti', 'README']
FOGLI_ESSENZIALI = ['Report_Manager', 'Legenda', 'Dettaglio_Campagne', 'Scenari_WhatIf']
FOGLI = ORDINE_FOGLI if EXPORT_COMPLETO else [f for f in ORDINE_FOGLI if f in FOGLI_ESSENZIALI]

# ---------------------------------------------------------------------------
# Derivazioni di presentazione per il Report_Manager (linguaggio piano)
# ---------------------------------------------------------------------------
_conv_att_tot = float(pd.to_numeric(SINTESI['conversioni_attuali'], errors='coerce').sum())
_conv_ott_tot = float(pd.to_numeric(SINTESI['conversioni_attese'], errors='coerce').sum())
_kpi_osservato = float(DF_BASE['kpi'].sum())

WHATIF_MGR = None
if len(WHATIF_DF):
    WHATIF_MGR = WHATIF_DF.copy()
    WHATIF_MGR['variazione_budget_%'] = ((WHATIF_MGR['moltiplicatore'] - 1) * 100).round(0)
    WHATIF_MGR['delta_conversioni_vs_oggi'] = (
        WHATIF_MGR['conversioni_incrementali_attese'] - _conv_att_tot).round(1)
    WHATIF_MGR['delta_%_vs_oggi'] = (100 * WHATIF_MGR['delta_conversioni_vs_oggi'] /
                                     max(_conv_att_tot, 1e-9)).round(1)

# Stima per sottocanale: conversioni del canale ripartite sulle quote di spesa
# (il modello stima il CANALE; questa e' una ripartizione proporzionale dichiarata).
SOTTOCANALI_STIMA = RIPARTO_CAMPAGNE.groupby(['canale', 'sottocanale'], as_index=False).agg(
    spesa_attuale=('spesa_attuale', 'sum'),
    spesa_consigliata=('spesa_consigliata_riparto', 'sum'),
    n_campagne=('campagna', 'nunique'))
SOTTOCANALI_STIMA['quota_nel_canale_%'] = (
    100 * SOTTOCANALI_STIMA['spesa_attuale'] /
    SOTTOCANALI_STIMA.groupby('canale')['spesa_attuale'].transform('sum')).round(1)
SOTTOCANALI_STIMA = SOTTOCANALI_STIMA.merge(
    SINTESI[['canale', 'conversioni_attuali']].rename(
        columns={'conversioni_attuali': '_conv_canale'}), on='canale', how='left')
SOTTOCANALI_STIMA['conversioni_stimate'] = (
    SOTTOCANALI_STIMA['_conv_canale'] * SOTTOCANALI_STIMA['quota_nel_canale_%'] / 100).round(1)
SOTTOCANALI_STIMA['cpa_stimato'] = (SOTTOCANALI_STIMA['spesa_attuale'] /
    SOTTOCANALI_STIMA['conversioni_stimate'].replace(0, np.nan)).round(2)
SOTTOCANALI_STIMA = SOTTOCANALI_STIMA.drop(columns=['_conv_canale']).sort_values(
    ['canale', 'spesa_attuale'], ascending=[True, False])

anteprima = SINTESI_TOT[['canale', 'spesa_attuale', 'spesa_consigliata', 'delta_%',
                         'conversioni_attuali', 'conversioni_attese']].to_string(index=False)
checkpoint("6. Export Excel", ["Fogli: " + ", ".join(FOGLI),
                               "Anteprima Sintesi_Budget:"] + anteprima.split(chr(10)))

NOME_XLSX = os.path.join(OUTPUT_DIR, 'mmm_risultati_' +
                         dt.date.today().strftime('%Y%m%d') + '.xlsx')

_BLU = PatternFill('solid', fgColor='1F4E78')
_GRIGIO = PatternFill('solid', fgColor='D9E1F2')

def scrivi_df(ws, df, riga=1, titolo=None):
    # Scrive un DataFrame con header formattato; restituisce la prossima riga libera.
    if titolo:
        ws.cell(row=riga, column=1, value=titolo).font = Font(bold=True, size=12)
        riga += 1
    for j, col in enumerate(df.columns, start=1):
        c = ws.cell(row=riga, column=j, value=str(col))
        c.font = Font(bold=True, color='FFFFFF'); c.fill = _BLU
    for i, (_, r) in enumerate(df.iterrows(), start=riga + 1):
        for j, col in enumerate(df.columns, start=1):
            v = r[col]
            if isinstance(v, (np.integer,)): v = int(v)
            if isinstance(v, (np.floating,)): v = float(v)
            if isinstance(v, pd.Timestamp): v = v.date()
            if isinstance(v, float) and (math.isnan(v) or math.isinf(v)): v = None
            ws.cell(row=i, column=j, value=v)
    for j, col in enumerate(df.columns, start=1):
        ws.column_dimensions[get_column_letter(j)].width = max(12, min(32, len(str(col)) + 4))
    return riga + len(df) + 2

def inserisci_png(ws, nome_png, cella):
    percorso = os.path.join(FIG_DIR, nome_png)
    if os.path.exists(percorso):
        try:
            img = XLImage(percorso); img.width = int(img.width * 0.75)
            img.height = int(img.height * 0.75)
            ws.add_image(img, cella)
        except Exception as e:
            print("[avviso] PNG " + nome_png + " non inserito: " + str(e))

def export_excel():
    wb = Workbook(); wb.remove(wb.active)

    # --- Report_Manager: quanto spendiamo, cosa otteniamo, cosa fare ---
    ws = wb.create_sheet('Report_Manager')
    quota_media = round(100 * _conv_att_tot / max(_kpi_osservato, 1e-9), 1)
    guadagno_r = _conv_ott_tot - _conv_att_tot
    sintesi_mgr = pd.DataFrame([
        ('Periodo analizzato', CONFIG['periodo'][0] + ' -> ' + CONFIG['periodo'][1]),
        ('Spesa media totale', format(int(BUDGET), ',') + ' ' + VALUTA_TARGET),
        ('Risultato misurato (KPI)', KPI_DESCR.split('::')[-1].split('(')[0].strip()),
        ('Conversioni totali del periodo', format(int(_kpi_osservato), ',')),
        ('  di cui generate dalla pubblicita (incrementali)',
         format(int(_conv_att_tot), ',') + '  (' + str(quota_media) + '% del totale)'),
        ('  che sarebbero arrivate comunque (baseline)',
         format(int(_kpi_osservato - _conv_att_tot), ',')),
        ('Costo per conversione incrementale (CPA medio)',
         str(round(BUDGET / max(_conv_att_tot, 1e-9), 2)) + ' ' + VALUTA_TARGET),
        ('Riallocando LO STESSO budget come consigliato',
         format(int(_conv_ott_tot), ',') + ' conversioni attese  (' +
         ('+' if guadagno_r >= 0 else '') + str(int(guadagno_r)) + ', ' +
         ('+' if guadagno_r >= 0 else '') +
         str(round(100 * guadagno_r / max(_conv_att_tot, 1e-9), 1)) + '%)'),
    ], columns=['voce', 'valore'])
    riga = scrivi_df(ws, sintesi_mgr, titolo='In sintesi')

    grp_mgr = SINTESI_GRUPPI.rename(columns={
        'gruppo': 'tipologia di canale', 'spesa_attuale': 'speso (' + VALUTA_TARGET + ')',
        'spesa_consigliata': 'spesa consigliata', 'conversioni_attuali': 'conversioni ottenute',
        'conversioni_attese': 'conversioni attese', 'delta_conversioni': 'conversioni in piu/meno',
        'quota_attuale_%': 'quota budget oggi %', 'quota_ottimale_%': 'quota consigliata %'})
    riga = scrivi_df(ws, grp_mgr.drop(columns=['delta'], errors='ignore'), riga,
                     titolo='Per tipologia di canale: quanto speso, cosa ottenuto, cosa consigliato')

    can_mgr = SINTESI[['canale', 'spesa_attuale', 'quota_attuale_%',
                       'conversioni_attuali', 'cpa_attuale', 'spesa_consigliata',
                       'delta', 'conversioni_attese']].rename(columns={
        'spesa_attuale': 'speso', 'quota_attuale_%': 'quota %',
        'conversioni_attuali': 'conversioni ottenute', 'cpa_attuale': 'CPA',
        'spesa_consigliata': 'spesa consigliata', 'delta': 'sposta (' + VALUTA_TARGET + ')',
        'conversioni_attese': 'conversioni attese'})
    riga = scrivi_df(ws, can_mgr, riga, titolo='Per canale')

    sc_mgr = SOTTOCANALI_STIMA.rename(columns={
        'spesa_attuale': 'speso', 'spesa_consigliata': 'spesa consigliata',
        'quota_nel_canale_%': 'quota nel canale %',
        'conversioni_stimate': 'conversioni stimate*', 'cpa_stimato': 'CPA stimato*'})
    riga = scrivi_df(ws, sc_mgr, riga,
                     titolo='Per tipologia di campagna (rilevata in automatico dai file)')

    top_camp = RIPARTO_CAMPAGNE[['canale', 'sottocanale', 'campagna', 'spesa_attuale',
                                 'quota_nel_canale_%', 'spesa_consigliata_riparto',
                                 'delta_riparto']].head(20).rename(columns={
        'spesa_attuale': 'speso', 'quota_nel_canale_%': 'quota nel canale %',
        'spesa_consigliata_riparto': 'spesa consigliata*', 'delta_riparto': 'sposta*'})
    riga = scrivi_df(ws, top_camp, riga, titolo='Le campagne principali (prime 20 per spesa)')

    if WHATIF_MGR is not None:
        wi_mgr = WHATIF_MGR[['budget', 'variazione_budget_%',
                             'conversioni_incrementali_attese',
                             'delta_conversioni_vs_oggi', 'delta_%_vs_oggi']].rename(columns={
            'budget': 'budget totale', 'variazione_budget_%': 'variazione budget %',
            'conversioni_incrementali_attese': 'conversioni attese',
            'delta_conversioni_vs_oggi': 'conversioni in piu/meno vs oggi',
            'delta_%_vs_oggi': 'variazione conversioni %'})
        riga = scrivi_df(ws, wi_mgr, riga,
                         titolo='Quanto dovrei spendere per ottenere di piu/meno? (budget ottimizzato)')
        try:
            obiettivo = _conv_att_tot * 1.10
            serie_c = WHATIF_MGR['conversioni_incrementali_attese']
            if serie_c.min() <= obiettivo <= serie_c.max():
                bdg10 = float(np.interp(obiettivo, serie_c, WHATIF_MGR['budget']))
                ws.cell(row=riga, column=1, value='Per un +10% di conversioni incrementali '
                        'servono circa ' + format(int(bdg10), ',') + ' ' + VALUTA_TARGET +
                        ' di budget totale (stima interpolata, budget riallocato in modo ottimale).')
                riga += 2
        except Exception:
            pass

    ws.cell(row=riga, column=1, value='Cosa fare (piano operativo)').font = Font(bold=True, size=12)
    riga += 1
    for d in DECISIONI:
        ws.cell(row=riga, column=1, value=d)
        riga += 1
    riga += 1

    ws.cell(row=riga, column=1,
            value='* Stime ripartite in proporzione alla spesa dentro il canale: il modello misura '
                  'l effetto a livello di canale. Campagne e tipologie sono rilevate in automatico '
                  'dai file caricati a ogni run.' +
                  ('' if EXPORT_COMPLETO else ' Per i dettagli tecnici (modello, incertezza, '
                   'diagnostica, dati puliti) riesegui l export con EXPORT_COMPLETO = True.'))
    ws.column_dimensions['A'].width = 46

    # --- README ---
    ws = wb.create_sheet('README')
    info = [('Generato il', str(dt.datetime.now())[:19]), ('Seed', SEED),
            ('Periodo modellazione', CONFIG['periodo'][0] + ' -> ' + CONFIG['periodo'][1]),
            ('Unione periodi fonti', CONFIG['periodo_unione'][0] + ' -> ' + CONFIG['periodo_unione'][1]),
            ('Granularita', 'settimana ISO (lunedi); fonti: ' + str(CONFIG['granularita'])),
            ('Livello geo', GEO_LIVELLO + ' (' + str(len(GEOS)) + ' geo)'),
            ('Valuta', VALUTA_TARGET + '; trovate: ' + str(CONFIG['valute_trovate']) +
             '; tassi usati: ' + str(TASSI_CAMBIO)),
            ('KPI', KPI_DESCR),
            ('revenue_per_kpi', (str(round(RPK_MEDIO, 2)) if RPK_MEDIO else 'assente') +
             ' -> outcome in ' + UNITA_OUTCOME),
            ('Modello', 'Google Meridian geo-gerarchico bayesiano, kpi_type=non_revenue'),
            ('Priori', ('ROI ~ LogNormal per canale, ancorato ai ROAS di riferimento '
                        '(sigma=' + str(PRIOR_ROI_SIGMA_INF) + '); '
                        if globals().get('PRIOR_INFO_ATTIVI') else
                        'ROI ~ LogNormal(' + str(PRIOR_ROI_MU) + ', ' +
                        str(PRIOR_ROI_SIGMA) + ') debolmente informativi; ') +
                       str(N_KNOTS) + ' nodi baseline'),
            # Due run con prior diversi devono essere distinguibili dal solo
            # workbook, senza tornare al notebook che li ha prodotti.
            ('Prior informativi', 'SI' if globals().get('PRIOR_INFO_ATTIVI') else 'NO'),
            ('Prior - valori usati', ' | '.join(globals().get('PRIOR_DETTAGLIO', []))
             or 'n/d'),
            ('Prior - ROAS di riferimento',
             json.dumps(globals().get('PRIOR_ROAS_CANALE', {}), ensure_ascii=False)
             if globals().get('PRIOR_INFO_ATTIVI') else 'nessuno (prior non informativi)'),
            ('Origine dati', str(globals().get('ORIGINE_DATI', 'n/d'))),
            ('Durata fit (min)', (str(round(float(globals().get('DURATA_FIT_MIN')), 1))
                                  if globals().get('DURATA_FIT_MIN') is not None else 'n/d')),
            ('Confronto benchmark', str(globals().get('CONFRONTO_SALTATO'))
             if globals().get('CONFRONTO_SALTATO') else 'eseguito'),
            ('Campionamento', str(N_CHAINS) + ' catene, adapt=' + str(N_ADAPT) + ', burnin=' +
             str(N_BURNIN) + ', keep=' + str(N_KEEP)),
            ('Goal ottimizzazione', GOAL_OTTIMIZZAZIONE + ' | budget=' + format(int(BUDGET), ',') +
             ' | vincoli +/-' + str(int(VINCOLO_SPESA_PCT * 100)) + '% ' + str(VINCOLI_PER_CANALE)),
            ('Benchmark ROAS', 'attribution multi-touch (ultimi 3 touchpoint): usato SOLO come ' +
             'benchmark, MAI come priore di calibrazione'),
            ('Convenzioni', 'media=0 -> nessuna attivita; NaN KPI gestiti come da data quality report; ' +
             'fonti mensili ripartite uniformemente sulle settimane'),
            ('Versioni', json.dumps(VERSIONI))]
    riga = scrivi_df(ws, pd.DataFrame(info, columns=['voce', 'valore']), titolo='MMM - README')
    riga = scrivi_df(ws, pd.DataFrame([{'file': l['file'], 'ruolo': l['ruolo_file'],
                                        'cadenza': str(l.get('cadenza')),
                                        'valute': str(l.get('valute')),
                                        'geo': str(l.get('geo'))} for _, l in INGEST]),
                     riga, titolo='Fonti dati')
    scrivi_df(ws, pd.DataFrame(DQ_REPORT) if DQ_REPORT else
              pd.DataFrame([{'problema': 'nessuno', 'scelta': ''}]),
              riga, titolo='Data quality report')

    # --- Legenda: come leggere il file, per chi non conosce l'MMM ---
    ws = wb.create_sheet('Legenda')
    mappa_fogli = pd.DataFrame([
        ('Report_Manager', 'IL FOGLIO DA CUI PARTIRE: quanto abbiamo speso, cosa abbiamo ottenuto (per tipologia di canale e di campagna) e quanto spendere per ottenere di piu\'. Senza tecnicismi.'),
        ('Legenda', 'Questo foglio: mappa dei fogli + glossario dei termini.'),
        ('Sintesi_Gruppi', 'VISTA AGGREGATA: allocazione attuale vs consigliata per GRUPPO (paid media, job board, owned...).'),
        ('Sintesi_Budget', 'VISTA PER CANALE: la tabella decisionale completa (spesa attuale vs consigliata + decisioni operative).'),
        ('Dettaglio_Campagne', 'VISTA FINE: sottocanali (es. Performance Max, Display) e singole campagne, con riparto del budget consigliato.'),
        ('Scenari_WhatIf', 'Cosa succederebbe con un budget totale diverso (-20%...+20%) o flessibile.'),
        ('Budget_Dettaglio', 'Un blocco di approfondimento per ogni canale: CPA, mROAS, curva di risposta.'),
        ('Modello_ROAS', '(tecnico) Risultati causali del modello per canale, con incertezza (intervalli).'),
        ('Contributi', '(tecnico) Quanto del KPI arriva da solo (baseline) e quanto e\' generato da ciascun canale.'),
        ('Confronto_ROAS', '(tecnico) ROAS del modello (incrementale) vs ROAS aziendale (attribuito), per trimestre. Il gap e\' un risultato, non un errore.'),
        ('Response_curves', '(tecnico) Rendimenti decrescenti: cosa succede alle conversioni spendendo di piu\'/di meno su ogni canale.'),
        ('EDA', '(tecnico) Esplorazione descrittiva: copertura, trend, correlazioni, collinearita\' (VIF). Non e\' ancora il modello.'),
        ('Diagnostica', '(tecnico) Qualita\' statistica del modello: dice quanto fidarsi dei numeri degli altri fogli.'),
        ('Dati_puliti', '(tecnico) La base dati armonizzata riga per riga (geo x settimana x canale). Per verifiche e analisi ad hoc.'),
        ('README', '(tecnico) Configurazione della run, fonti dati, scelte di pulizia. Spiega cosa c\'e\' dietro i numeri.'),
    ], columns=['foglio', 'cosa contiene e a cosa serve'])
    # nella versione essenziale la mappa elenca solo i fogli davvero presenti
    mappa_fogli = mappa_fogli[mappa_fogli['foglio'].isin(FOGLI)].reset_index(drop=True)
    riga = scrivi_df(ws, mappa_fogli, titolo='Mappa dei fogli (in ordine di lettura)')

    glossario = pd.DataFrame([
        ('Gerarchia', 'GRUPPO > CANALE > SOTTOCANALE > CAMPAGNA. Es: Paid media > Google Ads > Performance Max > "PMax Lombardia". Il modello stima gli effetti a livello CANALE; sottocanali e campagne sono viste descrittive.'),
        ('KPI / conversioni', 'L\'outcome che il modello spiega e ottimizza (es. candidature).'),
        ('Incrementale', 'Conversioni che NON ci sarebbero state senza quella spesa (effetto causale stimato dal modello). Diverso da "attribuito".'),
        ('Attribuito', 'Credito assegnato dai modelli di attribution (es. ultimi 3 touchpoint in GA): descrittivo, tende a premiare i canali vicini alla conversione.'),
        ('Baseline', 'Conversioni che arriverebbero comunque: brand, stagionalita\', domanda organica, controlli.'),
        ('ROAS', 'Ritorno per euro speso (outcome incrementale / spesa). Se monetario e >1, il canale ripaga la spesa.'),
        ('mROAS (marginale)', 'Ritorno dell\'ULTIMO euro speso. E\' la bussola delle decisioni: >1 conviene aumentare, <1 conviene ridurre. Cala man mano che il canale satura.'),
        ('CPA', 'Costo per conversione incrementale (spesa / conversioni incrementali).'),
        ('ci05 / ci95 (CI 90%)', 'Intervallo di credibilita\': il valore vero sta li\' dentro con probabilita\' 90%. Intervalli larghi = stima incerta.'),
        ('Saturazione', 'Oltre un certo livello di spesa ogni euro in piu\' rende meno (curve nel foglio Response_curves).'),
        ('Spesa consigliata', 'Allocazione che massimizza le conversioni incrementali a budget totale fisso, entro i vincoli (es. +/-30% per canale).'),
        ('Riparto per campagna', 'Suddivisione del budget consigliato del canale sulle campagne in proporzione alle quote storiche: punto di partenza operativo, NON stima causale della singola campagna.'),
        ('Sessioni', 'Se non presenti nei dati, vengono usati i click come proxy (indicato in tabella).'),
        ('VIF', 'Misura di collinearita\' tra canali: >5 = i canali si muovono insieme e le loro stime sono meno precise.'),
        ('R-hat / ESS / divergenze', 'Diagnostica del campionamento bayesiano: ok se R-hat <= 1.1, divergenze = 0, ESS >= 100.'),
        ('0 vs cella vuota', '0 = nessuna attivita\' in quella settimana; vuoto = dato mancante (gestito come da README).'),
        ('Benchmark', 'Il ROAS aziendale di attribution: usato SOLO come termine di confronto, mai per calibrare il modello.'),
    ], columns=['termine', 'significato'])
    if not EXPORT_COMPLETO:
        # versione business: solo i termini che compaiono davvero nei fogli essenziali
        termini_business = ['Gerarchia', 'KPI / conversioni', 'Incrementale', 'Baseline',
                            'CPA', 'mROAS (marginale)', 'Saturazione', 'Spesa consigliata',
                            'Riparto per campagna']
        glossario = glossario[glossario['termine'].isin(termini_business)].reset_index(drop=True)
    scrivi_df(ws, glossario, riga, titolo='Glossario')
    ws.column_dimensions['A'].width = 24
    ws.column_dimensions['B'].width = 140

    # --- Dati_puliti ---
    ws = wb.create_sheet('Dati_puliti')
    riga = scrivi_df(ws, DF_TIDY, titolo='Base tidy: geo x settimana x canale (' + VALUTA_TARGET + ')')
    scrivi_df(ws, DF_BASE, riga, titolo='KPI, population e controlli per geo x settimana')

    # --- EDA (tabelle + grafico nativo + PNG) ---
    ws = wb.create_sheet('EDA')
    riga = scrivi_df(ws, EDA_TABELLE['copertura'], titolo='Copertura e quote di spesa per canale')
    riga = scrivi_df(ws, EDA_TABELLE['gruppi'], riga, titolo='Vista aggregata per gruppo')
    riga = scrivi_df(ws, EDA_TABELLE['sottocanali'], riga,
                     titolo='Vista fine per sottocanale (descrittiva)')
    riga = scrivi_df(ws, EDA_TABELLE['vif'], riga, titolo='VIF (collinearita tra canali: >5 = alta)')
    riga = scrivi_df(ws, EDA_TABELLE['outlier'], riga, titolo='Outlier (oltre 3 IQR)')
    riga = scrivi_df(ws, EDA_TABELLE['correlazioni'], riga, titolo='Correlazioni spesa/KPI')
    riga_trend = riga
    trend = EDA_TABELLE['trend'].copy()
    trend['week'] = trend['week'].astype(str)
    riga = scrivi_df(ws, trend, riga, titolo='Serie settimanali (spesa per canale + KPI)')
    try:
        ch = LineChart(); ch.title = 'KPI per settimana'; ch.height = 8; ch.width = 22
        col_kpi = list(trend.columns).index('kpi') + 1
        ch.add_data(Reference(ws, min_col=col_kpi, min_row=riga_trend + 1,
                              max_row=riga_trend + 1 + len(trend)), titles_from_data=True)
        ch.set_categories(Reference(ws, min_col=1, min_row=riga_trend + 2,
                                    max_row=riga_trend + 1 + len(trend)))
        ws.add_chart(ch, 'N2')
    except Exception as e:
        print('[avviso] grafico nativo EDA non creato: ' + str(e))
    inserisci_png(ws, 'eda_trend.png', 'N20')
    inserisci_png(ws, 'eda_correlazioni.png', 'N45')
    inserisci_png(ws, 'eda_stagionalita.png', 'N70')
    inserisci_png(ws, 'eda_spesa_vs_kpi.png', 'N90')

    # --- Modello_ROAS ---
    ws = wb.create_sheet('Modello_ROAS')
    scrivi_df(ws, TAB_ROAS, titolo='ROAS/mROAS incrementali con intervalli di credibilita 90% (' +
              UNITA_OUTCOME + ' per ' + VALUTA_TARGET + ')')
    try:
        ch = BarChart(); ch.title = 'ROAS per canale'; ch.height = 8; ch.width = 18
        col_r = list(TAB_ROAS.columns).index('roas') + 1
        ch.add_data(Reference(ws, min_col=col_r, min_row=2, max_row=2 + len(TAB_ROAS)),
                    titles_from_data=True)
        ch.set_categories(Reference(ws, min_col=1, min_row=3, max_row=2 + len(TAB_ROAS)))
        ws.add_chart(ch, 'B' + str(len(TAB_ROAS) + 5))
    except Exception as e:
        print('[avviso] grafico nativo ROAS non creato: ' + str(e))
    inserisci_png(ws, 'modello_roas.png', 'L2')

    # --- Confronto_ROAS ---
    ws = wb.create_sheet('Confronto_ROAS')
    if CONFRONTO is not None:
        scrivi_df(ws, CONFRONTO, titolo='MMM incrementale vs benchmark attribuito (per trimestre): ' +
                  'il gap e un risultato da interpretare, non da azzerare')
        inserisci_png(ws, 'confronto_roas.png', 'M2')
    else:
        ws['A1'] = 'Confronto saltato: nessun file benchmark caricato.'

    # --- Contributi ---
    ws = wb.create_sheet('Contributi')
    scrivi_df(ws, TAB_CONTRIB, titolo='Scomposizione del KPI: baseline vs canali')
    inserisci_png(ws, 'contributi.png', 'F2')

    # --- Response_curves ---
    ws = wb.create_sheet('Response_curves')
    if RC_DF is not None:
        scrivi_df(ws, RC_DF.round(3), titolo='Curve di risposta (spesa -> outcome incrementale)')
    else:
        ws['A1'] = 'Curve di risposta non disponibili in questa versione di Meridian.'
    inserisci_png(ws, 'response_curves.png', 'H2')

    # --- Sintesi_Gruppi (vista aggregata) ---
    ws = wb.create_sheet('Sintesi_Gruppi')
    riga = scrivi_df(ws, SINTESI_GRUPPI,
                     titolo='Vista aggregata per gruppo: allocazione attuale vs consigliata')
    ws.cell(row=riga, column=1,
            value='Lettura: i gruppi aggregano i canali del modello (vedi Legenda). '
                  'Il dettaglio per canale e nel foglio Sintesi_Budget; '
                  'quello per campagna in Dettaglio_Campagne.')
    try:
        ch = BarChart(); ch.type = 'col'; ch.height = 9; ch.width = 18
        ch.title = 'Spesa attuale vs consigliata per gruppo'
        ch.add_data(Reference(ws, min_col=2, max_col=3, min_row=2, max_row=2 + len(SINTESI_GRUPPI)),
                    titles_from_data=True)
        ch.set_categories(Reference(ws, min_col=1, min_row=3, max_row=2 + len(SINTESI_GRUPPI)))
        ws.add_chart(ch, 'B' + str(len(SINTESI_GRUPPI) + 7))
    except Exception as e:
        print('[avviso] grafico Sintesi_Gruppi non creato: ' + str(e))

    # --- Sintesi_Budget (vista generica: una tabella per canale + riga totale) ---
    ws = wb.create_sheet('Sintesi_Budget')
    scrivi_df(ws, SINTESI_TOT, titolo='Allocazione: attuale vs ottimale (goal: ' +
              GOAL_OTTIMIZZAZIONE + ', sessioni = ' + _nota_sessioni + ')')
    try:
        ch = BarChart(); ch.type = 'col'; ch.height = 9; ch.width = 20
        ch.title = 'Spesa attuale vs consigliata'
        ch.add_data(Reference(ws, min_col=2, max_col=3, min_row=2, max_row=2 + len(SINTESI)),
                    titles_from_data=True)
        ch.set_categories(Reference(ws, min_col=1, min_row=3, max_row=2 + len(SINTESI)))
        ws.add_chart(ch, 'B' + str(len(SINTESI_TOT) + 6))
    except Exception as e:
        print('[avviso] grafico nativo Sintesi_Budget non creato: ' + str(e))
    riga = len(SINTESI_TOT) + 26
    ws.cell(row=riga, column=1, value='Decisioni (piano operativo)').font = Font(bold=True)
    for i, d in enumerate(DECISIONI, start=1):
        ws.cell(row=riga + i, column=1, value=d)
    inserisci_png(ws, 'allocazione.png', 'T2')

    # --- Budget_Dettaglio: un blocco per canale ---
    ws = wb.create_sheet('Budget_Dettaglio')
    riga = 1
    for c in CANALI_MODELLO:
        r = SINTESI[SINTESI['canale'] == c].iloc[0]
        det = pd.DataFrame([
            ('spesa attuale', r['spesa_attuale']), ('spesa consigliata', r['spesa_consigliata']),
            ('delta', r['delta']), ('delta %', r['delta_%']),
            ('conversioni attuali -> attese', str(r['conversioni_attuali']) + ' -> ' +
             str(r['conversioni_attese'])),
            ('CPA attuale -> atteso', str(r['cpa_attuale']) + ' -> ' + str(r['cpa_atteso'])),
            ('ROAS (CI 90%)', str(TAB_ROAS.set_index('canale').loc[c, 'roas']) + '  [' +
             str(TAB_ROAS.set_index('canale').loc[c, 'roas_ci05']) + '; ' +
             str(TAB_ROAS.set_index('canale').loc[c, 'roas_ci95']) + ']'),
            ('mROAS attuale', r['mroas_attuale']),
            ('metrica esecuzione', METRICA_CANALE.get(c, 'n/d')),
        ], columns=['voce', 'valore'])
        riga = scrivi_df(ws, det, riga, titolo='=== ' + c + ' ===')
        if RC_DF is not None:
            col_ch = next((x for x in RC_DF.columns if 'channel' in x), None)
            if col_ch is not None:
                rc_c = RC_DF[RC_DF[col_ch] == c].round(2)
                if len(rc_c):
                    riga = scrivi_df(ws, rc_c, riga, titolo='Curva di risposta ' + c)

    # --- Dettaglio_Campagne (vista fine: sottocanali e singole campagne) ---
    ws = wb.create_sheet('Dettaglio_Campagne')
    riga = scrivi_df(ws, EDA_TABELLE['sottocanali'],
                     titolo='Sottocanali per canale (es. Performance Max, Display dentro Google Ads)')
    col_business = ['canale', 'sottocanale', 'campagna', 'spesa_attuale',
                    'quota_nel_canale_%', 'spesa_consigliata_riparto', 'delta_riparto']
    riga = scrivi_df(ws, RIPARTO_CAMPAGNE[[c for c in col_business
                                           if c in RIPARTO_CAMPAGNE.columns]], riga,
                     titolo='Singole campagne: spesa storica e riparto del budget consigliato')
    ws.cell(row=riga, column=1,
            value='ATTENZIONE: il modello stima gli effetti CAUSALI solo a livello CANALE '
                  '(i ROAS incrementali dei fogli modello). Il riparto per campagna di questo '
                  'foglio e invece DESCRITTIVO: proporzionale alle quote di spesa storiche dei '
                  'dati di piattaforma, calcolato a valle del modello. Non e una stima causale '
                  'della singola campagna: per confrontare le campagne tra loro servono i CPA '
                  'di piattaforma o test dedicati.')
    if EXPORT_COMPLETO:
        piv_sc = DF_CAMPAGNE.copy()
        piv_sc['serie'] = piv_sc['channel'] + ' / ' + piv_sc['sottocanale']
        piv_sc = piv_sc.pivot_table(index='week', columns='serie', values='spend',
                                    aggfunc='sum').fillna(0).round(0).reset_index()
        piv_sc['week'] = piv_sc['week'].astype(str)
        scrivi_df(ws, piv_sc, riga + 2, titolo='Spesa settimanale per canale/sottocanale')

    # --- Scenari_WhatIf (in linguaggio piano) ---
    ws = wb.create_sheet('Scenari_WhatIf')
    ws.cell(row=1, column=1,
            value='Come leggere: ogni riga e un livello di budget totale alternativo; per ogni '
                  'livello la spesa e gia riallocata al meglio tra i canali. '
                  'La riga in grigio e il budget attuale.').font = Font(italic=True)
    riga = 3
    if WHATIF_MGR is not None:
        wi = WHATIF_MGR[['budget', 'variazione_budget_%', 'conversioni_incrementali_attese',
                         'delta_conversioni_vs_oggi', 'delta_%_vs_oggi',
                         'cpa_incrementale']].rename(columns={
            'budget': 'budget totale (' + VALUTA_TARGET + ')',
            'variazione_budget_%': 'variazione budget %',
            'conversioni_incrementali_attese': 'conversioni attese',
            'delta_conversioni_vs_oggi': 'conversioni in piu/meno vs oggi',
            'delta_%_vs_oggi': 'variazione conversioni %',
            'cpa_incrementale': 'costo per conversione'})
        riga_tab = riga
        riga = scrivi_df(ws, wi, riga, titolo='Se cambiassi il budget totale')
        for i in range(len(wi)):
            if abs(float(wi.iloc[i]['variazione budget %'])) < 1e-9:
                for j in range(1, len(wi.columns) + 1):
                    ws.cell(row=riga_tab + 2 + i, column=j).fill = _GRIGIO
        # grafico corretto: UNA curva, budget sull asse X, conversioni sull asse Y
        try:
            from openpyxl.chart import ScatterChart, Series
            ch = ScatterChart()
            ch.title = 'Conversioni attese al variare del budget'
            ch.style = 13; ch.height = 9; ch.width = 18
            ch.x_axis.title = 'budget totale (' + VALUTA_TARGET + ')'
            ch.y_axis.title = 'conversioni attese'
            xref = Reference(ws, min_col=1, min_row=riga_tab + 2, max_row=riga_tab + 1 + len(wi))
            yref = Reference(ws, min_col=3, min_row=riga_tab + 1, max_row=riga_tab + 1 + len(wi))
            serie = Series(yref, xref, title_from_data=True)
            ch.series.append(serie)
            ws.add_chart(ch, 'I3')
        except Exception as e:
            print('[avviso] grafico what-if non creato: ' + str(e))
    if FLEX_DF is not None:
        flex = FLEX_DF.rename(columns={
            'spesa_flessibile': 'spesa suggerita (' + VALUTA_TARGET + ')',
            'conversioni_attese': 'conversioni attese',
            'mroas': 'mROAS marginale'})
        riga = scrivi_df(ws, flex, riga,
                         titolo='Scenario a budget libero: spendo su ogni canale finche '
                                'l ultimo euro rende almeno ' + str(TARGET_MROI))
        ws.cell(row=riga, column=1,
                value='Budget totale suggerito da questo scenario: ' +
                      format(int(FLEX_DF['spesa_flessibile'].sum()), ',') + ' ' + VALUTA_TARGET +
                      ' (attuale: ' + format(int(BUDGET), ',') +
                      '). Riferimento teorico, senza tetto di budget.')
        riga += 2
    inserisci_png(ws, 'whatif.png', 'I22')

    # --- Diagnostica ---
    ws = wb.create_sheet('Diagnostica')
    riga = scrivi_df(ws, pd.DataFrame([
        {'indicatore': 'Divergenze NUTS', 'valore': DIVERGENZE, 'soglia': '0'},
        {'indicatore': 'R-hat max', 'valore': RHAT_MAX, 'soglia': '<= 1.1'},
        {'indicatore': 'ESS min', 'valore': float(DIAG_TABELLA['ess_min'].min()), 'soglia': '>= 100'},
    ]), titolo='Convergenza del campionamento')
    riga = scrivi_df(ws, DIAG_TABELLA, riga, titolo='R-hat / ESS per gruppo di parametri')
    if FIT_DF is not None:
        riga = scrivi_df(ws, FIT_DF.round(1), riga, titolo='Atteso (modello) vs osservato')
    inserisci_png(ws, 'diagnostica_fit.png', 'F2')

    # tiene solo i fogli richiesti (EXPORT_COMPLETO=False -> solo l'essenziale) e li ordina
    for nome in list(wb.sheetnames):
        if nome not in FOGLI:
            wb.remove(wb[nome])
    try:
        wb._sheets = [wb[nome] for nome in FOGLI if nome in wb.sheetnames]
    except Exception as e:
        print('[avviso] riordino fogli non riuscito: ' + str(e))
    wb.save(NOME_XLSX)
    return NOME_XLSX

percorso = export_excel()
print('Excel salvato: ' + percorso)

# --- requirements con versioni + pacchetto finale ---
# requirements_freeze.txt e' gia' stato scritto nella cella 1, subito dopo
# l'installazione. Qui viene riscritto per fotografare lo stato di fine run:
# se il run si e' interrotto prima, resta comunque la versione della cella 1.
try:
    with open(PERCORSO_FREEZE, 'w', encoding='utf-8') as _f:
        _f.write(__import__('subprocess').run(
            [__import__('sys').executable, '-m', 'pip', 'freeze'],
            capture_output=True, text=True, check=True).stdout)
except Exception as _e:
    print('[avviso] aggiornamento requirements_freeze.txt non riuscito: ' + str(_e))

import shutil
zip_path = shutil.make_archive('risultati_mmm', 'zip', OUTPUT_DIR)
from google.colab import files as colab_files
colab_files.download(percorso)
colab_files.download(zip_path)
print('Download avviato: Excel + zip con modello, figure, HTML e requirements_freeze.txt')

## Note finali e limiti (per la tesi)

- **Incrementale vs attribuito**: il gap tra ROAS MMM e benchmark di attribution e' un
  risultato da discutere (l'attribution premia i canali di chiusura; l'MMM misura
  causalita' incrementale). Non calibrare mai il modello sul benchmark.
- **Collinearita'**: canali con VIF alto hanno ROAS identificati male (intervalli larghi);
  e' un limite noto dei MMM, dichiaralo (vedi foglio EDA/Diagnostica).
- **Riesecuzione**: con `AUTO_CONFERMA = True` il notebook riesegue senza fermarsi ai
  checkpoint (usalo solo dopo una prima esecuzione validata a mano).
- **Estensioni**: nuovi sinonimi/canali/geo si aggiungono nella cella CONFIG + DIZIONARI;
  poi riesegui dalla cella 5 (ingestion) in poi.
- **Riproducibilita'**: seed fisso (42), versioni in `requirements_freeze.txt` e nel foglio
  README, modello salvato in `output_mmm/modello_meridian.pkl`.

## 14. Riepilogo finale

Cosa e' stato prodotto, dove si trova, e le tre cose da controllare prima di
usare questi numeri in tesi. Non ricalcola nulla: raccoglie in un posto solo
quello che le celle precedenti hanno gia' stampato in mezzo a centinaia di
righe di output.

In [ ]:
# ============================================================================
# CELLA 14 - RIEPILOGO FINALE: cosa e' stato prodotto, cosa e' stato saltato
# ============================================================================
# Nessun calcolo nuovo. A fine esecuzione l'output e' lungo e le cose che
# decidono se i numeri sono usabili sono sparse: qui stanno insieme.

def _g(nome, default=None):
    return globals().get(nome, default)

print()
print('=' * 76)
print('RIEPILOGO DELL' + chr(39) + 'ESECUZIONE')
print(intestazione_run() if callable(_g('intestazione_run')) else '')
print('=' * 76)

# --- 1. cosa e' stato prodotto ---------------------------------------------
print()
print('1. COSA E' + chr(39) + ' STATO PRODOTTO E DOVE')
_out = _g('OUTPUT_DIR', '.')
_prodotti = [
    ('Excel dei risultati', _g('percorso')),
    ('archivio zip completo', _g('zip_path')),
    ('modello salvato', _g('percorso_modello')),
    ('cartella figure', _g('FIG_DIR')),
    ('report HTML Meridian', os.path.join(_out, 'sintesi_modello.html')),
    ('versioni dei pacchetti', os.path.join(_out, 'requirements_freeze.txt')),
]
_fatti, _mancanti_out = [], []
for _et, _p in _prodotti:
    if not _p:
        _mancanti_out.append((_et, '(percorso non definito)'))
    elif os.path.exists(str(_p)):
        _fatti.append((_et, os.path.abspath(str(_p))))
    else:
        _mancanti_out.append((_et, str(_p)))

print()
print('   PRODOTTI (' + str(len(_fatti)) + '):')
if _fatti:
    for _et, _p in _fatti:
        print('      [x] ' + _et.ljust(24) + _p)
else:
    print('      nessuno.')
print()
print('   NON PRODOTTI (' + str(len(_mancanti_out)) + '):')
if _mancanti_out:
    for _et, _p in _mancanti_out:
        print('      [ ] ' + _et.ljust(24) + _p)
else:
    print('      nessuno: tutto prodotto.')

if _g('EXPORT_COMPLETO'):
    print()
    print('   EXPORT_COMPLETO = True: l' + chr(39) + 'Excel contiene anche i fogli tecnici')
    print('   (modello, EDA, diagnostica, incertezza, dati puliti).')
else:
    print()
    print('   ATTENZIONE: EXPORT_COMPLETO = False, Excel essenziale SENZA i fogli')
    print('   tecnici e senza gli intervalli. Per la tesi rimettilo a True.')

# --- 2. cosa e' stato SALTATO ----------------------------------------------
# Un passaggio saltato non si vede: non produce output. Se non viene detto qui
# a caratteri visibili, del buco ci si accorge solo trovando l'Excel vuoto.
print()
print('2. COSA E' + chr(39) + ' STATO SALTATO')
_salti = []
if _g('CONFRONTO') is None:
    _salti.append(('CONFRONTO COL BENCHMARK (cella 11)',
                   str(_g('CONFRONTO_SALTATO') or 'cella non eseguita'),
                   'foglio Confronto_ROAS vuoto; nessun gap incrementale vs attribuito'))
if _g('MMM') is None:
    _salti.append(('FIT DEL MODELLO (cella 9)', 'cella non eseguita',
                   'nessuna stima disponibile'))
if _g('QUOTE_VERO_VS_STIMATO') is None and _g('TAB_ROAS') is not None:
    _salti.append(('DIAGNOSTICA STIMATO vs VERO (cella 10-bis)',
                   'file di verita' + chr(39) + ' non trovato',
                   'nessun controllo del modello contro la verita' + chr(39) + ' simulata'))

if not _salti:
    print('   Niente: tutti i passaggi sono stati eseguiti.')
else:
    for _tit, _perche, _cosa in _salti:
        print()
        print('   ' + '!' * 68)
        print('   !!  ' + _tit)
        print('   !!  motivo:      ' + _perche)
        print('   !!  conseguenza: ' + _cosa)
        print('   ' + '!' * 68)

# --- 3. le tre cose da controllare -----------------------------------------
print()
print('3. LE TRE COSE DA CONTROLLARE PRIMA DI FIDARTI DEI NUMERI')

print()
print('   (a) CONVERGENZA - le catene hanno esplorato bene il posterior?')
_rhat, _div = _g('RHAT_MAX'), _g('DIVERGENZE')
_diag = _g('DIAG_TABELLA')
_ess = None
try:
    _ess = float(_diag['ess_min'].min())
except Exception:
    pass
if _rhat is None:
    print('       non disponibile: la cella 9 (fit) non e' + chr(39) + ' stata eseguita.')
else:
    print('       R-hat max ' + str(round(float(_rhat), 4)) +
          '   divergenze ' + str(_div) +
          ('   ESS min ' + str(int(_ess)) if _ess else ''))
    if _g('DURATA_FIT_MIN') is not None:
        print('       durata reale del fit: ' +
              str(round(float(_g('DURATA_FIT_MIN')), 1)) + ' minuti')
    _problemi = []
    if float(_rhat) > 1.1:
        _problemi.append('R-hat > 1,1: le catene NON hanno convergito, stime da buttare')
    elif float(_rhat) > 1.01:
        _problemi.append('R-hat sopra 1,01: convergenza accettabile ma non ottima')
    if _div and int(_div) > 0:
        _problemi.append(str(_div) + ' divergenze: geometria difficile in qualche zona')
    if _ess and _ess < 400:
        _problemi.append('ESS min sotto 400: pochi campioni indipendenti sulle code')
    if _problemi:
        for _p in _problemi:
            print('       - ' + _p)
    else:
        print('       OK.')

print()
print('   (b) QUOTA MEDIA SUL KPI - quanto il modello attribuisce ai media')
_inc, _tot = _g('inc_tot_kpi'), _g('KPI_TOTALE')
if _inc is None or not _tot:
    print('       non disponibile: la cella 10 (analisi) non e' + chr(39) + ' stata eseguita.')
else:
    _quota = 100.0 * _inc / _tot
    print('       ' + str(round(_quota, 1)) + '% del KPI attribuito ai media, ' +
          str(round(100 - _quota, 1)) + '% alla baseline.')
    if _quota > 60:
        print('       SOSPETTA: quote cosi' + chr(39) + ' alte nel recruiting sono rare. Il candidato')
        print('       piu' + chr(39) + ' probabile e' + chr(39) + ' il confondimento media/baseline (Sez. 4.8): parte')
        print('       dell' + chr(39) + 'organico finisce attribuita ai media.')
    elif _quota < 5:
        print('       SOSPETTA: quota molto bassa, controlla spesa e variabili di controllo.')
    else:
        print('       In banda plausibile per il settore.')
    _qv = _g('QUOTE_VERO_VS_STIMATO')
    if _qv is not None:
        try:
            _r = _qv[_qv['canale'] == 'TOTALE MEDIA'].iloc[0]
            print('       Verita' + chr(39) + ' del mondo simulato: ' + str(_r['quota_vera_%']) +
                  '% (scarto ' + str(_r['scarto_pp']) + ' p.p.) - vedi diagnostica 10-bis.')
        except Exception:
            pass

print()
print('   (c) AMPIEZZA DEGLI INTERVALLI - il ROAS e' + chr(39) + ' identificato o no?')
_tr = _g('TAB_ROAS')
if _tr is None:
    print('       non disponibile: la cella 10 (analisi) non e' + chr(39) + ' stata eseguita.')
else:
    try:
        _amp = ((_tr['roas_ci95'] - _tr['roas_ci05']) /
                _tr['roas'].replace(0, float('nan'))).abs()
        _n_larghi = 0
        print('       ampiezza dell' + chr(39) + 'intervallo CI90 in rapporto alla stima, per canale:')
        for _i in range(len(_tr)):
            _a = _amp.iloc[_i]
            if _a != _a:
                continue
            _largo = _a > 2.0
            _n_larghi += 1 if _largo else 0
            print('         ' + str(_tr['canale'].iloc[_i]).ljust(20) +
                  ('%.1fx' % _a) + ('   <- largo' if _largo else ''))
        if _n_larghi:
            print('       I canali marcati hanno un ROAS mal identificato: l' + chr(39) + 'intervallo')
            print('       vale piu' + chr(39) + ' del doppio della stima. Non usarli da soli per decidere.')
            print('       E' + chr(39) + ' un risultato da DICHIARARE, non da nascondere (Sez. 3.8).')
        else:
            print('       Nessun canale con intervallo patologicamente largo.')
    except Exception as _e:
        print('       non calcolabile: ' + str(_e))

# --- 4. i prior hanno deciso al posto dei dati? ----------------------------
if _g('PRIOR_INFO_ATTIVI'):
    print()
    print('   (d) PRIOR INFORMATIVI ATTIVI - le stime dipendono dai ROAS forniti')
    _pp = _g('PRIOR_VS_POSTERIOR')
    if _pp is None:
        print('       confronto prior/posterior non disponibile.')
    else:
        _rap = _pp['posterior_su_prior'].astype(float)
        _fermi = _pp.loc[(_rap - 1.0).abs() < 0.10, 'canale'].tolist()
        if _fermi:
            print('       posterior entro il 10% del prior su: ' + ', '.join(_fermi))
            print('       li' + chr(39) + ' il livello stimato e' + chr(39) + ' il riferimento dato in ingresso,')
            print('       non una conferma indipendente: da dichiarare.')
        else:
            print('       nessun canale col posterior incollato al prior.')

print()
print('=' * 76)
